# Anatomical explainability validation

**Public role:** Core anatomical localization and attribution notebook.

**Relation to the manuscript:** Supports Grad-CAM/Grad-CAM++/IG localization and anatomical enrichment analyses.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `08_anatomical_explainability_validation.ipynb`


In [ ]:
import os
# Cell 1: Setup for AQPR anatomical explainability validation on HYGD

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd

try:
    import torch
except Exception:
    torch = None

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass


# ---------------------------------------------------------------------
# Project configuration
# ---------------------------------------------------------------------

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
CONFIG_PATH = JOURNAL_ROOT / "configs" / "project_config.json"
PATH_REGISTRY_PATH = JOURNAL_ROOT / "configs" / "path_registry.csv"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing project config: {CONFIG_PATH}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

SEED = int(PROJECT_CONFIG.get("seed", 42))

random.seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch is not None and torch.cuda.is_available() else "cpu")
USE_AMP = bool(torch is not None and torch.cuda.is_available())

if PATH_REGISTRY_PATH.exists():
    path_registry = pd.read_csv(PATH_REGISTRY_PATH)
    PATHS = {
        row["path_key"]: Path(row["path"])
        for _, row in path_registry.iterrows()
    }
else:
    path_registry = pd.DataFrame()
    PATHS = {}


# ---------------------------------------------------------------------
# Notebook 08 paths
# ---------------------------------------------------------------------

RESULTS_06_DIR = JOURNAL_ROOT / "results" / "06_hygd_final_od_oc_proxy"
RESULTS_07_DIR = JOURNAL_ROOT / "results" / "07_airogs_light_glaucoma_classifier_benchmark"
RESULTS_08_DIR = JOURNAL_ROOT / "results" / "08_anatomical_explainability_validation"
RESULTS_08_DIR.mkdir(parents=True, exist_ok=True)

HYGD_PROXY_ROOT = JOURNAL_ROOT / "project_data" / "hygd_final_od_oc_proxy"
EXPLAINABILITY_PROJECT_DATA_DIR = JOURNAL_ROOT / "project_data" / "hygd_aqpr_anatomical_explainability"
EXPLAINABILITY_PROJECT_DATA_DIR.mkdir(parents=True, exist_ok=True)

CLASSIFIER_MODEL_ROOT = JOURNAL_ROOT / "models" / "airogs_light_glaucoma_classifiers"
FROZEN_PANEL_DIR = CLASSIFIER_MODEL_ROOT / "frozen_aqpr_model_panel_notebook07"

NOTEBOOK06_FINAL_STATUS_JSON = RESULTS_06_DIR / "notebook06_final_status.json"
NOTEBOOK07_FINAL_STATUS_JSON = RESULTS_07_DIR / "notebook07_final_status.json"

HYGD_PROXY_ANALYSIS_MANIFEST_CSV = HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_analysis_manifest.csv"
HYGD_PROXY_PATIENT_SUMMARY_CSV = HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_patient_summary.csv"

FROZEN_AQPR_PANEL_MANIFEST_CSV = FROZEN_PANEL_DIR / "notebook07_frozen_aqpr_model_panel_manifest.csv"
FROZEN_AQPR_PANEL_MANIFEST_JSON = FROZEN_PANEL_DIR / "notebook07_frozen_aqpr_model_panel_manifest.json"

EXPLAINABILITY_MANIFEST_CSV = (
    EXPLAINABILITY_PROJECT_DATA_DIR
    / "hygd_aqpr_panel_anatomical_explainability_manifest.csv"
)

RESULTS_EXPLAINABILITY_MANIFEST_CSV = (
    RESULTS_08_DIR
    / "notebook08_hygd_aqpr_panel_anatomical_explainability_manifest.csv"
)

SETUP_SUMMARY_JSON = RESULTS_08_DIR / "notebook08_setup_summary.json"
SETUP_AUDIT_CSV = RESULTS_08_DIR / "notebook08_input_artifact_audit.csv"
SETUP_DATASET_SUMMARY_CSV = RESULTS_08_DIR / "notebook08_setup_dataset_summary.csv"
SETUP_LABEL_SUMMARY_CSV = RESULTS_08_DIR / "notebook08_setup_label_summary.csv"


# ---------------------------------------------------------------------
# AQPR model panel
# ---------------------------------------------------------------------

PRIMARY_MODEL_NAME = "EfficientNetB0"
SECONDARY_AQPR_MODEL_NAMES = ["DenseNet121", "SwinTiny", "ConvNeXtTiny"]
AQPR_PANEL_MODEL_NAMES = [PRIMARY_MODEL_NAME] + SECONDARY_AQPR_MODEL_NAMES

AQPR_PANEL_ROLES = {
    "EfficientNetB0": "primary_validation_selected_classifier",
    "DenseNet121": "secondary_aqpr_architecture_panel",
    "SwinTiny": "secondary_aqpr_architecture_panel",
    "ConvNeXtTiny": "secondary_aqpr_architecture_panel",
}


# ---------------------------------------------------------------------
# Utility functions
# ---------------------------------------------------------------------

def read_json_if_exists(path):
    path = Path(path)

    if not path.exists():
        return {"status": "missing", "path": str(path)}

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def as_bool_series(series):
    if series.dtype == bool:
        return series.fillna(False)

    return series.astype(str).str.strip().str.lower().isin(
        ["true", "1", "1.0", "yes", "y"]
    )


def require_columns(df, required_cols, table_name):
    missing = [
        col for col in required_cols
        if col not in df.columns
    ]

    if missing:
        raise ValueError(f"Missing columns in {table_name}: {missing}")


def first_existing_column(df, candidates, table_name):
    for col in candidates:
        if col in df.columns:
            return col

    raise ValueError(
        f"No valid column found in {table_name}. Tried columns: {candidates}"
    )


def path_exists_or_false(path_value):
    path_text = str(path_value).strip()

    if path_text in {"", "nan", "None"}:
        return False

    return Path(path_text).exists()


def clean_string_id(series):
    return (
        series
        .astype(str)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan, "NaN": np.nan})
    )


# ---------------------------------------------------------------------
# Required artifact audit
# ---------------------------------------------------------------------

required_files = {
    "notebook06_final_status": NOTEBOOK06_FINAL_STATUS_JSON,
    "notebook07_final_status": NOTEBOOK07_FINAL_STATUS_JSON,
    "hygd_proxy_analysis_manifest": HYGD_PROXY_ANALYSIS_MANIFEST_CSV,
    "hygd_proxy_patient_summary": HYGD_PROXY_PATIENT_SUMMARY_CSV,
    "frozen_aqpr_panel_manifest_csv": FROZEN_AQPR_PANEL_MANIFEST_CSV,
    "frozen_aqpr_panel_manifest_json": FROZEN_AQPR_PANEL_MANIFEST_JSON,
}

artifact_audit = pd.DataFrame(
    [
        {
            "artifact": key,
            "path": str(path),
            "exists": Path(path).exists(),
        }
        for key, path in required_files.items()
    ]
)

artifact_audit.to_csv(SETUP_AUDIT_CSV, index=False)

missing_files = artifact_audit.loc[
    ~artifact_audit["exists"],
    "artifact",
].tolist()

if missing_files:
    print(artifact_audit.to_string(index=False))
    raise FileNotFoundError(
        "Missing required Notebook 08 inputs. "
        f"Missing artifacts: {missing_files}. "
        "Notebook 07 Cell 16 must be completed before Notebook 08."
    )


# ---------------------------------------------------------------------
# Load upstream statuses and manifests
# ---------------------------------------------------------------------

notebook06_status = read_json_if_exists(NOTEBOOK06_FINAL_STATUS_JSON)
notebook07_status = read_json_if_exists(NOTEBOOK07_FINAL_STATUS_JSON)

hygd_proxy_manifest = pd.read_csv(HYGD_PROXY_ANALYSIS_MANIFEST_CSV)
hygd_patient_proxy_summary = pd.read_csv(HYGD_PROXY_PATIENT_SUMMARY_CSV)
frozen_aqpr_panel_manifest = pd.read_csv(FROZEN_AQPR_PANEL_MANIFEST_CSV)

require_columns(
    frozen_aqpr_panel_manifest,
    [
        "model_name",
        "aqpr_panel_role",
        "checkpoint_path",
        "input_size",
        "validation_selected_threshold",
        "temperature",
        "internal_patient_level_claim_allowed",
    ],
    "frozen AQPR panel manifest",
)

observed_panel_models = frozen_aqpr_panel_manifest["model_name"].astype(str).tolist()

if observed_panel_models != AQPR_PANEL_MODEL_NAMES:
    raise RuntimeError(
        f"Unexpected AQPR panel. Expected {AQPR_PANEL_MODEL_NAMES}, "
        f"found {observed_panel_models}"
    )

if as_bool_series(frozen_aqpr_panel_manifest["internal_patient_level_claim_allowed"]).any():
    raise RuntimeError("AIROGS-light internal patient-level claims must remain disabled.")

frozen_aqpr_panel_manifest["checkpoint_exists"] = frozen_aqpr_panel_manifest["checkpoint_path"].apply(
    path_exists_or_false
)

missing_checkpoints = frozen_aqpr_panel_manifest.loc[
    ~frozen_aqpr_panel_manifest["checkpoint_exists"],
    ["model_name", "checkpoint_path"],
]

if len(missing_checkpoints) > 0:
    print(missing_checkpoints.to_string(index=False))
    raise FileNotFoundError("One or more frozen AQPR model checkpoints are missing.")


# ---------------------------------------------------------------------
# Normalize HYGD anatomical proxy manifest
# ---------------------------------------------------------------------

require_columns(
    hygd_proxy_manifest,
    [
        "sample_id",
        "patient_id",
        "gon_binary",
    ],
    "HYGD proxy manifest",
)

hygd_proxy_manifest = hygd_proxy_manifest.copy()

hygd_proxy_manifest["sample_id"] = clean_string_id(hygd_proxy_manifest["sample_id"])
hygd_proxy_manifest["patient_id"] = clean_string_id(hygd_proxy_manifest["patient_id"])
hygd_proxy_manifest["gon_binary"] = pd.to_numeric(
    hygd_proxy_manifest["gon_binary"],
    errors="coerce",
).astype("Int64")

if "gon_label" not in hygd_proxy_manifest.columns:
    hygd_proxy_manifest["gon_label"] = np.select(
        [
            hygd_proxy_manifest["gon_binary"].eq(1),
            hygd_proxy_manifest["gon_binary"].eq(0),
        ],
        [
            "GON",
            "No GON",
        ],
        default="unknown",
    )
else:
    hygd_proxy_manifest["gon_label"] = hygd_proxy_manifest["gon_label"].astype(str)

image_path_col = first_existing_column(
    hygd_proxy_manifest,
    [
        "classifier_image_path",
        "image_path_for_inference",
        "image_path",
    ],
    "HYGD proxy manifest",
)

hygd_proxy_manifest["classifier_image_path"] = hygd_proxy_manifest[image_path_col].astype(str)

if "mask_512_path" not in hygd_proxy_manifest.columns:
    hygd_proxy_manifest["mask_512_path"] = hygd_proxy_manifest["sample_id"].apply(
        lambda sample_id: str(HYGD_PROXY_ROOT / "masks_512" / f"{sample_id}_od_oc_mask_512.png")
    )

if "overlay_512_path" not in hygd_proxy_manifest.columns:
    hygd_proxy_manifest["overlay_512_path"] = hygd_proxy_manifest["sample_id"].apply(
        lambda sample_id: str(HYGD_PROXY_ROOT / "overlays_512" / f"{sample_id}_od_oc_overlay_512.jpg")
    )

require_columns(
    hygd_proxy_manifest,
    [
        "include_strict_anatomical_analysis",
        "include_high_confidence_anatomical_analysis",
    ],
    "HYGD proxy manifest",
)

for bool_col in [
    "include_strict_anatomical_analysis",
    "include_high_confidence_anatomical_analysis",
    "low_confidence_anatomical_proxy",
    "excluded_from_anatomical_analysis",
]:
    if bool_col in hygd_proxy_manifest.columns:
        hygd_proxy_manifest[bool_col] = as_bool_series(hygd_proxy_manifest[bool_col])
    else:
        hygd_proxy_manifest[bool_col] = False

hygd_proxy_manifest["image_exists"] = hygd_proxy_manifest["classifier_image_path"].apply(
    path_exists_or_false
)

hygd_proxy_manifest["mask_512_exists"] = hygd_proxy_manifest["mask_512_path"].apply(
    path_exists_or_false
)

hygd_proxy_manifest["overlay_512_exists"] = hygd_proxy_manifest["overlay_512_path"].apply(
    path_exists_or_false
)


# ---------------------------------------------------------------------
# Build Notebook 08 HYGD AQPR explainability manifest
# ---------------------------------------------------------------------

hygd_explainability_manifest = hygd_proxy_manifest.copy()

hygd_explainability_manifest["evaluation_dataset"] = "HYGD"
hygd_explainability_manifest["evaluation_role"] = "external_anatomical_patient_level_aqpr_audit"
hygd_explainability_manifest["trained_on_dataset"] = False
hygd_explainability_manifest["evaluation_split"] = "hygd_external"

hygd_explainability_manifest["eligible_for_aqpr_anatomical_xai"] = (
    hygd_explainability_manifest["image_exists"]
    & hygd_explainability_manifest["mask_512_exists"]
    & hygd_explainability_manifest["include_strict_anatomical_analysis"]
    & hygd_explainability_manifest["gon_binary"].notna()
    & hygd_explainability_manifest["patient_id"].notna()
)

hygd_explainability_manifest["eligible_for_high_confidence_proxy_xai"] = (
    hygd_explainability_manifest["eligible_for_aqpr_anatomical_xai"]
    & hygd_explainability_manifest["include_high_confidence_anatomical_analysis"]
)

hygd_explainability_manifest["n_frozen_aqpr_models"] = len(AQPR_PANEL_MODEL_NAMES)
hygd_explainability_manifest["primary_model_name"] = PRIMARY_MODEL_NAME
hygd_explainability_manifest["secondary_aqpr_model_names"] = "|".join(SECONDARY_AQPR_MODEL_NAMES)

hygd_explainability_manifest.to_csv(EXPLAINABILITY_MANIFEST_CSV, index=False)
hygd_explainability_manifest.to_csv(RESULTS_EXPLAINABILITY_MANIFEST_CSV, index=False)


# ---------------------------------------------------------------------
# Summaries
# ---------------------------------------------------------------------

dataset_summary = pd.DataFrame(
    [
        {
            "evaluation_dataset": "HYGD",
            "evaluation_role": "external_anatomical_patient_level_aqpr_audit",
            "n_images": int(hygd_explainability_manifest["sample_id"].count()),
            "n_patients": int(hygd_explainability_manifest["patient_id"].nunique(dropna=True)),
            "n_existing_images": int(hygd_explainability_manifest["image_exists"].sum()),
            "n_existing_masks": int(hygd_explainability_manifest["mask_512_exists"].sum()),
            "n_strict_anatomical_proxy": int(
                hygd_explainability_manifest["include_strict_anatomical_analysis"].sum()
            ),
            "n_high_confidence_anatomical_proxy": int(
                hygd_explainability_manifest["include_high_confidence_anatomical_analysis"].sum()
            ),
            "n_aqpr_xai_eligible": int(
                hygd_explainability_manifest["eligible_for_aqpr_anatomical_xai"].sum()
            ),
            "n_high_confidence_xai_eligible": int(
                hygd_explainability_manifest["eligible_for_high_confidence_proxy_xai"].sum()
            ),
        }
    ]
)

label_summary = (
    hygd_explainability_manifest
    .groupby("gon_label", dropna=False)
    .agg(
        n_images=("sample_id", "count"),
        n_patients=("patient_id", "nunique"),
        n_aqpr_xai_eligible=("eligible_for_aqpr_anatomical_xai", "sum"),
        n_high_confidence_xai_eligible=("eligible_for_high_confidence_proxy_xai", "sum"),
    )
    .reset_index()
)

dataset_summary.to_csv(SETUP_DATASET_SUMMARY_CSV, index=False)
label_summary.to_csv(SETUP_LABEL_SUMMARY_CSV, index=False)

n_total = int(len(hygd_explainability_manifest))
n_patients = int(hygd_explainability_manifest["patient_id"].nunique(dropna=True))
n_missing_images = int((~hygd_explainability_manifest["image_exists"]).sum())
n_missing_masks = int((~hygd_explainability_manifest["mask_512_exists"]).sum())
n_eligible = int(hygd_explainability_manifest["eligible_for_aqpr_anatomical_xai"].sum())
n_high_conf_eligible = int(hygd_explainability_manifest["eligible_for_high_confidence_proxy_xai"].sum())

setup_summary = {
    "notebook": "08_anatomical_explainability_validation",
    "status": "PASS",
    "seed": int(SEED),
    "device": str(DEVICE),
    "use_amp": bool(USE_AMP),
    "project_goal": "AQPR anatomical explainability validation of a frozen AIROGS-light model panel on HYGD.",
    "notebook06_status": notebook06_status.get("status", "unknown"),
    "notebook07_status": notebook07_status.get("status", "unknown"),
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_anatomical_dataset": "HYGD",
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "frozen_aqpr_panel_manifest_csv": str(FROZEN_AQPR_PANEL_MANIFEST_CSV),
    "hygd_proxy_manifest_csv": str(HYGD_PROXY_ANALYSIS_MANIFEST_CSV),
    "hygd_explainability_manifest_csv": str(EXPLAINABILITY_MANIFEST_CSV),
    "results_explainability_manifest_csv": str(RESULTS_EXPLAINABILITY_MANIFEST_CSV),
    "n_hygd_images": n_total,
    "n_hygd_patients": n_patients,
    "n_missing_images": n_missing_images,
    "n_missing_masks": n_missing_masks,
    "n_aqpr_xai_eligible_images": n_eligible,
    "n_high_confidence_xai_eligible_images": n_high_conf_eligible,
    "patient_level_external_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
    "setup_artifact_audit_csv": str(SETUP_AUDIT_CSV),
    "dataset_summary_csv": str(SETUP_DATASET_SUMMARY_CSV),
    "label_summary_csv": str(SETUP_LABEL_SUMMARY_CSV),
}

with open(SETUP_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(setup_summary, f, indent=2)

print("Notebook 08 setup summary:")
print(json.dumps(setup_summary, indent=2))

print("\nFrozen AQPR model panel:")
print(
    frozen_aqpr_panel_manifest[
        [
            "model_name",
            "aqpr_panel_role",
            "checkpoint_exists",
            "input_size",
            "validation_selected_threshold",
            "temperature",
        ]
    ].to_string(index=False)
)

print("\nHYGD dataset summary:")
print(dataset_summary.to_string(index=False))

print("\nHYGD label summary:")
print(label_summary.to_string(index=False))

print(f"\nExplainability manifest: {EXPLAINABILITY_MANIFEST_CSV}")
print(f"Results manifest: {RESULTS_EXPLAINABILITY_MANIFEST_CSV}")
print(f"Setup summary JSON: {SETUP_SUMMARY_JSON}")
print(f"Artifact audit CSV: {SETUP_AUDIT_CSV}")

if notebook06_status.get("status") != "PASS":
    raise RuntimeError("Notebook 06 status is not PASS.")

if notebook07_status.get("status") != "PASS":
    raise RuntimeError("Notebook 07 status is not PASS.")

if n_total <= 0:
    raise RuntimeError("HYGD manifest is empty.")

if n_patients <= 0:
    raise RuntimeError("HYGD patient IDs are missing or invalid.")

if n_missing_images > 0:
    raise RuntimeError(f"Missing HYGD image files: {n_missing_images}")

if n_missing_masks > 0:
    raise RuntimeError(f"Missing HYGD OD/OC mask files: {n_missing_masks}")

if n_eligible <= 0:
    raise RuntimeError("No HYGD images are eligible for AQPR anatomical XAI.")

print("Status: PASS")

In [ ]:
# Cell 2: Define HYGD AQPR XAI cohorts and load frozen model-panel registry

import sys
import subprocess
import importlib.util

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from IPython.display import display

if importlib.util.find_spec("timm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])

import timm


# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------

EXPLAINABILITY_COHORT_SUMMARY_CSV = RESULTS_08_DIR / "notebook08_hygd_aqpr_xai_cohort_summary.csv"
AQPR_MODEL_REGISTRY_CSV = RESULTS_08_DIR / "notebook08_aqpr_model_registry_for_xai.csv"
MODEL_LOADING_SMOKE_CSV = RESULTS_08_DIR / "notebook08_aqpr_model_loading_smoke_test.csv"
MODEL_LOADING_SMOKE_JSON = RESULTS_08_DIR / "notebook08_aqpr_model_loading_smoke_test.json"
PRIMARY_GRADCAM_SMOKE_FIGURE = RESULTS_08_DIR / "notebook08_primary_model_gradcam_smoke_test_overlay.png"


# ---------------------------------------------------------------------
# Load manifest created in Cell 1
# ---------------------------------------------------------------------

if not EXPLAINABILITY_MANIFEST_CSV.exists():
    raise FileNotFoundError(f"Missing explainability manifest from Cell 1: {EXPLAINABILITY_MANIFEST_CSV}")

if not FROZEN_AQPR_PANEL_MANIFEST_CSV.exists():
    raise FileNotFoundError(f"Missing frozen AQPR panel manifest: {FROZEN_AQPR_PANEL_MANIFEST_CSV}")

explainability_manifest = pd.read_csv(EXPLAINABILITY_MANIFEST_CSV)
frozen_aqpr_panel_manifest = pd.read_csv(FROZEN_AQPR_PANEL_MANIFEST_CSV)

require_columns(
    explainability_manifest,
    [
        "sample_id",
        "patient_id",
        "gon_binary",
        "gon_label",
        "classifier_image_path",
        "mask_512_path",
        "eligible_for_aqpr_anatomical_xai",
        "eligible_for_high_confidence_proxy_xai",
    ],
    "Notebook 08 explainability manifest",
)

require_columns(
    frozen_aqpr_panel_manifest,
    [
        "model_name",
        "aqpr_panel_role",
        "checkpoint_path",
        "input_size",
        "batch_size",
        "validation_selected_threshold",
        "temperature",
        "internal_patient_level_claim_allowed",
    ],
    "frozen AQPR panel manifest",
)

explainability_manifest["sample_id"] = explainability_manifest["sample_id"].astype(str)
explainability_manifest["patient_id"] = explainability_manifest["patient_id"].astype(str)
explainability_manifest["gon_binary"] = pd.to_numeric(
    explainability_manifest["gon_binary"],
    errors="coerce",
).astype(int)

for bool_col in [
    "eligible_for_aqpr_anatomical_xai",
    "eligible_for_high_confidence_proxy_xai",
    "include_strict_anatomical_analysis",
    "include_high_confidence_anatomical_analysis",
    "low_confidence_anatomical_proxy",
    "excluded_from_anatomical_analysis",
    "image_exists",
    "mask_512_exists",
]:
    if bool_col in explainability_manifest.columns:
        explainability_manifest[bool_col] = as_bool_series(explainability_manifest[bool_col])

frozen_aqpr_panel_manifest["model_name"] = frozen_aqpr_panel_manifest["model_name"].astype(str)

if frozen_aqpr_panel_manifest["model_name"].tolist() != AQPR_PANEL_MODEL_NAMES:
    raise RuntimeError(
        f"Unexpected AQPR panel order. Expected {AQPR_PANEL_MODEL_NAMES}, "
        f"found {frozen_aqpr_panel_manifest['model_name'].tolist()}"
    )

if as_bool_series(frozen_aqpr_panel_manifest["internal_patient_level_claim_allowed"]).any():
    raise RuntimeError("Internal AIROGS-light patient-level claims must remain disabled.")


# ---------------------------------------------------------------------
# HYGD AQPR XAI cohorts
# ---------------------------------------------------------------------

hygd_strict_xai_df = (
    explainability_manifest[explainability_manifest["eligible_for_aqpr_anatomical_xai"]]
    .copy()
    .reset_index(drop=True)
)

hygd_high_confidence_xai_df = (
    explainability_manifest[explainability_manifest["eligible_for_high_confidence_proxy_xai"]]
    .copy()
    .reset_index(drop=True)
)

analysis_cohorts = {
    "hygd_external_strict_anatomical": hygd_strict_xai_df,
    "hygd_external_high_confidence_anatomical": hygd_high_confidence_xai_df,
}

cohort_rows = []

for cohort_name, cohort_df in analysis_cohorts.items():
    cohort_rows.append(
        {
            "cohort_name": cohort_name,
            "evaluation_dataset": "HYGD",
            "evaluation_role": "external_anatomical_patient_level_aqpr_audit",
            "n_images": int(len(cohort_df)),
            "n_patients": int(cohort_df["patient_id"].nunique(dropna=True)),
            "n_gon_positive_images": int((cohort_df["gon_binary"] == 1).sum()),
            "n_gon_negative_images": int((cohort_df["gon_binary"] == 0).sum()),
            "n_gon_positive_patients": int(
                cohort_df.loc[cohort_df["gon_binary"] == 1, "patient_id"].nunique(dropna=True)
            ),
            "n_gon_negative_patients": int(
                cohort_df.loc[cohort_df["gon_binary"] == 0, "patient_id"].nunique(dropna=True)
            ),
            "n_low_confidence_proxy": int(
                cohort_df["low_confidence_anatomical_proxy"].sum()
            ) if "low_confidence_anatomical_proxy" in cohort_df.columns else 0,
            "n_failed_proxy": int(
                cohort_df["excluded_from_anatomical_analysis"].sum()
            ) if "excluded_from_anatomical_analysis" in cohort_df.columns else 0,
        }
    )

cohort_summary = pd.DataFrame(cohort_rows)
cohort_summary.to_csv(EXPLAINABILITY_COHORT_SUMMARY_CSV, index=False)


# Backward-compatible aliases for later cells that we will also adapt.
primary_hygd_xai_df = hygd_strict_xai_df
primary_test_df = hygd_strict_xai_df
primary_test_high_confidence_df = hygd_high_confidence_xai_df


# ---------------------------------------------------------------------
# Model-name resolution
# ---------------------------------------------------------------------

MODEL_TIMM_CANDIDATES = {
    "EfficientNetB0": ["efficientnet_b0", "tf_efficientnet_b0"],
    "DenseNet121": ["densenet121"],
    "SwinTiny": ["swin_tiny_patch4_window7_224", "swin_tiny_patch4_window7_224.ms_in1k"],
    "ConvNeXtTiny": ["convnext_tiny", "convnext_tiny.fb_in22k_ft_in1k"],
}

FULL_TRAINING_RANKING_CSV = RESULTS_07_DIR / "notebook07_full_training_validation_ranking.csv"

ranking_timm_map = {}

if FULL_TRAINING_RANKING_CSV.exists():
    full_training_ranking = pd.read_csv(FULL_TRAINING_RANKING_CSV)

    if "model_name" in full_training_ranking.columns and "timm_model_name" in full_training_ranking.columns:
        ranking_timm_map = {
            str(row["model_name"]): str(row["timm_model_name"])
            for _, row in full_training_ranking.iterrows()
            if str(row["model_name"]) in AQPR_PANEL_MODEL_NAMES
        }


def candidate_timm_names(model_name):
    candidates = []

    if model_name in ranking_timm_map:
        candidates.append(ranking_timm_map[model_name])

    if "timm_model_name" in frozen_aqpr_panel_manifest.columns:
        row = frozen_aqpr_panel_manifest[
            frozen_aqpr_panel_manifest["model_name"] == model_name
        ]
        if len(row) == 1:
            value = str(row.iloc[0]["timm_model_name"])
            if value not in {"", "nan", "None"}:
                candidates.append(value)

    candidates.extend(MODEL_TIMM_CANDIDATES.get(model_name, []))

    seen = set()
    unique_candidates = []

    for name in candidates:
        if name not in seen:
            unique_candidates.append(name)
            seen.add(name)

    if not unique_candidates:
        raise RuntimeError(f"No timm candidates available for {model_name}")

    return unique_candidates


def create_timm_binary_classifier(timm_model_name, input_size):
    try:
        model = timm.create_model(
            timm_model_name,
            pretrained=False,
            num_classes=1,
            img_size=int(input_size),
        )
    except TypeError:
        model = timm.create_model(
            timm_model_name,
            pretrained=False,
            num_classes=1,
        )

    return model


def normalize_checkpoint_state_dict(state_dict):
    if all(str(key).startswith("module.") for key in state_dict.keys()):
        return {
            str(key).replace("module.", "", 1): value
            for key, value in state_dict.items()
        }

    return state_dict


def load_aqpr_classifier(model_name, checkpoint_path, input_size, device=DEVICE):
    checkpoint_path = Path(str(checkpoint_path))

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing checkpoint for {model_name}: {checkpoint_path}")

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )

    state_dict = normalize_checkpoint_state_dict(checkpoint["model_state_dict"])

    load_errors = []

    for timm_model_name in candidate_timm_names(model_name):
        model = create_timm_binary_classifier(
            timm_model_name=timm_model_name,
            input_size=input_size,
        ).to(device)

        try:
            model.load_state_dict(state_dict, strict=True)
            model.eval()

            return model, checkpoint, timm_model_name

        except RuntimeError as exc:
            load_errors.append(
                {
                    "timm_model_name": timm_model_name,
                    "error": str(exc)[:500],
                }
            )

            del model

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    raise RuntimeError(
        f"Could not load checkpoint for {model_name}. "
        f"Tried candidates: {load_errors}"
    )


# ---------------------------------------------------------------------
# Image preprocessing and inference helpers
# ---------------------------------------------------------------------

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def pad_to_square_rgb(image_rgb, pad_value=0):
    height, width = image_rgb.shape[:2]

    if height == width:
        return image_rgb

    side = max(height, width)

    padded = np.full(
        (side, side, 3),
        fill_value=pad_value,
        dtype=image_rgb.dtype,
    )

    y0 = (side - height) // 2
    x0 = (side - width) // 2

    padded[y0:y0 + height, x0:x0 + width] = image_rgb

    return padded


def resize_rgb(image_rgb, input_size):
    return cv2.resize(
        image_rgb,
        (int(input_size), int(input_size)),
        interpolation=cv2.INTER_AREA,
    )


def preprocess_classifier_image_from_path(image_path, input_size):
    image_path = Path(str(image_path))

    image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

    if image_bgr is None:
        raise RuntimeError(f"Unreadable image: {image_path}")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_rgb_square = pad_to_square_rgb(image_rgb, pad_value=0)
    image_rgb_resized = resize_rgb(image_rgb_square, input_size=input_size)

    image_float = image_rgb_resized.astype(np.float32) / 255.0
    image_float = (image_float - IMAGENET_MEAN) / IMAGENET_STD
    image_chw = np.transpose(image_float, (2, 0, 1))

    image_tensor = torch.from_numpy(np.ascontiguousarray(image_chw)).float()

    return image_tensor, image_rgb_square, image_rgb_resized


def flatten_logits(logits):
    if logits.ndim == 2 and logits.shape[1] == 1:
        return logits[:, 0]

    if logits.ndim == 1:
        return logits

    raise RuntimeError(f"Unexpected logits shape: {tuple(logits.shape)}")


def probability_from_logit(logit):
    return float(torch.sigmoid(torch.tensor(float(logit))).item())


def temperature_scaled_probability_from_logit(logit, temperature):
    return float(torch.sigmoid(torch.tensor(float(logit) / float(temperature))).item())


@torch.no_grad()
def predict_single_image(model, image_tensor, device=DEVICE):
    model.eval()

    if image_tensor.ndim == 3:
        image_tensor = image_tensor.unsqueeze(0)

    image_tensor = image_tensor.to(device)

    logits = flatten_logits(model(image_tensor))
    probability = torch.sigmoid(logits)

    return {
        "logit": float(logits[0].detach().cpu()),
        "probability": float(probability[0].detach().cpu()),
    }


def find_last_conv2d_layer(model):
    last_name = None
    last_module = None

    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            last_name = name
            last_module = module

    if last_module is None:
        return None, None

    return last_name, last_module


class GradCAMCapture:
    def __init__(self, target_layer):
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.handles = []

    def _forward_hook(self, module, inputs, output):
        self.activations = output

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def __enter__(self):
        self.handles.append(self.target_layer.register_forward_hook(self._forward_hook))
        self.handles.append(self.target_layer.register_full_backward_hook(self._backward_hook))
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        for handle in self.handles:
            handle.remove()
        self.handles = []


def compute_gradcam_heatmap(model, image_tensor, target_layer, target_mode="positive"):
    if target_layer is None:
        raise RuntimeError("Grad-CAM target layer is None.")

    model.eval()
    model.zero_grad(set_to_none=True)

    if image_tensor.ndim == 3:
        image_tensor = image_tensor.unsqueeze(0)

    image_tensor = image_tensor.to(DEVICE)

    with GradCAMCapture(target_layer) as capture:
        logits = flatten_logits(model(image_tensor))
        probs = torch.sigmoid(logits)

        if target_mode == "positive":
            score = logits.sum()
        elif target_mode == "negative":
            score = (-logits).sum()
        elif target_mode == "predicted":
            signs = torch.where(probs >= 0.5, 1.0, -1.0)
            score = (signs * logits).sum()
        else:
            raise ValueError(f"Unknown Grad-CAM target mode: {target_mode}")

        score.backward()

        activations = capture.activations
        gradients = capture.gradients

    if activations is None or gradients is None:
        raise RuntimeError("Grad-CAM activations or gradients were not captured.")

    weights = gradients.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weights * activations).sum(dim=1))

    cam_min = cam.amin(dim=(1, 2), keepdim=True)
    cam_max = cam.amax(dim=(1, 2), keepdim=True)

    heatmap = ((cam - cam_min) / (cam_max - cam_min + 1e-8))[0]
    heatmap = heatmap.detach().cpu().numpy()

    return {
        "logit": float(logits[0].detach().cpu()),
        "probability": float(probs[0].detach().cpu()),
        "heatmap": heatmap,
        "activation_shape": tuple(activations.shape),
        "gradient_shape": tuple(gradients.shape),
    }


def make_gradcam_overlay(image_rgb_resized, heatmap, alpha=0.40):
    heatmap_resized = cv2.resize(
        heatmap.astype(np.float32),
        (image_rgb_resized.shape[1], image_rgb_resized.shape[0]),
        interpolation=cv2.INTER_LINEAR,
    )

    heatmap_uint8 = np.uint8(255 * np.clip(heatmap_resized, 0.0, 1.0))
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    overlay = (
        (1.0 - alpha) * image_rgb_resized.astype(np.float32)
        + alpha * heatmap_color.astype(np.float32)
    )

    return np.clip(overlay, 0, 255).astype(np.uint8)


# ---------------------------------------------------------------------
# Build AQPR model registry and smoke-test model loading
# ---------------------------------------------------------------------

AQPR_MODEL_INFO = {}
smoke_rows = []

if len(hygd_strict_xai_df) == 0:
    raise RuntimeError("HYGD strict AQPR XAI cohort is empty.")

smoke_case = (
    hygd_strict_xai_df
    .sort_values(["gon_binary", "sample_id"], ascending=[False, True])
    .iloc[0]
)

for _, model_row in frozen_aqpr_panel_manifest.iterrows():
    model_name = str(model_row["model_name"])
    checkpoint_path = Path(str(model_row["checkpoint_path"]))
    input_size = int(model_row["input_size"])
    batch_size = int(model_row["batch_size"])
    threshold = float(model_row["validation_selected_threshold"])
    temperature = float(model_row["temperature"])

    image_tensor, _, _ = preprocess_classifier_image_from_path(
        image_path=smoke_case["classifier_image_path"],
        input_size=input_size,
    )

    model, checkpoint, resolved_timm_name = load_aqpr_classifier(
        model_name=model_name,
        checkpoint_path=checkpoint_path,
        input_size=input_size,
        device=DEVICE,
    )

    pred = predict_single_image(
        model=model,
        image_tensor=image_tensor,
        device=DEVICE,
    )

    target_layer_name, target_layer = find_last_conv2d_layer(model)

    AQPR_MODEL_INFO[model_name] = {
        "model_name": model_name,
        "aqpr_panel_role": str(model_row["aqpr_panel_role"]),
        "family": str(model_row["family"]) if "family" in model_row.index else "",
        "checkpoint_path": checkpoint_path,
        "input_size": input_size,
        "batch_size": batch_size,
        "threshold": threshold,
        "temperature": temperature,
        "resolved_timm_model_name": resolved_timm_name,
        "target_layer_name": target_layer_name,
        "has_conv2d_target_layer": target_layer is not None,
    }

    smoke_rows.append(
        {
            "model_name": model_name,
            "aqpr_panel_role": str(model_row["aqpr_panel_role"]),
            "resolved_timm_model_name": resolved_timm_name,
            "checkpoint_path": str(checkpoint_path),
            "checkpoint_exists": checkpoint_path.exists(),
            "input_size": input_size,
            "batch_size": batch_size,
            "validation_selected_threshold": threshold,
            "temperature": temperature,
            "smoke_sample_id": str(smoke_case["sample_id"]),
            "smoke_patient_id": str(smoke_case["patient_id"]),
            "smoke_y_true": int(smoke_case["gon_binary"]),
            "smoke_logit": pred["logit"],
            "smoke_probability": pred["probability"],
            "smoke_temperature_scaled_probability": temperature_scaled_probability_from_logit(
                pred["logit"],
                temperature,
            ),
            "target_layer_name": target_layer_name if target_layer_name is not None else "",
            "has_conv2d_target_layer": target_layer is not None,
            "device": str(DEVICE),
        }
    )

    del model, checkpoint

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

aqpr_model_registry = pd.DataFrame(smoke_rows)
aqpr_model_registry.to_csv(AQPR_MODEL_REGISTRY_CSV, index=False)
aqpr_model_registry.to_csv(MODEL_LOADING_SMOKE_CSV, index=False)


# ---------------------------------------------------------------------
# Load primary model for primary Grad-CAM smoke test
# ---------------------------------------------------------------------

primary_row = frozen_aqpr_panel_manifest[
    frozen_aqpr_panel_manifest["model_name"] == PRIMARY_MODEL_NAME
].iloc[0]

SELECTED_MODEL_NAME = PRIMARY_MODEL_NAME
selected_model_name = PRIMARY_MODEL_NAME
selected_checkpoint_path = Path(str(primary_row["checkpoint_path"]))
SELECTED_INPUT_SIZE = int(primary_row["input_size"])
SELECTED_BATCH_SIZE = int(primary_row["batch_size"])
SELECTED_THRESHOLD = float(primary_row["validation_selected_threshold"])
SELECTED_TEMPERATURE = float(primary_row["temperature"])

selected_classifier, selected_checkpoint, SELECTED_TIMM_MODEL_NAME = load_aqpr_classifier(
    model_name=PRIMARY_MODEL_NAME,
    checkpoint_path=selected_checkpoint_path,
    input_size=SELECTED_INPUT_SIZE,
    device=DEVICE,
)

primary_classifier = selected_classifier
primary_checkpoint = selected_checkpoint

target_layer_name, target_layer = find_last_conv2d_layer(primary_classifier)

if target_layer is None:
    raise RuntimeError(f"No Conv2d target layer found for primary model {PRIMARY_MODEL_NAME}.")

primary_target_layer_name = target_layer_name
primary_target_layer = target_layer

smoke_image_tensor, smoke_rgb_square, smoke_rgb_resized = preprocess_classifier_image_from_path(
    image_path=smoke_case["classifier_image_path"],
    input_size=SELECTED_INPUT_SIZE,
)

gradcam_result = compute_gradcam_heatmap(
    model=primary_classifier,
    image_tensor=smoke_image_tensor,
    target_layer=primary_target_layer,
    target_mode="positive",
)

smoke_overlay = make_gradcam_overlay(
    image_rgb_resized=smoke_rgb_resized,
    heatmap=gradcam_result["heatmap"],
    alpha=0.40,
)

fig, axes = plt.subplots(1, 3, figsize=(12.0, 4.2))

axes[0].imshow(smoke_rgb_resized)
axes[0].set_title("HYGD input")
axes[0].axis("off")

axes[1].imshow(gradcam_result["heatmap"], cmap="jet")
axes[1].set_title("Primary Grad-CAM")
axes[1].axis("off")

axes[2].imshow(smoke_overlay)
axes[2].set_title("Overlay")
axes[2].axis("off")

fig.suptitle(
    f"{PRIMARY_MODEL_NAME} Grad-CAM smoke | "
    f"sample={smoke_case['sample_id']} | "
    f"p={gradcam_result['probability']:.3f}",
    fontsize=11,
)

fig.tight_layout()
display(fig)
fig.savefig(PRIMARY_GRADCAM_SMOKE_FIGURE, dpi=250, bbox_inches="tight")
plt.close(fig)

model_smoke_summary = {
    "status": "PASS",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_xai_dataset": "HYGD",
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "primary_timm_model_name": SELECTED_TIMM_MODEL_NAME,
    "primary_checkpoint": str(selected_checkpoint_path),
    "primary_input_size": int(SELECTED_INPUT_SIZE),
    "primary_batch_size": int(SELECTED_BATCH_SIZE),
    "primary_target_layer_name": primary_target_layer_name,
    "smoke_sample_id": str(smoke_case["sample_id"]),
    "smoke_patient_id": str(smoke_case["patient_id"]),
    "smoke_y_true": int(smoke_case["gon_binary"]),
    "smoke_primary_probability": float(gradcam_result["probability"]),
    "smoke_primary_logit": float(gradcam_result["logit"]),
    "heatmap_shape": list(gradcam_result["heatmap"].shape),
    "heatmap_min": float(np.min(gradcam_result["heatmap"])),
    "heatmap_max": float(np.max(gradcam_result["heatmap"])),
    "gradcam_smoke_figure": str(PRIMARY_GRADCAM_SMOKE_FIGURE),
    "hygd_strict_xai_images": int(len(hygd_strict_xai_df)),
    "hygd_strict_xai_patients": int(hygd_strict_xai_df["patient_id"].nunique(dropna=True)),
    "hygd_high_confidence_xai_images": int(len(hygd_high_confidence_xai_df)),
    "hygd_high_confidence_xai_patients": int(
        hygd_high_confidence_xai_df["patient_id"].nunique(dropna=True)
    ),
    "patient_level_external_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
}

with open(MODEL_LOADING_SMOKE_JSON, "w", encoding="utf-8") as f:
    json.dump(model_smoke_summary, f, indent=2)

print("HYGD AQPR XAI cohorts:")
print(cohort_summary.to_string(index=False))

print("\nAQPR model loading smoke test:")
print(aqpr_model_registry.to_string(index=False))

print("\nPrimary Grad-CAM smoke summary:")
print(json.dumps(model_smoke_summary, indent=2))

print(f"\nCohort summary: {EXPLAINABILITY_COHORT_SUMMARY_CSV}")
print(f"AQPR model registry: {AQPR_MODEL_REGISTRY_CSV}")
print(f"Model loading smoke CSV: {MODEL_LOADING_SMOKE_CSV}")
print(f"Model loading smoke JSON: {MODEL_LOADING_SMOKE_JSON}")
print(f"Primary Grad-CAM smoke figure: {PRIMARY_GRADCAM_SMOKE_FIGURE}")

if len(hygd_strict_xai_df) <= 0:
    raise RuntimeError("HYGD strict AQPR XAI cohort is empty.")

if len(hygd_high_confidence_xai_df) <= 0:
    raise RuntimeError("HYGD high-confidence AQPR XAI cohort is empty.")

if len(aqpr_model_registry) != 4:
    raise RuntimeError(f"Expected 4 AQPR model smoke rows, found {len(aqpr_model_registry)}")

if not aqpr_model_registry["checkpoint_exists"].all():
    raise RuntimeError("One or more AQPR checkpoints do not exist.")

if not np.isfinite(aqpr_model_registry["smoke_probability"]).all():
    raise RuntimeError("Non-finite smoke probabilities found.")

if not Path(PRIMARY_GRADCAM_SMOKE_FIGURE).exists():
    raise RuntimeError("Primary Grad-CAM smoke figure was not saved.")

if gradcam_result["heatmap"].shape[0] <= 1 or gradcam_result["heatmap"].shape[1] <= 1:
    raise RuntimeError("Invalid primary Grad-CAM heatmap shape.")

if not np.isfinite(gradcam_result["heatmap"]).all():
    raise RuntimeError("Non-finite Grad-CAM heatmap values found.")

print("Status: PASS")

In [ ]:
# Cell 3: Define AQPR multi-XAI framework and run primary-model smoke test

MULTIXAI_DIR = RESULTS_08_DIR / "multi_xai_smoke_tests"
MULTIXAI_DIR.mkdir(parents=True, exist_ok=True)

XAI_METHOD_REGISTRY_CSV = RESULTS_08_DIR / "notebook08_aqpr_xai_method_registry.csv"
XAI_METHOD_REGISTRY_JSON = RESULTS_08_DIR / "notebook08_aqpr_xai_method_registry.json"
MULTIXAI_SMOKE_JSON = RESULTS_08_DIR / "notebook08_aqpr_multixai_smoke_test_summary.json"
MULTIXAI_SMOKE_FIGURE = MULTIXAI_DIR / "notebook08_aqpr_multixai_smoke_test_panel.png"
MULTIXAI_SMOKE_FIGURE_PDF = MULTIXAI_DIR / "notebook08_aqpr_multixai_smoke_test_panel.pdf"
MULTIXAI_SMOKE_FIGURE_SVG = MULTIXAI_DIR / "notebook08_aqpr_multixai_smoke_test_panel.svg"

ENERGY_SMOKE_CSV = MULTIXAI_DIR / "notebook08_aqpr_multixai_energy_fraction_smoke_test.csv"
OCCLUSION_SMOKE_CSV = MULTIXAI_DIR / "notebook08_aqpr_anatomical_occlusion_smoke_test.csv"

if "selected_classifier" not in globals():
    raise RuntimeError("Cell 2 must be executed before Cell 3: selected_classifier is missing.")

if "primary_target_layer" not in globals():
    raise RuntimeError("Cell 2 must be executed before Cell 3: primary_target_layer is missing.")

if "hygd_strict_xai_df" not in globals():
    raise RuntimeError("Cell 2 must be executed before Cell 3: hygd_strict_xai_df is missing.")

if len(hygd_strict_xai_df) == 0:
    raise RuntimeError("HYGD strict AQPR XAI cohort is empty.")


# ---------------------------------------------------------------------
# XAI method registry
# ---------------------------------------------------------------------

XAI_METHOD_REGISTRY = [
    {
        "method_name": "GradCAM",
        "method_family": "class_activation_mapping",
        "role": "baseline_cam",
        "target": "positive_class_logit",
        "primary_statistics": False,
        "sensitivity_statistics": True,
        "applies_to": "models_with_spatial_feature_maps",
        "rationale": "Baseline CAM method for coarse class-discriminative localization.",
    },
    {
        "method_name": "GradCAMPlusPlus",
        "method_family": "class_activation_mapping",
        "role": "primary_cam",
        "target": "positive_class_logit",
        "primary_statistics": True,
        "sensitivity_statistics": True,
        "applies_to": "models_with_spatial_feature_maps",
        "rationale": "Primary CAM-style localization method for OD/OC enrichment analysis.",
    },
    {
        "method_name": "IntegratedGradients",
        "method_family": "input_gradient_attribution",
        "role": "primary_input_attribution",
        "target": "positive_class_logit",
        "primary_statistics": True,
        "sensitivity_statistics": True,
        "applies_to": "all_differentiable_panel_models",
        "rationale": "Input-space attribution independent of the final convolutional feature map.",
    },
    {
        "method_name": "AnatomicalOcclusion",
        "method_family": "anatomy_guided_perturbation",
        "role": "primary_perturbation",
        "target": "positive_class_probability",
        "primary_statistics": True,
        "sensitivity_statistics": True,
        "applies_to": "all_panel_models",
        "rationale": "Perturbation-based test of whether OD/OC regions causally affect model output.",
    },
]

xai_method_registry = pd.DataFrame(XAI_METHOD_REGISTRY)
xai_method_registry.to_csv(XAI_METHOD_REGISTRY_CSV, index=False)

xai_registry_json = {
    "notebook": "08_anatomical_explainability_validation",
    "xai_design": "AQPR multi-method anatomical validation",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_anatomical_dataset": "HYGD",
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "primary_cohort": "hygd_external_strict_anatomical",
    "secondary_cohort": "hygd_external_high_confidence_anatomical",
    "primary_xai_methods": [
        "GradCAMPlusPlus",
        "IntegratedGradients",
        "AnatomicalOcclusion",
    ],
    "sensitivity_xai_methods": [
        "GradCAM",
    ],
    "primary_anatomical_endpoint": "OD_plus_OC_heatmap_enrichment_vs_matched_random_controls",
    "perturbation_endpoint": "probability_drop_after_OD_plus_OC_occlusion_vs_matched_random_controls",
    "patient_level_external_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
    "important_note": (
        "Visual heatmaps are not treated as sufficient evidence. "
        "Each method must be quantified against OD/OC proxy masks and matched random controls."
    ),
    "methods": XAI_METHOD_REGISTRY,
}

with open(XAI_METHOD_REGISTRY_JSON, "w", encoding="utf-8") as f:
    json.dump(xai_registry_json, f, indent=2)


# ---------------------------------------------------------------------
# Helper functions for heatmaps and perturbation
# ---------------------------------------------------------------------

def rgb_to_classifier_tensor(image_rgb):
    image_float = image_rgb.astype(np.float32) / 255.0
    image_float = (image_float - IMAGENET_MEAN) / IMAGENET_STD
    image_chw = np.transpose(image_float, (2, 0, 1))
    return torch.from_numpy(np.ascontiguousarray(image_chw)).float()


def predict_probability_from_rgb(model, image_rgb):
    model.eval()

    image_tensor = rgb_to_classifier_tensor(image_rgb).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = flatten_logits(model(image_tensor))
        probability = torch.sigmoid(logits)[0].detach().cpu().item()

    return float(probability)


def normalize_heatmap(heatmap):
    heatmap = np.asarray(heatmap, dtype=np.float32)

    heatmap = np.nan_to_num(
        heatmap,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    heatmap = np.clip(heatmap, 0.0, None)

    h_min = float(np.min(heatmap))
    h_max = float(np.max(heatmap))

    if h_max <= h_min:
        return np.zeros_like(heatmap, dtype=np.float32)

    return ((heatmap - h_min) / (h_max - h_min + 1e-8)).astype(np.float32)


def resize_heatmap_to_input(heatmap, input_size):
    heatmap_resized = cv2.resize(
        np.asarray(heatmap, dtype=np.float32),
        (int(input_size), int(input_size)),
        interpolation=cv2.INTER_LINEAR,
    )

    return normalize_heatmap(heatmap_resized)


def compute_gradcam_plus_plus_heatmap(model, image_tensor, target_layer, target_mode="positive"):
    model.eval()
    model.zero_grad(set_to_none=True)

    if image_tensor.ndim == 3:
        image_tensor = image_tensor.unsqueeze(0)

    image_tensor = image_tensor.to(DEVICE)

    with GradCAMCapture(target_layer) as capture:
        logits = flatten_logits(model(image_tensor))
        probs = torch.sigmoid(logits)

        if target_mode == "positive":
            score = logits.sum()
        elif target_mode == "negative":
            score = (-logits).sum()
        elif target_mode == "predicted":
            signs = torch.where(probs >= 0.5, 1.0, -1.0)
            score = (signs * logits).sum()
        else:
            raise ValueError(f"Unknown target_mode: {target_mode}")

        score.backward()

        activations = capture.activations
        gradients = capture.gradients

    if activations is None or gradients is None:
        raise RuntimeError("Grad-CAM++ activations or gradients were not captured.")

    grads_power_2 = gradients.pow(2)
    grads_power_3 = gradients.pow(3)

    spatial_sum = (activations * grads_power_3).sum(dim=(2, 3), keepdim=True)

    alpha = grads_power_2 / (
        2.0 * grads_power_2
        + spatial_sum
        + 1e-8
    )

    positive_gradients = F.relu(gradients)
    weights = (alpha * positive_gradients).sum(dim=(2, 3), keepdim=True)

    cam = F.relu((weights * activations).sum(dim=1))
    cam = normalize_heatmap(cam[0].detach().cpu().numpy())

    return {
        "logit": float(logits[0].detach().cpu()),
        "probability": float(probs[0].detach().cpu()),
        "heatmap": cam,
        "activation_shape": list(activations.shape),
        "gradient_shape": list(gradients.shape),
    }


def compute_integrated_gradients_heatmap(
    model,
    image_tensor,
    n_steps=16,
    baseline_mode="zero",
    target_mode="positive",
):
    model.eval()
    model.zero_grad(set_to_none=True)

    if image_tensor.ndim == 3:
        image_tensor = image_tensor.unsqueeze(0)

    image_tensor = image_tensor.to(DEVICE)

    if baseline_mode == "zero":
        baseline = torch.zeros_like(image_tensor)
    elif baseline_mode == "blur":
        image_np = image_tensor[0].detach().cpu().numpy()
        image_hwc = np.transpose(image_np, (1, 2, 0))
        blurred = cv2.GaussianBlur(image_hwc, (31, 31), 0)
        baseline = torch.from_numpy(
            np.transpose(blurred, (2, 0, 1))
        ).float().unsqueeze(0).to(DEVICE)
    else:
        raise ValueError(f"Unknown baseline_mode: {baseline_mode}")

    total_gradients = torch.zeros_like(image_tensor)

    alphas = torch.linspace(
        0.0,
        1.0,
        steps=int(n_steps),
        device=DEVICE,
    )

    for alpha in alphas:
        interpolated = baseline + alpha * (image_tensor - baseline)
        interpolated.requires_grad_(True)

        logits = flatten_logits(model(interpolated))
        probs = torch.sigmoid(logits)

        if target_mode == "positive":
            score = logits.sum()
        elif target_mode == "negative":
            score = (-logits).sum()
        elif target_mode == "predicted":
            signs = torch.where(probs >= 0.5, 1.0, -1.0)
            score = (signs * logits).sum()
        else:
            raise ValueError(f"Unknown target_mode: {target_mode}")

        gradients = torch.autograd.grad(
            outputs=score,
            inputs=interpolated,
            retain_graph=False,
            create_graph=False,
        )[0]

        total_gradients += gradients.detach()

    average_gradients = total_gradients / float(n_steps)
    attributions = (image_tensor - baseline) * average_gradients

    attribution_map = F.relu(attributions).sum(dim=1)
    heatmap = normalize_heatmap(attribution_map[0].detach().cpu().numpy())

    with torch.no_grad():
        final_logits = flatten_logits(model(image_tensor))
        final_probability = torch.sigmoid(final_logits)[0].detach().cpu().item()

    return {
        "probability": float(final_probability),
        "heatmap": heatmap,
        "n_steps": int(n_steps),
        "baseline_mode": baseline_mode,
    }


def load_od_oc_masks_for_input(mask_path, input_size):
    mask_path = Path(str(mask_path))

    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    if mask is None:
        raise RuntimeError(f"Unreadable OD/OC mask: {mask_path}")

    mask_resized = cv2.resize(
        mask,
        (int(input_size), int(input_size)),
        interpolation=cv2.INTER_NEAREST,
    )

    od_mask = mask_resized > 0
    oc_mask = mask_resized == 2
    rim_mask = mask_resized == 1
    od_plus_oc_mask = od_mask.copy()

    return {
        "mask_resized": mask_resized,
        "od_mask": od_mask,
        "oc_mask": oc_mask,
        "rim_mask": rim_mask,
        "od_plus_oc_mask": od_plus_oc_mask,
    }


def heatmap_energy_inside_mask(heatmap, binary_mask):
    heatmap = np.asarray(heatmap, dtype=np.float32)
    binary_mask = np.asarray(binary_mask).astype(bool)

    if heatmap.shape != binary_mask.shape:
        raise ValueError(f"Shape mismatch: heatmap={heatmap.shape}, mask={binary_mask.shape}")

    positive_heatmap = np.clip(heatmap, 0.0, None)
    total_energy = float(positive_heatmap.sum())

    if total_energy <= 0:
        return np.nan

    inside_energy = float(positive_heatmap[binary_mask].sum())

    return inside_energy / total_energy


def perturb_rgb_with_mask(image_rgb, binary_mask, fill_mode="outside_mean"):
    perturbed = image_rgb.copy()
    binary_mask = np.asarray(binary_mask).astype(bool)

    if binary_mask.sum() == 0:
        return perturbed

    if fill_mode == "outside_mean":
        outside_pixels = image_rgb[~binary_mask]

        if len(outside_pixels) == 0:
            fill_value = np.array([0, 0, 0], dtype=np.float32)
        else:
            fill_value = outside_pixels.mean(axis=0)

        perturbed[binary_mask] = np.clip(fill_value, 0, 255).astype(np.uint8)

    elif fill_mode == "blur":
        blurred = cv2.GaussianBlur(image_rgb, (31, 31), 0)
        perturbed[binary_mask] = blurred[binary_mask]

    elif fill_mode == "black":
        perturbed[binary_mask] = 0

    else:
        raise ValueError(f"Unknown fill_mode: {fill_mode}")

    return perturbed


def compute_anatomical_occlusion_smoke(model, image_rgb, masks, fill_mode="outside_mean"):
    original_probability = predict_probability_from_rgb(model, image_rgb)

    rows = []

    for region_name, region_mask in [
        ("OD", masks["od_mask"]),
        ("OC", masks["oc_mask"]),
        ("Rim", masks["rim_mask"]),
        ("OD_plus_OC", masks["od_plus_oc_mask"]),
    ]:
        perturbed_rgb = perturb_rgb_with_mask(
            image_rgb=image_rgb,
            binary_mask=region_mask,
            fill_mode=fill_mode,
        )

        perturbed_probability = predict_probability_from_rgb(
            model=model,
            image_rgb=perturbed_rgb,
        )

        rows.append(
            {
                "region": region_name,
                "mask_pixels": int(region_mask.sum()),
                "original_probability": float(original_probability),
                "perturbed_probability": float(perturbed_probability),
                "probability_drop": float(original_probability - perturbed_probability),
                "fill_mode": fill_mode,
            }
        )

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------
# Multi-XAI smoke test on one HYGD external case
# ---------------------------------------------------------------------

smoke_case_xai = (
    hygd_strict_xai_df
    .copy()
    .sort_values(["gon_binary", "sample_id"], ascending=[False, True])
    .iloc[0]
)

smoke_image_tensor, smoke_rgb_square, smoke_rgb_resized = preprocess_classifier_image_from_path(
    image_path=smoke_case_xai["classifier_image_path"],
    input_size=SELECTED_INPUT_SIZE,
)

smoke_masks = load_od_oc_masks_for_input(
    mask_path=smoke_case_xai["mask_512_path"],
    input_size=SELECTED_INPUT_SIZE,
)

gradcam_result = compute_gradcam_heatmap(
    model=selected_classifier,
    image_tensor=smoke_image_tensor,
    target_layer=primary_target_layer,
    target_mode="positive",
)

gradcam_pp_result = compute_gradcam_plus_plus_heatmap(
    model=selected_classifier,
    image_tensor=smoke_image_tensor,
    target_layer=primary_target_layer,
    target_mode="positive",
)

ig_result = compute_integrated_gradients_heatmap(
    model=selected_classifier,
    image_tensor=smoke_image_tensor,
    n_steps=16,
    baseline_mode="zero",
    target_mode="positive",
)

gradcam_heatmap_input = resize_heatmap_to_input(
    gradcam_result["heatmap"],
    SELECTED_INPUT_SIZE,
)

gradcam_pp_heatmap_input = resize_heatmap_to_input(
    gradcam_pp_result["heatmap"],
    SELECTED_INPUT_SIZE,
)

ig_heatmap_input = resize_heatmap_to_input(
    ig_result["heatmap"],
    SELECTED_INPUT_SIZE,
)

gradcam_overlay = make_gradcam_overlay(
    image_rgb_resized=smoke_rgb_resized,
    heatmap=gradcam_heatmap_input,
    alpha=0.40,
)

gradcam_pp_overlay = make_gradcam_overlay(
    image_rgb_resized=smoke_rgb_resized,
    heatmap=gradcam_pp_heatmap_input,
    alpha=0.40,
)

ig_overlay = make_gradcam_overlay(
    image_rgb_resized=smoke_rgb_resized,
    heatmap=ig_heatmap_input,
    alpha=0.40,
)

occlusion_smoke = compute_anatomical_occlusion_smoke(
    model=selected_classifier,
    image_rgb=smoke_rgb_resized,
    masks=smoke_masks,
    fill_mode="outside_mean",
)

occlusion_smoke.to_csv(OCCLUSION_SMOKE_CSV, index=False)

energy_rows = []

for method_name, heatmap in [
    ("GradCAM", gradcam_heatmap_input),
    ("GradCAMPlusPlus", gradcam_pp_heatmap_input),
    ("IntegratedGradients", ig_heatmap_input),
]:
    energy_rows.append(
        {
            "method_name": method_name,
            "od_energy_fraction": heatmap_energy_inside_mask(heatmap, smoke_masks["od_mask"]),
            "oc_energy_fraction": heatmap_energy_inside_mask(heatmap, smoke_masks["oc_mask"]),
            "rim_energy_fraction": heatmap_energy_inside_mask(heatmap, smoke_masks["rim_mask"]),
            "od_plus_oc_energy_fraction": heatmap_energy_inside_mask(
                heatmap,
                smoke_masks["od_plus_oc_mask"],
            ),
            "outside_od_energy_fraction": heatmap_energy_inside_mask(
                heatmap,
                ~smoke_masks["od_mask"],
            ),
        }
    )

energy_smoke = pd.DataFrame(energy_rows)
energy_smoke.to_csv(ENERGY_SMOKE_CSV, index=False)

mask_display = np.zeros((*smoke_masks["mask_resized"].shape, 3), dtype=np.uint8)
mask_display[smoke_masks["od_mask"]] = [0, 255, 0]
mask_display[smoke_masks["oc_mask"]] = [255, 0, 0]

fig, axes = plt.subplots(2, 4, figsize=(16.0, 8.2))

axes[0, 0].imshow(smoke_rgb_resized)
axes[0, 0].set_title("HYGD input")
axes[0, 0].axis("off")

axes[0, 1].imshow(mask_display)
axes[0, 1].set_title("OD/OC proxy")
axes[0, 1].axis("off")

axes[0, 2].imshow(gradcam_heatmap_input, cmap="jet")
axes[0, 2].set_title("Grad-CAM")
axes[0, 2].axis("off")

axes[0, 3].imshow(gradcam_overlay)
axes[0, 3].set_title("Grad-CAM overlay")
axes[0, 3].axis("off")

axes[1, 0].imshow(gradcam_pp_heatmap_input, cmap="jet")
axes[1, 0].set_title("Grad-CAM++")
axes[1, 0].axis("off")

axes[1, 1].imshow(gradcam_pp_overlay)
axes[1, 1].set_title("Grad-CAM++ overlay")
axes[1, 1].axis("off")

axes[1, 2].imshow(ig_heatmap_input, cmap="jet")
axes[1, 2].set_title("Integrated Gradients")
axes[1, 2].axis("off")

axes[1, 3].imshow(ig_overlay)
axes[1, 3].set_title("IG overlay")
axes[1, 3].axis("off")

fig.suptitle(
    f"AQPR multi-XAI smoke test | {PRIMARY_MODEL_NAME} | "
    f"sample={smoke_case_xai['sample_id']} | "
    f"p={gradcam_result['probability']:.3f}",
    fontsize=12,
    fontweight="bold",
)

fig.tight_layout()
display(fig)

fig.savefig(MULTIXAI_SMOKE_FIGURE, dpi=300, bbox_inches="tight")
fig.savefig(MULTIXAI_SMOKE_FIGURE_PDF, bbox_inches="tight")
fig.savefig(MULTIXAI_SMOKE_FIGURE_SVG, bbox_inches="tight")
plt.close(fig)

multixai_smoke_summary = {
    "status": "PASS",
    "notebook": "08_anatomical_explainability_validation",
    "analysis": "AQPR multi-XAI smoke test",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_anatomical_dataset": "HYGD",
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "smoke_sample_id": str(smoke_case_xai["sample_id"]),
    "smoke_patient_id": str(smoke_case_xai["patient_id"]),
    "smoke_gon_binary": int(smoke_case_xai["gon_binary"]),
    "target_mode": "positive_class_logit",
    "input_size": int(SELECTED_INPUT_SIZE),
    "target_layer_name": str(primary_target_layer_name),
    "gradcam_probability": float(gradcam_result["probability"]),
    "gradcam_pp_probability": float(gradcam_pp_result["probability"]),
    "integrated_gradients_probability": float(ig_result["probability"]),
    "integrated_gradients_steps": int(ig_result["n_steps"]),
    "od_pixels": int(smoke_masks["od_mask"].sum()),
    "oc_pixels": int(smoke_masks["oc_mask"].sum()),
    "rim_pixels": int(smoke_masks["rim_mask"].sum()),
    "xai_method_registry_csv": str(XAI_METHOD_REGISTRY_CSV),
    "xai_method_registry_json": str(XAI_METHOD_REGISTRY_JSON),
    "energy_smoke_csv": str(ENERGY_SMOKE_CSV),
    "occlusion_smoke_csv": str(OCCLUSION_SMOKE_CSV),
    "multixai_smoke_figure_png": str(MULTIXAI_SMOKE_FIGURE),
    "multixai_smoke_figure_pdf": str(MULTIXAI_SMOKE_FIGURE_PDF),
    "multixai_smoke_figure_svg": str(MULTIXAI_SMOKE_FIGURE_SVG),
    "patient_level_external_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
}

with open(MULTIXAI_SMOKE_JSON, "w", encoding="utf-8") as f:
    json.dump(multixai_smoke_summary, f, indent=2)

print("AQPR XAI method registry:")
print(
    xai_method_registry[
        [
            "method_name",
            "method_family",
            "role",
            "primary_statistics",
            "sensitivity_statistics",
            "applies_to",
        ]
    ].to_string(index=False)
)

print("\nSmoke-test anatomical energy fractions:")
print(energy_smoke.to_string(index=False))

print("\nSmoke-test anatomical occlusion:")
print(occlusion_smoke.to_string(index=False))

print("\nAQPR multi-XAI smoke summary:")
print(json.dumps(multixai_smoke_summary, indent=2))

print(f"\nXAI method registry CSV: {XAI_METHOD_REGISTRY_CSV}")
print(f"XAI method registry JSON: {XAI_METHOD_REGISTRY_JSON}")
print(f"Energy smoke CSV: {ENERGY_SMOKE_CSV}")
print(f"Occlusion smoke CSV: {OCCLUSION_SMOKE_CSV}")
print(f"Multi-XAI smoke figure: {MULTIXAI_SMOKE_FIGURE}")
print(f"Multi-XAI smoke summary JSON: {MULTIXAI_SMOKE_JSON}")

required_outputs = [
    XAI_METHOD_REGISTRY_CSV,
    XAI_METHOD_REGISTRY_JSON,
    ENERGY_SMOKE_CSV,
    OCCLUSION_SMOKE_CSV,
    MULTIXAI_SMOKE_FIGURE,
    MULTIXAI_SMOKE_FIGURE_PDF,
    MULTIXAI_SMOKE_FIGURE_SVG,
    MULTIXAI_SMOKE_JSON,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing AQPR multi-XAI smoke-test outputs: {missing_outputs}")

for col in [
    "od_energy_fraction",
    "oc_energy_fraction",
    "rim_energy_fraction",
    "od_plus_oc_energy_fraction",
    "outside_od_energy_fraction",
]:
    if not np.isfinite(energy_smoke[col]).all():
        raise RuntimeError(f"Non-finite {col} found in smoke test.")

if not np.isfinite(occlusion_smoke["probability_drop"]).all():
    raise RuntimeError("Non-finite occlusion probability drop found in smoke test.")

if not np.isfinite(gradcam_result["heatmap"]).all():
    raise RuntimeError("Non-finite Grad-CAM heatmap values found.")

if not np.isfinite(gradcam_pp_result["heatmap"]).all():
    raise RuntimeError("Non-finite Grad-CAM++ heatmap values found.")

if not np.isfinite(ig_result["heatmap"]).all():
    raise RuntimeError("Non-finite Integrated Gradients heatmap values found.")

print("Status: PASS")

In [ ]:
# Cell 4: Define AQPR anatomical quantification metrics and matched random-control masks

from tqdm.auto import tqdm

ANATOMICAL_METRIC_DIR = RESULTS_08_DIR / "anatomical_metric_smoke_tests"
ANATOMICAL_METRIC_DIR.mkdir(parents=True, exist_ok=True)

ANATOMICAL_METRIC_CONFIG_JSON = RESULTS_08_DIR / "notebook08_aqpr_anatomical_metric_config.json"
RANDOM_CONTROL_SMOKE_CSV = ANATOMICAL_METRIC_DIR / "notebook08_aqpr_random_control_smoke_test.csv"
ANATOMICAL_ENRICHMENT_SMOKE_CSV = ANATOMICAL_METRIC_DIR / "notebook08_aqpr_anatomical_enrichment_smoke_test.csv"
ANATOMICAL_SMOKE_FIGURE = ANATOMICAL_METRIC_DIR / "notebook08_aqpr_anatomical_random_control_smoke_test.png"
ANATOMICAL_SMOKE_FIGURE_PDF = ANATOMICAL_METRIC_DIR / "notebook08_aqpr_anatomical_random_control_smoke_test.pdf"

# For the full analysis, 128 is strong enough and computationally manageable.
# The smoke test uses the same policy.
N_RANDOM_CONTROLS = 128
RANDOM_CONTROL_MAX_ATTEMPTS = 5000
RANDOM_CONTROL_MIN_FOV_FRACTION = 0.985
RANDOM_CONTROL_MAX_IOU_WITH_TARGET = 0.03
RANDOM_CONTROL_SEED = SEED

PRIMARY_CAM_METHOD = "GradCAM"
PRIMARY_ATTRIBUTION_METHOD = "IntegratedGradients"
SENSITIVITY_CAM_METHOD = "GradCAMPlusPlus"

PRIMARY_ANATOMICAL_REGION = "OD_plus_OC"
ANATOMICAL_REGIONS = ["OD", "OC", "Rim", "OD_plus_OC"]

AQPR_XAI_METHOD_POLICY = {
    "primary_cam_method": PRIMARY_CAM_METHOD,
    "primary_attribution_method": PRIMARY_ATTRIBUTION_METHOD,
    "sensitivity_cam_method": SENSITIVITY_CAM_METHOD,
    "reason_for_gradcampp_demotion": (
        "Grad-CAM++ was retained as a sensitivity method because the smoke test showed "
        "border-dominant activation in the primary model on HYGD."
    ),
}


def compute_fov_mask_from_rgb(image_rgb, threshold=8):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

    fov_mask = gray > threshold

    kernel = np.ones((7, 7), np.uint8)

    fov_mask = cv2.morphologyEx(
        fov_mask.astype(np.uint8),
        cv2.MORPH_CLOSE,
        kernel,
    ).astype(bool)

    fov_mask = cv2.morphologyEx(
        fov_mask.astype(np.uint8),
        cv2.MORPH_OPEN,
        kernel,
    ).astype(bool)

    return fov_mask


def binary_iou(mask_a, mask_b):
    mask_a = np.asarray(mask_a).astype(bool)
    mask_b = np.asarray(mask_b).astype(bool)

    intersection = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()

    if union == 0:
        return 0.0

    return float(intersection / union)


def mask_centroid(binary_mask):
    binary_mask = np.asarray(binary_mask).astype(bool)

    ys, xs = np.where(binary_mask)

    if len(xs) == 0:
        return None

    return float(ys.mean()), float(xs.mean())


def translate_binary_mask_to_center(source_mask, new_center_y, new_center_x):
    source_mask = np.asarray(source_mask).astype(bool)

    output = np.zeros_like(source_mask, dtype=bool)
    coords = np.argwhere(source_mask)

    if len(coords) == 0:
        return output

    current_center_y = coords[:, 0].mean()
    current_center_x = coords[:, 1].mean()

    shift_y = int(round(new_center_y - current_center_y))
    shift_x = int(round(new_center_x - current_center_x))

    new_y = coords[:, 0] + shift_y
    new_x = coords[:, 1] + shift_x

    valid = (
        (new_y >= 0)
        & (new_y < source_mask.shape[0])
        & (new_x >= 0)
        & (new_x < source_mask.shape[1])
    )

    output[new_y[valid], new_x[valid]] = True

    return output


def generate_matched_random_masks(
    target_mask,
    fov_mask,
    exclusion_mask=None,
    n_controls=N_RANDOM_CONTROLS,
    rng=None,
    max_attempts=RANDOM_CONTROL_MAX_ATTEMPTS,
    min_fov_fraction=RANDOM_CONTROL_MIN_FOV_FRACTION,
    max_iou_with_target=RANDOM_CONTROL_MAX_IOU_WITH_TARGET,
):
    target_mask = np.asarray(target_mask).astype(bool)
    fov_mask = np.asarray(fov_mask).astype(bool)

    if exclusion_mask is None:
        exclusion_mask = target_mask.copy()
    else:
        exclusion_mask = np.asarray(exclusion_mask).astype(bool)

    if rng is None:
        rng = np.random.default_rng(RANDOM_CONTROL_SEED)

    if target_mask.sum() == 0:
        raise RuntimeError("Target mask is empty; cannot generate matched random controls.")

    target_centroid = mask_centroid(target_mask)

    if target_centroid is None:
        raise RuntimeError("Target mask centroid is undefined.")

    valid_center_mask = fov_mask & (~exclusion_mask)

    candidate_y, candidate_x = np.where(valid_center_mask)

    if len(candidate_y) == 0:
        raise RuntimeError("No valid candidate centers for random controls.")

    controls = []
    audit_rows = []
    attempts = 0

    while len(controls) < int(n_controls) and attempts < int(max_attempts):
        attempts += 1

        idx = int(rng.integers(0, len(candidate_y)))
        center_y = int(candidate_y[idx])
        center_x = int(candidate_x[idx])

        candidate_mask = translate_binary_mask_to_center(
            source_mask=target_mask,
            new_center_y=center_y,
            new_center_x=center_x,
        )

        if candidate_mask.sum() == 0:
            continue

        fov_fraction = float(candidate_mask[fov_mask].sum() / candidate_mask.sum())
        iou_with_target = binary_iou(candidate_mask, target_mask)
        overlap_with_exclusion = float(np.logical_and(candidate_mask, exclusion_mask).sum() / candidate_mask.sum())

        accepted = (
            fov_fraction >= float(min_fov_fraction)
            and iou_with_target <= float(max_iou_with_target)
            and overlap_with_exclusion <= float(max_iou_with_target)
        )

        audit_rows.append(
            {
                "attempt": int(attempts),
                "accepted": bool(accepted),
                "center_y": center_y,
                "center_x": center_x,
                "mask_pixels": int(candidate_mask.sum()),
                "fov_fraction": fov_fraction,
                "iou_with_target": iou_with_target,
                "overlap_with_exclusion": overlap_with_exclusion,
            }
        )

        if accepted:
            controls.append(candidate_mask)

    if len(controls) < int(n_controls):
        raise RuntimeError(
            f"Only generated {len(controls)} random controls out of {n_controls} "
            f"after {max_attempts} attempts."
        )

    return controls, pd.DataFrame(audit_rows)


def heatmap_energy_statistics(heatmap, region_mask, random_masks):
    heatmap = normalize_heatmap(heatmap)
    region_mask = np.asarray(region_mask).astype(bool)

    observed_fraction = heatmap_energy_inside_mask(
        heatmap=heatmap,
        binary_mask=region_mask,
    )

    random_fractions = [
        heatmap_energy_inside_mask(
            heatmap=heatmap,
            binary_mask=random_mask,
        )
        for random_mask in random_masks
    ]

    random_fractions = np.asarray(random_fractions, dtype=float)
    random_fractions = random_fractions[np.isfinite(random_fractions)]

    if len(random_fractions) == 0:
        random_mean = np.nan
        random_std = np.nan
        observed_minus_random_mean = np.nan
        enrichment_ratio = np.nan
        empirical_p_greater_equal = np.nan
    else:
        random_mean = float(np.mean(random_fractions))
        random_std = float(np.std(random_fractions, ddof=1)) if len(random_fractions) > 1 else 0.0
        observed_minus_random_mean = float(observed_fraction - random_mean)
        enrichment_ratio = float(observed_fraction / (random_mean + 1e-8))
        empirical_p_greater_equal = float(
            (np.sum(random_fractions >= observed_fraction) + 1)
            / (len(random_fractions) + 1)
        )

    return {
        "observed_energy_fraction": float(observed_fraction),
        "random_energy_fraction_mean": random_mean,
        "random_energy_fraction_std": random_std,
        "observed_minus_random_mean": observed_minus_random_mean,
        "enrichment_ratio": enrichment_ratio,
        "empirical_p_greater_equal": empirical_p_greater_equal,
        "n_random_controls": int(len(random_fractions)),
    }


def build_region_masks_from_mask_path(mask_path, input_size):
    masks = load_od_oc_masks_for_input(
        mask_path=mask_path,
        input_size=input_size,
    )

    region_masks = {
        "OD": masks["od_mask"],
        "OC": masks["oc_mask"],
        "Rim": masks["rim_mask"],
        "OD_plus_OC": masks["od_plus_oc_mask"],
    }

    return masks, region_masks


def make_random_control_visualization(image_rgb, target_mask, random_masks, max_controls_to_show=12):
    vis = image_rgb.copy()

    target_outline = target_mask.astype(np.uint8)
    target_contours, _ = cv2.findContours(
        target_outline,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE,
    )

    cv2.drawContours(vis, target_contours, -1, (0, 255, 0), 2)

    for random_mask in random_masks[:max_controls_to_show]:
        control_uint8 = random_mask.astype(np.uint8)
        contours, _ = cv2.findContours(
            control_uint8,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE,
        )

        cv2.drawContours(vis, contours, -1, (255, 255, 0), 1)

    return vis


def stable_rng_for_case(*parts):
    text = "|".join([str(part) for part in parts])
    seed_value = abs(hash(text)) % (2**32 - 1)
    return np.random.default_rng(seed_value)


# ---------------------------------------------------------------------
# Smoke test: random controls + anatomical enrichment on the same primary case
# ---------------------------------------------------------------------

if "smoke_case_xai" not in globals():
    smoke_case_xai = (
        hygd_strict_xai_df
        .copy()
        .sort_values(["gon_binary", "sample_id"], ascending=[False, True])
        .iloc[0]
    )

smoke_tensor, smoke_rgb_square, smoke_rgb_resized = preprocess_classifier_image_from_path(
    image_path=smoke_case_xai["classifier_image_path"],
    input_size=SELECTED_INPUT_SIZE,
)

smoke_masks, smoke_region_masks = build_region_masks_from_mask_path(
    mask_path=smoke_case_xai["mask_512_path"],
    input_size=SELECTED_INPUT_SIZE,
)

smoke_fov_mask = compute_fov_mask_from_rgb(
    image_rgb=smoke_rgb_resized,
    threshold=8,
)

smoke_target_mask = smoke_region_masks[PRIMARY_ANATOMICAL_REGION]
smoke_exclusion_mask = smoke_region_masks["OD_plus_OC"]

smoke_rng = stable_rng_for_case(
    "cell4",
    smoke_case_xai["sample_id"],
    PRIMARY_ANATOMICAL_REGION,
    RANDOM_CONTROL_SEED,
)

smoke_random_masks, smoke_random_audit = generate_matched_random_masks(
    target_mask=smoke_target_mask,
    fov_mask=smoke_fov_mask,
    exclusion_mask=smoke_exclusion_mask,
    n_controls=N_RANDOM_CONTROLS,
    rng=smoke_rng,
)

smoke_random_audit["sample_id"] = str(smoke_case_xai["sample_id"])
smoke_random_audit["region"] = PRIMARY_ANATOMICAL_REGION
smoke_random_audit.to_csv(RANDOM_CONTROL_SMOKE_CSV, index=False)

# Use Grad-CAM as primary CAM method after smoke-test instability of Grad-CAM++.
smoke_gradcam_result = compute_gradcam_heatmap(
    model=selected_classifier,
    image_tensor=smoke_tensor,
    target_layer=primary_target_layer,
    target_mode="positive",
)

smoke_gradcam_heatmap = resize_heatmap_to_input(
    smoke_gradcam_result["heatmap"],
    SELECTED_INPUT_SIZE,
)

smoke_enrichment = heatmap_energy_statistics(
    heatmap=smoke_gradcam_heatmap,
    region_mask=smoke_target_mask,
    random_masks=smoke_random_masks,
)

smoke_enrichment_row = {
    "sample_id": str(smoke_case_xai["sample_id"]),
    "patient_id": str(smoke_case_xai["patient_id"]),
    "model_name": PRIMARY_MODEL_NAME,
    "xai_method": PRIMARY_CAM_METHOD,
    "region": PRIMARY_ANATOMICAL_REGION,
    "gon_binary": int(smoke_case_xai["gon_binary"]),
    "model_probability": float(smoke_gradcam_result["probability"]),
    "mask_pixels": int(smoke_target_mask.sum()),
    "fov_pixels": int(smoke_fov_mask.sum()),
    **smoke_enrichment,
}

smoke_enrichment_df = pd.DataFrame([smoke_enrichment_row])
smoke_enrichment_df.to_csv(ANATOMICAL_ENRICHMENT_SMOKE_CSV, index=False)

random_control_overlay = make_random_control_visualization(
    image_rgb=smoke_rgb_resized,
    target_mask=smoke_target_mask,
    random_masks=smoke_random_masks,
    max_controls_to_show=12,
)

heatmap_overlay = make_gradcam_overlay(
    image_rgb_resized=smoke_rgb_resized,
    heatmap=smoke_gradcam_heatmap,
    alpha=0.40,
)

fig, axes = plt.subplots(1, 4, figsize=(16.0, 4.2))

axes[0].imshow(smoke_rgb_resized)
axes[0].set_title("HYGD input")
axes[0].axis("off")

axes[1].imshow(smoke_target_mask, cmap="gray")
axes[1].set_title("OD+OC target")
axes[1].axis("off")

axes[2].imshow(random_control_overlay)
axes[2].set_title("Target + random controls")
axes[2].axis("off")

axes[3].imshow(heatmap_overlay)
axes[3].set_title(f"{PRIMARY_CAM_METHOD} overlay")
axes[3].axis("off")

fig.suptitle(
    f"AQPR anatomical metric smoke | "
    f"sample={smoke_case_xai['sample_id']} | "
    f"obs-rand={smoke_enrichment['observed_minus_random_mean']:.4f}",
    fontsize=12,
    fontweight="bold",
)

fig.tight_layout()
display(fig)

fig.savefig(ANATOMICAL_SMOKE_FIGURE, dpi=300, bbox_inches="tight")
fig.savefig(ANATOMICAL_SMOKE_FIGURE_PDF, bbox_inches="tight")
plt.close(fig)

anatomical_metric_config = {
    "notebook": "08_anatomical_explainability_validation",
    "cell": "Cell 4",
    "analysis": "AQPR anatomical quantification and matched random-control masks",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_anatomical_dataset": "HYGD",
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "primary_cam_method": PRIMARY_CAM_METHOD,
    "primary_attribution_method": PRIMARY_ATTRIBUTION_METHOD,
    "sensitivity_cam_method": SENSITIVITY_CAM_METHOD,
    "primary_anatomical_region": PRIMARY_ANATOMICAL_REGION,
    "anatomical_regions": ANATOMICAL_REGIONS,
    "n_random_controls": int(N_RANDOM_CONTROLS),
    "random_control_policy": {
        "matched_shape": True,
        "within_fov": True,
        "min_fov_fraction": float(RANDOM_CONTROL_MIN_FOV_FRACTION),
        "max_iou_with_target": float(RANDOM_CONTROL_MAX_IOU_WITH_TARGET),
        "max_attempts": int(RANDOM_CONTROL_MAX_ATTEMPTS),
        "excludes_od_oc_region": True,
    },
    "primary_endpoint": "observed_minus_random_mean",
    "secondary_endpoints": [
        "observed_energy_fraction",
        "enrichment_ratio",
        "empirical_p_greater_equal",
    ],
    "xai_method_policy": AQPR_XAI_METHOD_POLICY,
    "patient_level_external_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
    "smoke_sample_id": str(smoke_case_xai["sample_id"]),
    "smoke_enrichment_csv": str(ANATOMICAL_ENRICHMENT_SMOKE_CSV),
    "smoke_random_control_csv": str(RANDOM_CONTROL_SMOKE_CSV),
    "smoke_figure_png": str(ANATOMICAL_SMOKE_FIGURE),
}

with open(ANATOMICAL_METRIC_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(anatomical_metric_config, f, indent=2)

print("AQPR anatomical metric config:")
print(json.dumps(anatomical_metric_config, indent=2))

print("\nRandom-control smoke audit:")
print(
    smoke_random_audit[
        [
            "accepted",
            "fov_fraction",
            "iou_with_target",
            "overlap_with_exclusion",
        ]
    ]
    .query("accepted == True")
    .describe()
    .to_string()
)

print("\nAnatomical enrichment smoke test:")
print(smoke_enrichment_df.to_string(index=False))

print(f"\nRandom-control smoke CSV: {RANDOM_CONTROL_SMOKE_CSV}")
print(f"Anatomical enrichment smoke CSV: {ANATOMICAL_ENRICHMENT_SMOKE_CSV}")
print(f"Smoke figure: {ANATOMICAL_SMOKE_FIGURE}")
print(f"Anatomical metric config JSON: {ANATOMICAL_METRIC_CONFIG_JSON}")

if len(smoke_random_masks) != N_RANDOM_CONTROLS:
    raise RuntimeError(
        f"Expected {N_RANDOM_CONTROLS} random controls, found {len(smoke_random_masks)}"
    )

if not np.isfinite(smoke_enrichment_df["observed_minus_random_mean"]).all():
    raise RuntimeError("Non-finite observed-minus-random enrichment in smoke test.")

if not np.isfinite(smoke_enrichment_df["enrichment_ratio"]).all():
    raise RuntimeError("Non-finite enrichment ratio in smoke test.")

for required_output in [
    RANDOM_CONTROL_SMOKE_CSV,
    ANATOMICAL_ENRICHMENT_SMOKE_CSV,
    ANATOMICAL_SMOKE_FIGURE,
    ANATOMICAL_SMOKE_FIGURE_PDF,
    ANATOMICAL_METRIC_CONFIG_JSON,
]:
    if not Path(required_output).exists():
        raise RuntimeError(f"Missing Cell 4 output: {required_output}")

print("Status: PASS")

In [ ]:
# Cell 5: Run one resumable AQPR heatmap-XAI task on HYGD

import time
import hashlib
import gc
from tqdm.auto import tqdm

HEATMAP_ANALYSIS_ROOT = RESULTS_08_DIR / "aqpr_panel_heatmap_anatomical_xai"
HEATMAP_ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

HEATMAP_CACHE_ROOT = EXPLAINABILITY_PROJECT_DATA_DIR / "aqpr_panel_heatmap_cache"
HEATMAP_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

HEATMAP_PLAN_CSV = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_plan.csv"
HEATMAP_PROGRESS_CSV = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_progress.csv"
HEATMAP_CONFIG_JSON = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_config.json"

# Automatic mode: leave both as None.
# The cell will run the next incomplete model-method task.
MODEL_TO_RUN = None
METHOD_TO_RUN = None

# Re-run this cell until all tasks are complete.
MAX_CASES_PER_EXECUTION = 96

RUN_HEATMAPS_ON_CUDA_ONLY = True
SAVE_HEATMAPS = True
IG_STEPS_FOR_FULL_ANALYSIS = 16

FULL_HEATMAP_METHODS_BY_MODEL = {
    "EfficientNetB0": ["GradCAM", "IntegratedGradients"],
    "DenseNet121": ["GradCAM", "IntegratedGradients"],
    "SwinTiny": ["IntegratedGradients"],
    "ConvNeXtTiny": ["GradCAM", "IntegratedGradients"],
}

VALID_FULL_HEATMAP_METHODS = ["GradCAM", "IntegratedGradients"]


def stable_int_seed(*parts, modulo=2**32 - 1):
    text = "|".join([str(part) for part in parts])
    digest = hashlib.md5(text.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % modulo


def logit_from_probability_np(probability):
    probability = float(probability)
    probability = np.clip(probability, 1e-7, 1.0 - 1e-7)
    return float(np.log(probability / (1.0 - probability)))


def temperature_scaled_probability_from_probability(probability, temperature):
    logit = logit_from_probability_np(probability)
    scaled_logit = logit / float(temperature)
    return float(1.0 / (1.0 + np.exp(-scaled_logit)))


def get_model_row_from_frozen_manifest(model_name):
    rows = frozen_aqpr_panel_manifest[
        frozen_aqpr_panel_manifest["model_name"].astype(str) == str(model_name)
    ].copy()

    if len(rows) != 1:
        raise RuntimeError(f"Expected one frozen manifest row for {model_name}, found {len(rows)}")

    return rows.iloc[0].to_dict()


def heatmap_cache_path(model_name, method_name, sample_id):
    model_method_dir = HEATMAP_CACHE_ROOT / model_name / method_name
    model_method_dir.mkdir(parents=True, exist_ok=True)

    return model_method_dir / f"{sample_id}_{model_name}_{method_name}_heatmap.npy"


def task_output_paths(model_name, method_name):
    task_dir = HEATMAP_ANALYSIS_ROOT / model_name / method_name
    task_dir.mkdir(parents=True, exist_ok=True)

    return {
        "task_dir": task_dir,
        "metrics_csv": task_dir / f"notebook08_{model_name}_{method_name}_hygd_anatomical_metrics.csv",
        "case_log_csv": task_dir / f"notebook08_{model_name}_{method_name}_hygd_case_log.csv",
        "random_control_summary_csv": task_dir / f"notebook08_{model_name}_{method_name}_hygd_random_control_summary.csv",
        "region_summary_csv": task_dir / f"notebook08_{model_name}_{method_name}_hygd_region_summary.csv",
        "summary_json": task_dir / f"notebook08_{model_name}_{method_name}_hygd_summary.json",
    }


def load_existing_csv(path):
    path = Path(path)

    if path.exists():
        return pd.read_csv(path)

    return pd.DataFrame()


def get_completed_sample_ids(case_log_csv):
    case_log = load_existing_csv(case_log_csv)

    if len(case_log) == 0:
        return set()

    if "completed" not in case_log.columns:
        return set()

    completed = case_log[as_bool_series(case_log["completed"])].copy()

    return set(completed["sample_id"].astype(str))


def build_heatmap_plan(analysis_df):
    rows = []

    for model_name in AQPR_PANEL_MODEL_NAMES:
        for method_name in FULL_HEATMAP_METHODS_BY_MODEL[model_name]:
            model_row = get_model_row_from_frozen_manifest(model_name)
            paths = task_output_paths(model_name, method_name)

            rows.append(
                {
                    "model_name": model_name,
                    "method_name": method_name,
                    "aqpr_panel_role": AQPR_PANEL_ROLES[model_name],
                    "checkpoint_path": str(model_row["checkpoint_path"]),
                    "input_size": int(model_row["input_size"]),
                    "temperature": float(model_row["temperature"]),
                    "threshold": float(model_row["validation_selected_threshold"]),
                    "expected_images": int(len(analysis_df)),
                    "metrics_csv": str(paths["metrics_csv"]),
                    "case_log_csv": str(paths["case_log_csv"]),
                    "random_control_summary_csv": str(paths["random_control_summary_csv"]),
                    "region_summary_csv": str(paths["region_summary_csv"]),
                    "summary_json": str(paths["summary_json"]),
                }
            )

    return pd.DataFrame(rows)


def compute_heatmap_for_task(model, method_name, image_tensor, target_layer, input_size):
    if method_name == "GradCAM":
        if target_layer is None:
            raise RuntimeError("GradCAM requires a valid target layer.")

        result = compute_gradcam_heatmap(
            model=model,
            image_tensor=image_tensor,
            target_layer=target_layer,
            target_mode="positive",
        )

        heatmap = resize_heatmap_to_input(
            result["heatmap"],
            input_size,
        )

        return {
            "logit": float(result["logit"]),
            "probability": float(result["probability"]),
            "heatmap": heatmap.astype(np.float32),
        }

    if method_name == "IntegratedGradients":
        result = compute_integrated_gradients_heatmap(
            model=model,
            image_tensor=image_tensor,
            n_steps=IG_STEPS_FOR_FULL_ANALYSIS,
            baseline_mode="zero",
            target_mode="positive",
        )

        heatmap = resize_heatmap_to_input(
            result["heatmap"],
            input_size,
        )

        return {
            "logit": np.nan,
            "probability": float(result["probability"]),
            "heatmap": heatmap.astype(np.float32),
        }

    raise ValueError(f"Unknown method_name: {method_name}")


def compute_case_anatomical_metrics(
    heatmap,
    region_masks,
    fov_mask,
    sample_id,
    patient_id,
    model_name,
    method_name,
    row,
    xai_probability,
    heatmap_path,
):
    metric_rows = []
    random_summary_rows = []

    model_manifest_row = get_model_row_from_frozen_manifest(model_name)
    threshold = float(model_manifest_row["validation_selected_threshold"])
    temperature = float(model_manifest_row["temperature"])

    predicted_label = int(float(xai_probability) >= threshold)
    prediction_correct = bool(predicted_label == int(row["gon_binary"]))

    temperature_scaled_probability = temperature_scaled_probability_from_probability(
        probability=float(xai_probability),
        temperature=temperature,
    )

    for region_name in ANATOMICAL_REGIONS:
        target_mask = region_masks[region_name].astype(bool)

        if target_mask.sum() == 0:
            continue

        rng_case_region = np.random.default_rng(
            stable_int_seed(
                "cell5",
                model_name,
                method_name,
                sample_id,
                region_name,
                RANDOM_CONTROL_SEED,
            )
        )

        random_masks, random_audit = generate_matched_random_masks(
            target_mask=target_mask,
            fov_mask=fov_mask,
            exclusion_mask=region_masks["OD_plus_OC"],
            n_controls=N_RANDOM_CONTROLS,
            rng=rng_case_region,
        )

        stats = heatmap_energy_statistics(
            heatmap=heatmap,
            region_mask=target_mask,
            random_masks=random_masks,
        )

        metric_rows.append(
            {
                "sample_id": str(sample_id),
                "patient_id": str(patient_id),
                "model_name": model_name,
                "method_name": method_name,
                "aqpr_panel_role": AQPR_PANEL_ROLES[model_name],
                "region_name": region_name,
                "evaluation_dataset": "HYGD",
                "evaluation_split": "hygd_external_strict_anatomical",
                "y_true": int(row["gon_binary"]),
                "gon_label": str(row["gon_label"]),
                "xai_probability": float(xai_probability),
                "temperature_scaled_probability": float(temperature_scaled_probability),
                "threshold": threshold,
                "predicted_label": predicted_label,
                "prediction_correct": prediction_correct,
                "mask_pixels": int(target_mask.sum()),
                "fov_pixels": int(fov_mask.sum()),
                "heatmap_path": str(heatmap_path),
                **stats,
            }
        )

        accepted_random = random_audit[random_audit["accepted"].astype(bool)].copy()

        random_summary_rows.append(
            {
                "sample_id": str(sample_id),
                "patient_id": str(patient_id),
                "model_name": model_name,
                "method_name": method_name,
                "region_name": region_name,
                "n_controls_requested": int(N_RANDOM_CONTROLS),
                "n_controls_accepted": int(len(accepted_random)),
                "attempts_total": int(len(random_audit)),
                "accepted_fov_fraction_mean": float(accepted_random["fov_fraction"].mean()),
                "accepted_iou_with_target_max": float(accepted_random["iou_with_target"].max()),
                "accepted_overlap_with_exclusion_max": float(accepted_random["overlap_with_exclusion"].max()),
            }
        )

    return pd.DataFrame(metric_rows), pd.DataFrame(random_summary_rows)


if DEVICE.type != "cuda" and RUN_HEATMAPS_ON_CUDA_ONLY:
    print(f"Device: {DEVICE}")
    print("Heatmap generation skipped because CUDA is not available.")
    print("Set RUN_HEATMAPS_ON_CUDA_ONLY = False only for intentional CPU execution.")
    print("Status: PASS")

else:
    if "hygd_strict_xai_df" not in globals():
        raise RuntimeError("Cell 2 must be executed before Cell 5.")

    if "load_aqpr_classifier" not in globals():
        raise RuntimeError("Cell 2 must be executed before Cell 5: load_aqpr_classifier is missing.")

    if "compute_gradcam_heatmap" not in globals():
        raise RuntimeError("Cell 2 must be executed before Cell 5: compute_gradcam_heatmap is missing.")

    if "compute_integrated_gradients_heatmap" not in globals():
        raise RuntimeError("Cell 3 must be executed before Cell 5: compute_integrated_gradients_heatmap is missing.")

    if "heatmap_energy_statistics" not in globals():
        raise RuntimeError("Cell 4 must be executed before Cell 5: heatmap_energy_statistics is missing.")

    if len(hygd_strict_xai_df) == 0:
        raise RuntimeError("HYGD strict AQPR XAI cohort is empty.")

    analysis_df = (
        hygd_strict_xai_df
        .copy()
        .sort_values(["patient_id", "sample_id"])
        .reset_index(drop=True)
    )

    analysis_df["sample_id"] = analysis_df["sample_id"].astype(str)
    analysis_df["patient_id"] = analysis_df["patient_id"].astype(str)

    heatmap_plan = build_heatmap_plan(analysis_df)

    progress_rows = []

    for _, plan_row in heatmap_plan.iterrows():
        completed_ids = get_completed_sample_ids(plan_row["case_log_csv"])
        progress_rows.append(
            {
                **plan_row.to_dict(),
                "completed_images": int(len(completed_ids)),
                "pending_images": int(len(analysis_df) - len(completed_ids)),
                "task_complete": int(len(completed_ids)) == int(len(analysis_df)),
            }
        )

    heatmap_progress = pd.DataFrame(progress_rows)

    if MODEL_TO_RUN is not None or METHOD_TO_RUN is not None:
        if MODEL_TO_RUN is None or METHOD_TO_RUN is None:
            raise ValueError("Set both MODEL_TO_RUN and METHOD_TO_RUN, or leave both as None.")

        selected_tasks = heatmap_progress[
            (heatmap_progress["model_name"].astype(str) == str(MODEL_TO_RUN))
            & (heatmap_progress["method_name"].astype(str) == str(METHOD_TO_RUN))
        ].copy()

        if len(selected_tasks) != 1:
            raise RuntimeError(f"Could not identify requested task: {MODEL_TO_RUN}, {METHOD_TO_RUN}")

        task_row = selected_tasks.iloc[0].to_dict()

    else:
        pending_tasks = heatmap_progress[~heatmap_progress["task_complete"]].copy()

        if len(pending_tasks) == 0:
            heatmap_plan.to_csv(HEATMAP_PLAN_CSV, index=False)
            heatmap_progress.to_csv(HEATMAP_PROGRESS_CSV, index=False)

            print("All AQPR heatmap-XAI tasks are already complete.")
            print(
                heatmap_progress[
                    [
                        "model_name",
                        "method_name",
                        "completed_images",
                        "pending_images",
                        "task_complete",
                    ]
                ].to_string(index=False)
            )
            print("Status: PASS")

            raise SystemExit

        task_row = pending_tasks.iloc[0].to_dict()

    model_name = str(task_row["model_name"])
    method_name = str(task_row["method_name"])

    if method_name not in VALID_FULL_HEATMAP_METHODS:
        raise ValueError(f"Invalid heatmap method: {method_name}")

    model_row = get_model_row_from_frozen_manifest(model_name)
    checkpoint_path = Path(str(model_row["checkpoint_path"]))
    input_size = int(model_row["input_size"])

    task_paths = task_output_paths(model_name, method_name)

    completed_ids = get_completed_sample_ids(task_paths["case_log_csv"])

    pending_df = analysis_df[
        ~analysis_df["sample_id"].astype(str).isin(completed_ids)
    ].copy().reset_index(drop=True)

    if MAX_CASES_PER_EXECUTION is not None:
        pending_df = pending_df.head(int(MAX_CASES_PER_EXECUTION)).copy()

    print("AQPR heatmap-XAI task:")
    print(f"  Model: {model_name}")
    print(f"  Method: {method_name}")
    print(f"  Device: {DEVICE}")
    print(f"  Input size: {input_size}")
    print(f"  Expected HYGD strict images: {len(analysis_df)}")
    print(f"  Already completed for this task: {len(completed_ids)}")
    print(f"  Cases scheduled in this execution: {len(pending_df)}")
    print(f"  Random controls per region: {N_RANDOM_CONTROLS}")
    print(f"  Save heatmaps: {SAVE_HEATMAPS}")

    if len(pending_df) == 0:
        print("This task is already complete. Re-run the cell to select the next pending task.")
        print("Status: PASS")

        raise SystemExit

    model, checkpoint, resolved_timm_name = load_aqpr_classifier(
        model_name=model_name,
        checkpoint_path=checkpoint_path,
        input_size=input_size,
        device=DEVICE,
    )

    target_layer_name, target_layer = find_last_conv2d_layer(model)

    if method_name == "GradCAM" and target_layer is None:
        raise RuntimeError(f"No Conv2d target layer found for GradCAM in {model_name}.")

    existing_metrics = load_existing_csv(task_paths["metrics_csv"])
    existing_case_log = load_existing_csv(task_paths["case_log_csv"])
    existing_random_summary = load_existing_csv(task_paths["random_control_summary_csv"])

    new_metric_parts = []
    new_case_rows = []
    new_random_summary_parts = []

    start_time = time.time()

    iterator = tqdm(
        pending_df.iterrows(),
        total=len(pending_df),
        desc=f"{model_name} | {method_name}",
    )

    for _, row in iterator:
        sample_id = str(row["sample_id"])
        patient_id = str(row["patient_id"])
        heatmap_path = heatmap_cache_path(model_name, method_name, sample_id)

        case_start = time.time()

        try:
            image_tensor, image_rgb_square, image_rgb_resized = preprocess_classifier_image_from_path(
                image_path=row["classifier_image_path"],
                input_size=input_size,
            )

            _, region_masks = build_region_masks_from_mask_path(
                mask_path=row["mask_512_path"],
                input_size=input_size,
            )

            fov_mask = compute_fov_mask_from_rgb(
                image_rgb=image_rgb_resized,
                threshold=8,
            )

            heatmap_result = compute_heatmap_for_task(
                model=model,
                method_name=method_name,
                image_tensor=image_tensor,
                target_layer=target_layer,
                input_size=input_size,
            )

            heatmap = normalize_heatmap(heatmap_result["heatmap"]).astype(np.float32)

            if SAVE_HEATMAPS:
                np.save(str(heatmap_path), heatmap)

            metrics_df, random_summary_df = compute_case_anatomical_metrics(
                heatmap=heatmap,
                region_masks=region_masks,
                fov_mask=fov_mask,
                sample_id=sample_id,
                patient_id=patient_id,
                model_name=model_name,
                method_name=method_name,
                row=row,
                xai_probability=float(heatmap_result["probability"]),
                heatmap_path=heatmap_path,
            )

            new_metric_parts.append(metrics_df)
            new_random_summary_parts.append(random_summary_df)

            new_case_rows.append(
                {
                    "sample_id": sample_id,
                    "patient_id": patient_id,
                    "model_name": model_name,
                    "method_name": method_name,
                    "completed": True,
                    "status": "pass",
                    "error": "",
                    "heatmap_path": str(heatmap_path),
                    "heatmap_saved": bool(Path(heatmap_path).exists()),
                    "elapsed_sec": round(time.time() - case_start, 3),
                    "xai_probability": float(heatmap_result["probability"]),
                    "heatmap_min": float(np.min(heatmap)),
                    "heatmap_max": float(np.max(heatmap)),
                    "heatmap_mean": float(np.mean(heatmap)),
                }
            )

        except Exception as exc:
            new_case_rows.append(
                {
                    "sample_id": sample_id,
                    "patient_id": patient_id,
                    "model_name": model_name,
                    "method_name": method_name,
                    "completed": False,
                    "status": "fail",
                    "error": repr(exc),
                    "heatmap_path": str(heatmap_path),
                    "heatmap_saved": False,
                    "elapsed_sec": round(time.time() - case_start, 3),
                    "xai_probability": np.nan,
                    "heatmap_min": np.nan,
                    "heatmap_max": np.nan,
                    "heatmap_mean": np.nan,
                }
            )

        current_case_log = pd.DataFrame(new_case_rows)

        combined_case_log = pd.concat(
            [existing_case_log, current_case_log],
            axis=0,
            ignore_index=True,
        )

        combined_case_log = (
            combined_case_log
            .drop_duplicates(
                subset=["sample_id", "model_name", "method_name"],
                keep="last",
            )
            .reset_index(drop=True)
        )

        combined_case_log.to_csv(task_paths["case_log_csv"], index=False)

        if len(new_metric_parts) > 0:
            combined_metrics = pd.concat(
                [existing_metrics] + new_metric_parts,
                axis=0,
                ignore_index=True,
            )

            combined_metrics = (
                combined_metrics
                .drop_duplicates(
                    subset=["sample_id", "model_name", "method_name", "region_name"],
                    keep="last",
                )
                .reset_index(drop=True)
            )

            combined_metrics.to_csv(task_paths["metrics_csv"], index=False)

        if len(new_random_summary_parts) > 0:
            combined_random_summary = pd.concat(
                [existing_random_summary] + new_random_summary_parts,
                axis=0,
                ignore_index=True,
            )

            combined_random_summary = (
                combined_random_summary
                .drop_duplicates(
                    subset=["sample_id", "model_name", "method_name", "region_name"],
                    keep="last",
                )
                .reset_index(drop=True)
            )

            combined_random_summary.to_csv(task_paths["random_control_summary_csv"], index=False)

    elapsed_min = round((time.time() - start_time) / 60.0, 3)

    case_log = pd.read_csv(task_paths["case_log_csv"])

    failed_rows = case_log[~as_bool_series(case_log["completed"])].copy()

    if len(failed_rows) > 0 and not Path(task_paths["metrics_csv"]).exists():
        print("\nFirst failed cases:")
        print(
            failed_rows[
                [
                    "sample_id",
                    "patient_id",
                    "model_name",
                    "method_name",
                    "status",
                    "error",
                ]
            ].head(20).to_string(index=False)
        )

        raise RuntimeError(
            "No anatomical metrics file was generated because all scheduled cases failed. "
            "The errors above identify the cause."
        )

    if not Path(task_paths["metrics_csv"]).exists():
        raise FileNotFoundError(f"Metrics CSV was not created: {task_paths['metrics_csv']}")

    if not Path(task_paths["random_control_summary_csv"]).exists():
        raise FileNotFoundError(
            f"Random-control summary CSV was not created: {task_paths['random_control_summary_csv']}"
        )

    task_metrics = pd.read_csv(task_paths["metrics_csv"])
    random_summary = pd.read_csv(task_paths["random_control_summary_csv"])

    completed_total = int(as_bool_series(case_log["completed"]).sum())
    failed_total = int((~as_bool_series(case_log["completed"])).sum())
    task_complete = completed_total == len(analysis_df)

    region_summary = (
        task_metrics
        .groupby("region_name")
        .agg(
            n_rows=("sample_id", "count"),
            n_images=("sample_id", "nunique"),
            observed_energy_fraction_mean=("observed_energy_fraction", "mean"),
            observed_minus_random_mean_mean=("observed_minus_random_mean", "mean"),
            enrichment_ratio_median=("enrichment_ratio", "median"),
            empirical_p_greater_equal_mean=("empirical_p_greater_equal", "mean"),
        )
        .reset_index()
    )

    region_summary.to_csv(task_paths["region_summary_csv"], index=False)

    task_summary = {
        "status": "PASS" if task_complete else "IN_PROGRESS",
        "model_name": model_name,
        "method_name": method_name,
        "resolved_timm_model_name": resolved_timm_name,
        "target_layer_name": target_layer_name if target_layer_name is not None else "",
        "external_dataset": "HYGD",
        "cohort": "hygd_external_strict_anatomical",
        "expected_images": int(len(analysis_df)),
        "completed_images": completed_total,
        "failed_images": failed_total,
        "scheduled_this_execution": int(len(pending_df)),
        "n_random_controls": int(N_RANDOM_CONTROLS),
        "ig_steps": int(IG_STEPS_FOR_FULL_ANALYSIS) if method_name == "IntegratedGradients" else None,
        "elapsed_min_current_execution": elapsed_min,
        "metrics_csv": str(task_paths["metrics_csv"]),
        "case_log_csv": str(task_paths["case_log_csv"]),
        "random_control_summary_csv": str(task_paths["random_control_summary_csv"]),
        "region_summary_csv": str(task_paths["region_summary_csv"]),
        "heatmap_cache_root": str(HEATMAP_CACHE_ROOT / model_name / method_name),
        "patient_level_external_analysis_allowed": True,
        "internal_airogslight_patient_level_claim_allowed": False,
    }

    with open(task_paths["summary_json"], "w", encoding="utf-8") as f:
        json.dump(task_summary, f, indent=2)

    progress_rows = []

    for _, plan_row in heatmap_plan.iterrows():
        completed_ids_updated = get_completed_sample_ids(plan_row["case_log_csv"])
        progress_rows.append(
            {
                **plan_row.to_dict(),
                "completed_images": int(len(completed_ids_updated)),
                "pending_images": int(len(analysis_df) - len(completed_ids_updated)),
                "task_complete": int(len(completed_ids_updated)) == int(len(analysis_df)),
            }
        )

    heatmap_progress = pd.DataFrame(progress_rows)

    heatmap_plan.to_csv(HEATMAP_PLAN_CSV, index=False)
    heatmap_progress.to_csv(HEATMAP_PROGRESS_CSV, index=False)

    heatmap_config = {
        "notebook": "08_anatomical_explainability_validation",
        "cell": "Cell 5",
        "analysis": "AQPR panel heatmap-based anatomical XAI on HYGD",
        "external_dataset": "HYGD",
        "cohort": "hygd_external_strict_anatomical",
        "model_panel": AQPR_PANEL_MODEL_NAMES,
        "methods_by_model": FULL_HEATMAP_METHODS_BY_MODEL,
        "primary_cam_method": "GradCAM",
        "primary_attribution_method": "IntegratedGradients",
        "excluded_from_full_heatmap_analysis": {
            "GradCAMPlusPlus": (
                "Retained only as smoke/sensitivity because initial HYGD smoke test showed "
                "border-dominant activation."
            )
        },
        "n_random_controls": int(N_RANDOM_CONTROLS),
        "max_cases_per_execution": (
            None if MAX_CASES_PER_EXECUTION is None else int(MAX_CASES_PER_EXECUTION)
        ),
        "ig_steps": int(IG_STEPS_FOR_FULL_ANALYSIS),
        "progress_csv": str(HEATMAP_PROGRESS_CSV),
        "plan_csv": str(HEATMAP_PLAN_CSV),
    }

    with open(HEATMAP_CONFIG_JSON, "w", encoding="utf-8") as f:
        json.dump(heatmap_config, f, indent=2)

    del model, checkpoint

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

    print("\nCurrent task summary:")
    print(json.dumps(task_summary, indent=2))

    print("\nRegion summary for current task:")
    print(region_summary.to_string(index=False))

    print("\nOverall AQPR heatmap-XAI progress:")
    print(
        heatmap_progress[
            [
                "model_name",
                "method_name",
                "completed_images",
                "pending_images",
                "task_complete",
            ]
        ].to_string(index=False)
    )

    print(f"\nMetrics CSV: {task_paths['metrics_csv']}")
    print(f"Case log CSV: {task_paths['case_log_csv']}")
    print(f"Random-control summary CSV: {task_paths['random_control_summary_csv']}")
    print(f"Region summary CSV: {task_paths['region_summary_csv']}")
    print(f"Overall progress CSV: {HEATMAP_PROGRESS_CSV}")
    print(f"Config JSON: {HEATMAP_CONFIG_JSON}")

    if failed_total > 0:
        print("\nFailed cases:")
        print(
            failed_rows[
                [
                    "sample_id",
                    "patient_id",
                    "model_name",
                    "method_name",
                    "status",
                    "error",
                ]
            ].tail(20).to_string(index=False)
        )

        raise RuntimeError(f"{model_name} | {method_name} has failed cases: {failed_total}")

    if len(task_metrics) == 0:
        raise RuntimeError("No anatomical metrics were generated for current task.")

    od_plus_oc_metrics = task_metrics[
        task_metrics["region_name"].astype(str) == "OD_plus_OC"
    ].copy()

    if len(od_plus_oc_metrics) == 0:
        raise RuntimeError("No OD_plus_OC metrics generated for current task.")

    if not np.isfinite(od_plus_oc_metrics["observed_minus_random_mean"]).all():
        raise RuntimeError("Non-finite OD_plus_OC observed-minus-random metrics found.")

    if heatmap_progress["task_complete"].all():
        print("All AQPR heatmap-XAI tasks are complete.")
        print("Status: PASS")
    else:
        print("Status: IN_PROGRESS - re-run Cell 5 until all tasks show task_complete=True.")

In [ ]:
# Cell 6: Consolidate AQPR heatmap-XAI metrics and run image-/patient-level statistics

from scipy.stats import wilcoxon, mannwhitneyu

HEATMAP_ANALYSIS_ROOT = RESULTS_08_DIR / "aqpr_panel_heatmap_anatomical_xai"
HEATMAP_PROGRESS_CSV = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_progress.csv"

HEATMAP_CONSOLIDATED_CSV = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_metrics_consolidated.csv"
HEATMAP_PATIENT_LEVEL_CSV = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_patient_level_metrics.csv"
HEATMAP_IMAGE_LEVEL_STATS_CSV = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_image_level_statistics.csv"
HEATMAP_PATIENT_LEVEL_STATS_CSV = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_patient_level_statistics.csv"
HEATMAP_MODEL_COMPARISON_CSV = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_paired_model_comparisons.csv"
HEATMAP_METHOD_COMPARISON_CSV = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_paired_method_comparisons.csv"
HEATMAP_LABEL_STRATA_CSV = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_label_strata_statistics.csv"
HEATMAP_CORRECTNESS_STRATA_CSV = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_correctness_strata_statistics.csv"
HEATMAP_STATISTICAL_SUMMARY_JSON = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_statistical_summary.json"

PRIMARY_ENDPOINT = "observed_minus_random_mean"
SECONDARY_ENDPOINTS = [
    "observed_energy_fraction",
    "enrichment_ratio",
    "empirical_p_greater_equal",
]

PRIMARY_REGION = "OD_plus_OC"
BOOTSTRAP_N = 5000
BOOTSTRAP_SEED = SEED

rng = np.random.default_rng(BOOTSTRAP_SEED)


def bootstrap_mean_ci(values, n_bootstrap=BOOTSTRAP_N, ci_low=2.5, ci_high=97.5):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan, 0

    if len(values) == 1:
        return float(values[0]), float(values[0]), 1

    sampled = rng.choice(values, size=(int(n_bootstrap), len(values)), replace=True)
    boot_means = sampled.mean(axis=1)

    return (
        float(np.percentile(boot_means, ci_low)),
        float(np.percentile(boot_means, ci_high)),
        int(len(boot_means)),
    )


def safe_wilcoxon_greater(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 2:
        return np.nan, np.nan, len(values)

    if np.allclose(values, 0.0):
        return 0.0, 1.0, len(values)

    try:
        stat, p_value = wilcoxon(
            values,
            zero_method="wilcox",
            alternative="greater",
        )
        return float(stat), float(p_value), len(values)
    except ValueError:
        return np.nan, np.nan, len(values)


def safe_wilcoxon_paired(delta_values):
    delta_values = np.asarray(delta_values, dtype=float)
    delta_values = delta_values[np.isfinite(delta_values)]

    if len(delta_values) < 2:
        return np.nan, np.nan, len(delta_values)

    if np.allclose(delta_values, 0.0):
        return 0.0, 1.0, len(delta_values)

    try:
        stat, p_value = wilcoxon(
            delta_values,
            zero_method="wilcox",
            alternative="two-sided",
        )
        return float(stat), float(p_value), len(delta_values)
    except ValueError:
        return np.nan, np.nan, len(delta_values)


def safe_mannwhitney(group_a, group_b):
    group_a = np.asarray(group_a, dtype=float)
    group_b = np.asarray(group_b, dtype=float)

    group_a = group_a[np.isfinite(group_a)]
    group_b = group_b[np.isfinite(group_b)]

    if len(group_a) < 2 or len(group_b) < 2:
        return np.nan, np.nan, len(group_a), len(group_b)

    try:
        stat, p_value = mannwhitneyu(
            group_a,
            group_b,
            alternative="two-sided",
        )
        return float(stat), float(p_value), len(group_a), len(group_b)
    except ValueError:
        return np.nan, np.nan, len(group_a), len(group_b)


def bh_fdr(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full_like(p_values, fill_value=np.nan, dtype=float)

    valid = np.isfinite(p_values)

    if valid.sum() == 0:
        return adjusted

    p = p_values[valid]
    m = len(p)
    order = np.argsort(p)
    ranked = p[order]

    adjusted_ranked = ranked * m / (np.arange(m) + 1)
    adjusted_ranked = np.minimum.accumulate(adjusted_ranked[::-1])[::-1]
    adjusted_ranked = np.clip(adjusted_ranked, 0.0, 1.0)

    valid_indices = np.where(valid)[0]
    adjusted[valid_indices[order]] = adjusted_ranked

    return adjusted


def summarize_endpoint(df, level_name):
    rows = []

    group_cols = [
        "model_name",
        "method_name",
        "aqpr_panel_role",
        "region_name",
    ]

    for keys, group in df.groupby(group_cols):
        model_name, method_name, role, region_name = keys

        values = pd.to_numeric(group[PRIMARY_ENDPOINT], errors="coerce").values
        values = values[np.isfinite(values)]

        ci_low, ci_high, n_valid_bootstrap = bootstrap_mean_ci(values)
        wilcoxon_stat, wilcoxon_p, n_valid_test = safe_wilcoxon_greater(values)

        row = {
            "level": level_name,
            "model_name": model_name,
            "method_name": method_name,
            "aqpr_panel_role": role,
            "region_name": region_name,
            "n_units": int(len(values)),
            "primary_endpoint": PRIMARY_ENDPOINT,
            "primary_endpoint_mean": float(np.mean(values)) if len(values) > 0 else np.nan,
            "primary_endpoint_median": float(np.median(values)) if len(values) > 0 else np.nan,
            "primary_endpoint_std": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
            "primary_endpoint_ci_low": ci_low,
            "primary_endpoint_ci_high": ci_high,
            "n_valid_bootstrap": n_valid_bootstrap,
            "wilcoxon_greater_than_zero_statistic": wilcoxon_stat,
            "wilcoxon_greater_than_zero_p": wilcoxon_p,
            "n_valid_wilcoxon": n_valid_test,
        }

        for endpoint in SECONDARY_ENDPOINTS:
            endpoint_values = pd.to_numeric(group[endpoint], errors="coerce").values
            endpoint_values = endpoint_values[np.isfinite(endpoint_values)]

            row[f"{endpoint}_mean"] = (
                float(np.mean(endpoint_values))
                if len(endpoint_values) > 0
                else np.nan
            )
            row[f"{endpoint}_median"] = (
                float(np.median(endpoint_values))
                if len(endpoint_values) > 0
                else np.nan
            )

        rows.append(row)

    output = pd.DataFrame(rows)

    if len(output) > 0:
        output["wilcoxon_greater_than_zero_p_fdr"] = bh_fdr(
            output["wilcoxon_greater_than_zero_p"].values
        )

    return output


def build_patient_level_metrics(metrics_df):
    label_consistency = (
        metrics_df
        .groupby("patient_id")
        .agg(
            n_unique_labels=("y_true", "nunique"),
            patient_y_true=("y_true", "max"),
        )
        .reset_index()
    )

    inconsistent_patients = label_consistency[
        label_consistency["n_unique_labels"] > 1
    ].copy()

    if len(inconsistent_patients) > 0:
        print("Warning: patients with mixed image-level labels detected.")
        print(inconsistent_patients.head(20).to_string(index=False))

    patient_df = (
        metrics_df
        .groupby(
            [
                "model_name",
                "method_name",
                "aqpr_panel_role",
                "region_name",
                "patient_id",
            ]
        )
        .agg(
            n_images=("sample_id", "nunique"),
            y_true=("y_true", "max"),
            gon_label=("gon_label", "first"),
            xai_probability_mean=("xai_probability", "mean"),
            temperature_scaled_probability_mean=("temperature_scaled_probability", "mean"),
            prediction_correct_fraction=("prediction_correct", "mean"),
            observed_energy_fraction=("observed_energy_fraction", "mean"),
            random_energy_fraction_mean=("random_energy_fraction_mean", "mean"),
            observed_minus_random_mean=("observed_minus_random_mean", "mean"),
            enrichment_ratio=("enrichment_ratio", "median"),
            empirical_p_greater_equal=("empirical_p_greater_equal", "mean"),
            mask_pixels_mean=("mask_pixels", "mean"),
            fov_pixels_mean=("fov_pixels", "mean"),
        )
        .reset_index()
    )

    return patient_df, inconsistent_patients


def paired_model_comparisons(df, level_name):
    rows = []

    reference_model = PRIMARY_MODEL_NAME

    for method_name in sorted(df["method_name"].astype(str).unique()):
        method_df = df[df["method_name"].astype(str) == method_name].copy()

        for region_name in sorted(method_df["region_name"].astype(str).unique()):
            region_df = method_df[method_df["region_name"].astype(str) == region_name].copy()

            id_col = "sample_id" if level_name == "image" else "patient_id"

            reference = region_df[
                region_df["model_name"].astype(str) == reference_model
            ][[id_col, PRIMARY_ENDPOINT]].copy()

            reference = reference.rename(
                columns={PRIMARY_ENDPOINT: f"{reference_model}_{PRIMARY_ENDPOINT}"}
            )

            for model_name in SECONDARY_AQPR_MODEL_NAMES:
                candidate = region_df[
                    region_df["model_name"].astype(str) == model_name
                ][[id_col, PRIMARY_ENDPOINT]].copy()

                if len(candidate) == 0 or len(reference) == 0:
                    continue

                candidate = candidate.rename(
                    columns={PRIMARY_ENDPOINT: f"{model_name}_{PRIMARY_ENDPOINT}"}
                )

                paired = reference.merge(candidate, on=id_col, how="inner")

                if len(paired) == 0:
                    continue

                delta = (
                    paired[f"{model_name}_{PRIMARY_ENDPOINT}"]
                    - paired[f"{reference_model}_{PRIMARY_ENDPOINT}"]
                )

                stat, p_value, n_valid = safe_wilcoxon_paired(delta.values)
                ci_low, ci_high, n_boot = bootstrap_mean_ci(delta.values)

                rows.append(
                    {
                        "level": level_name,
                        "reference_model": reference_model,
                        "comparison_model": model_name,
                        "method_name": method_name,
                        "region_name": region_name,
                        "paired_units": int(len(paired)),
                        "endpoint": PRIMARY_ENDPOINT,
                        "delta_comparison_minus_reference_mean": float(np.mean(delta)),
                        "delta_comparison_minus_reference_median": float(np.median(delta)),
                        "delta_ci_low": ci_low,
                        "delta_ci_high": ci_high,
                        "wilcoxon_two_sided_statistic": stat,
                        "wilcoxon_two_sided_p": p_value,
                        "n_valid_wilcoxon": n_valid,
                        "n_valid_bootstrap": n_boot,
                    }
                )

    output = pd.DataFrame(rows)

    if len(output) > 0:
        output["wilcoxon_two_sided_p_fdr"] = bh_fdr(
            output["wilcoxon_two_sided_p"].values
        )

    return output


def paired_method_comparisons(df, level_name):
    rows = []

    id_col = "sample_id" if level_name == "image" else "patient_id"

    for model_name in sorted(df["model_name"].astype(str).unique()):
        model_df = df[df["model_name"].astype(str) == model_name].copy()

        if not {"GradCAM", "IntegratedGradients"}.issubset(
            set(model_df["method_name"].astype(str))
        ):
            continue

        for region_name in sorted(model_df["region_name"].astype(str).unique()):
            region_df = model_df[model_df["region_name"].astype(str) == region_name].copy()

            gradcam = region_df[
                region_df["method_name"].astype(str) == "GradCAM"
            ][[id_col, PRIMARY_ENDPOINT]].copy()

            ig = region_df[
                region_df["method_name"].astype(str) == "IntegratedGradients"
            ][[id_col, PRIMARY_ENDPOINT]].copy()

            gradcam = gradcam.rename(columns={PRIMARY_ENDPOINT: "GradCAM"})
            ig = ig.rename(columns={PRIMARY_ENDPOINT: "IntegratedGradients"})

            paired = gradcam.merge(ig, on=id_col, how="inner")

            if len(paired) == 0:
                continue

            delta = paired["IntegratedGradients"] - paired["GradCAM"]
            stat, p_value, n_valid = safe_wilcoxon_paired(delta.values)
            ci_low, ci_high, n_boot = bootstrap_mean_ci(delta.values)

            rows.append(
                {
                    "level": level_name,
                    "model_name": model_name,
                    "region_name": region_name,
                    "paired_units": int(len(paired)),
                    "endpoint": PRIMARY_ENDPOINT,
                    "delta_ig_minus_gradcam_mean": float(np.mean(delta)),
                    "delta_ig_minus_gradcam_median": float(np.median(delta)),
                    "delta_ci_low": ci_low,
                    "delta_ci_high": ci_high,
                    "wilcoxon_two_sided_statistic": stat,
                    "wilcoxon_two_sided_p": p_value,
                    "n_valid_wilcoxon": n_valid,
                    "n_valid_bootstrap": n_boot,
                }
            )

    output = pd.DataFrame(rows)

    if len(output) > 0:
        output["wilcoxon_two_sided_p_fdr"] = bh_fdr(
            output["wilcoxon_two_sided_p"].values
        )

    return output


def strata_statistics(df, level_name, strata_col, positive_value, negative_value):
    rows = []

    group_cols = [
        "model_name",
        "method_name",
        "aqpr_panel_role",
        "region_name",
    ]

    for keys, group in df.groupby(group_cols):
        model_name, method_name, role, region_name = keys

        group_a = pd.to_numeric(
            group.loc[group[strata_col] == positive_value, PRIMARY_ENDPOINT],
            errors="coerce",
        ).values

        group_b = pd.to_numeric(
            group.loc[group[strata_col] == negative_value, PRIMARY_ENDPOINT],
            errors="coerce",
        ).values

        stat, p_value, n_a, n_b = safe_mannwhitney(group_a, group_b)

        group_a = group_a[np.isfinite(group_a)]
        group_b = group_b[np.isfinite(group_b)]

        rows.append(
            {
                "level": level_name,
                "model_name": model_name,
                "method_name": method_name,
                "aqpr_panel_role": role,
                "region_name": region_name,
                "strata_column": strata_col,
                "positive_value": positive_value,
                "negative_value": negative_value,
                "n_positive_group": int(n_a),
                "n_negative_group": int(n_b),
                "positive_group_mean": float(np.mean(group_a)) if len(group_a) > 0 else np.nan,
                "negative_group_mean": float(np.mean(group_b)) if len(group_b) > 0 else np.nan,
                "difference_positive_minus_negative": (
                    float(np.mean(group_a) - np.mean(group_b))
                    if len(group_a) > 0 and len(group_b) > 0
                    else np.nan
                ),
                "mannwhitney_two_sided_statistic": stat,
                "mannwhitney_two_sided_p": p_value,
            }
        )

    output = pd.DataFrame(rows)

    if len(output) > 0:
        output["mannwhitney_two_sided_p_fdr"] = bh_fdr(
            output["mannwhitney_two_sided_p"].values
        )

    return output


if not HEATMAP_PROGRESS_CSV.exists():
    raise FileNotFoundError(
        f"Missing Cell 5 progress CSV: {HEATMAP_PROGRESS_CSV}. "
        "Run Cell 5 until all tasks are complete."
    )

heatmap_progress = pd.read_csv(HEATMAP_PROGRESS_CSV)

if "task_complete" not in heatmap_progress.columns:
    raise ValueError("Missing task_complete column in heatmap progress CSV.")

heatmap_progress["task_complete"] = as_bool_series(heatmap_progress["task_complete"])

if not heatmap_progress["task_complete"].all():
    print("Incomplete Cell 5 tasks:")
    print(
        heatmap_progress[
            [
                "model_name",
                "method_name",
                "completed_images",
                "pending_images",
                "task_complete",
            ]
        ].to_string(index=False)
    )

    raise RuntimeError("Cell 5 is not complete. Re-run Cell 5 until all tasks are complete.")

metric_parts = []

for _, row in heatmap_progress.iterrows():
    metrics_path = Path(str(row["metrics_csv"]))

    if not metrics_path.exists():
        raise FileNotFoundError(f"Missing metrics CSV: {metrics_path}")

    task_metrics = pd.read_csv(metrics_path)
    metric_parts.append(task_metrics)

heatmap_metrics = pd.concat(metric_parts, axis=0, ignore_index=True)

required_metric_cols = [
    "sample_id",
    "patient_id",
    "model_name",
    "method_name",
    "aqpr_panel_role",
    "region_name",
    "y_true",
    "gon_label",
    "xai_probability",
    "temperature_scaled_probability",
    "threshold",
    "predicted_label",
    "prediction_correct",
    "observed_energy_fraction",
    "random_energy_fraction_mean",
    "observed_minus_random_mean",
    "enrichment_ratio",
    "empirical_p_greater_equal",
]

missing_cols = [
    col for col in required_metric_cols
    if col not in heatmap_metrics.columns
]

if missing_cols:
    raise ValueError(f"Missing columns in consolidated heatmap metrics: {missing_cols}")

heatmap_metrics["sample_id"] = heatmap_metrics["sample_id"].astype(str)
heatmap_metrics["patient_id"] = heatmap_metrics["patient_id"].astype(str)
heatmap_metrics["model_name"] = heatmap_metrics["model_name"].astype(str)
heatmap_metrics["method_name"] = heatmap_metrics["method_name"].astype(str)
heatmap_metrics["region_name"] = heatmap_metrics["region_name"].astype(str)
heatmap_metrics["prediction_correct"] = as_bool_series(heatmap_metrics["prediction_correct"])

numeric_cols = [
    "y_true",
    "xai_probability",
    "temperature_scaled_probability",
    "threshold",
    "predicted_label",
    "observed_energy_fraction",
    "random_energy_fraction_mean",
    "observed_minus_random_mean",
    "enrichment_ratio",
    "empirical_p_greater_equal",
    "mask_pixels",
    "fov_pixels",
]

for col in numeric_cols:
    if col in heatmap_metrics.columns:
        heatmap_metrics[col] = pd.to_numeric(heatmap_metrics[col], errors="coerce")

heatmap_metrics.to_csv(HEATMAP_CONSOLIDATED_CSV, index=False)

patient_metrics, inconsistent_patients = build_patient_level_metrics(heatmap_metrics)
patient_metrics.to_csv(HEATMAP_PATIENT_LEVEL_CSV, index=False)

image_stats = summarize_endpoint(
    df=heatmap_metrics,
    level_name="image",
)

patient_stats = summarize_endpoint(
    df=patient_metrics,
    level_name="patient",
)

image_stats.to_csv(HEATMAP_IMAGE_LEVEL_STATS_CSV, index=False)
patient_stats.to_csv(HEATMAP_PATIENT_LEVEL_STATS_CSV, index=False)

image_model_comparisons = paired_model_comparisons(
    df=heatmap_metrics,
    level_name="image",
)

patient_model_comparisons = paired_model_comparisons(
    df=patient_metrics,
    level_name="patient",
)

model_comparisons = pd.concat(
    [image_model_comparisons, patient_model_comparisons],
    axis=0,
    ignore_index=True,
)

model_comparisons.to_csv(HEATMAP_MODEL_COMPARISON_CSV, index=False)

image_method_comparisons = paired_method_comparisons(
    df=heatmap_metrics,
    level_name="image",
)

patient_method_comparisons = paired_method_comparisons(
    df=patient_metrics,
    level_name="patient",
)

method_comparisons = pd.concat(
    [image_method_comparisons, patient_method_comparisons],
    axis=0,
    ignore_index=True,
)

method_comparisons.to_csv(HEATMAP_METHOD_COMPARISON_CSV, index=False)

image_label_strata = strata_statistics(
    df=heatmap_metrics,
    level_name="image",
    strata_col="y_true",
    positive_value=1,
    negative_value=0,
)

patient_label_strata = strata_statistics(
    df=patient_metrics,
    level_name="patient",
    strata_col="y_true",
    positive_value=1,
    negative_value=0,
)

label_strata = pd.concat(
    [image_label_strata, patient_label_strata],
    axis=0,
    ignore_index=True,
)

label_strata.to_csv(HEATMAP_LABEL_STRATA_CSV, index=False)

image_correctness_strata = strata_statistics(
    df=heatmap_metrics,
    level_name="image",
    strata_col="prediction_correct",
    positive_value=True,
    negative_value=False,
)

patient_correctness = patient_metrics.copy()
patient_correctness["prediction_correct"] = patient_correctness["prediction_correct_fraction"] >= 0.5

patient_correctness_strata = strata_statistics(
    df=patient_correctness,
    level_name="patient",
    strata_col="prediction_correct",
    positive_value=True,
    negative_value=False,
)

correctness_strata = pd.concat(
    [image_correctness_strata, patient_correctness_strata],
    axis=0,
    ignore_index=True,
)

correctness_strata.to_csv(HEATMAP_CORRECTNESS_STRATA_CSV, index=False)

primary_patient_result = patient_stats[
    (patient_stats["model_name"].astype(str) == PRIMARY_MODEL_NAME)
    & (patient_stats["method_name"].astype(str) == "GradCAM")
    & (patient_stats["region_name"].astype(str) == PRIMARY_REGION)
].copy()

if len(primary_patient_result) != 1:
    raise RuntimeError("Expected exactly one primary patient-level GradCAM OD_plus_OC result.")

primary_patient_result = primary_patient_result.iloc[0].to_dict()

summary_json = {
    "notebook": "08_anatomical_explainability_validation",
    "cell": "Cell 6",
    "analysis": "AQPR heatmap-XAI anatomical enrichment statistics",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_anatomical_dataset": "HYGD",
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "primary_endpoint": PRIMARY_ENDPOINT,
    "primary_region": PRIMARY_REGION,
    "n_models": int(heatmap_metrics["model_name"].nunique()),
    "n_model_method_tasks": int(
        heatmap_metrics[["model_name", "method_name"]].drop_duplicates().shape[0]
    ),
    "n_image_level_rows": int(len(heatmap_metrics)),
    "n_patient_level_rows": int(len(patient_metrics)),
    "n_unique_images": int(heatmap_metrics["sample_id"].nunique()),
    "n_unique_patients": int(heatmap_metrics["patient_id"].nunique()),
    "n_inconsistent_patient_labels": int(len(inconsistent_patients)),
    "primary_patient_gradcam_od_plus_oc_mean": float(primary_patient_result["primary_endpoint_mean"]),
    "primary_patient_gradcam_od_plus_oc_ci_low": float(primary_patient_result["primary_endpoint_ci_low"]),
    "primary_patient_gradcam_od_plus_oc_ci_high": float(primary_patient_result["primary_endpoint_ci_high"]),
    "primary_patient_gradcam_od_plus_oc_wilcoxon_p": float(
        primary_patient_result["wilcoxon_greater_than_zero_p"]
    ),
    "primary_patient_gradcam_od_plus_oc_wilcoxon_p_fdr": float(
        primary_patient_result["wilcoxon_greater_than_zero_p_fdr"]
    ),
    "patient_level_external_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
    "heatmap_consolidated_csv": str(HEATMAP_CONSOLIDATED_CSV),
    "heatmap_patient_level_csv": str(HEATMAP_PATIENT_LEVEL_CSV),
    "image_level_stats_csv": str(HEATMAP_IMAGE_LEVEL_STATS_CSV),
    "patient_level_stats_csv": str(HEATMAP_PATIENT_LEVEL_STATS_CSV),
    "model_comparison_csv": str(HEATMAP_MODEL_COMPARISON_CSV),
    "method_comparison_csv": str(HEATMAP_METHOD_COMPARISON_CSV),
    "label_strata_csv": str(HEATMAP_LABEL_STRATA_CSV),
    "correctness_strata_csv": str(HEATMAP_CORRECTNESS_STRATA_CSV),
}

with open(HEATMAP_STATISTICAL_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=2)

print("Consolidated AQPR heatmap-XAI tasks:")
print(
    heatmap_progress[
        [
            "model_name",
            "method_name",
            "completed_images",
            "pending_images",
            "task_complete",
        ]
    ].to_string(index=False)
)

print("\nPatient-level primary endpoint statistics:")
print(
    patient_stats[
        patient_stats["region_name"].astype(str).eq(PRIMARY_REGION)
    ][
        [
            "model_name",
            "method_name",
            "region_name",
            "n_units",
            "primary_endpoint_mean",
            "primary_endpoint_ci_low",
            "primary_endpoint_ci_high",
            "wilcoxon_greater_than_zero_p",
            "wilcoxon_greater_than_zero_p_fdr",
            "observed_energy_fraction_mean",
            "enrichment_ratio_median",
        ]
    ].to_string(index=False)
)

print("\nImage-level primary endpoint statistics:")
print(
    image_stats[
        image_stats["region_name"].astype(str).eq(PRIMARY_REGION)
    ][
        [
            "model_name",
            "method_name",
            "region_name",
            "n_units",
            "primary_endpoint_mean",
            "primary_endpoint_ci_low",
            "primary_endpoint_ci_high",
            "wilcoxon_greater_than_zero_p",
            "wilcoxon_greater_than_zero_p_fdr",
            "observed_energy_fraction_mean",
            "enrichment_ratio_median",
        ]
    ].to_string(index=False)
)

print("\nPaired model comparisons vs EfficientNetB0:")
print(
    model_comparisons[
        model_comparisons["region_name"].astype(str).eq(PRIMARY_REGION)
    ][
        [
            "level",
            "comparison_model",
            "method_name",
            "region_name",
            "paired_units",
            "delta_comparison_minus_reference_mean",
            "delta_ci_low",
            "delta_ci_high",
            "wilcoxon_two_sided_p",
            "wilcoxon_two_sided_p_fdr",
        ]
    ].to_string(index=False)
)

print("\nPaired method comparisons:")
print(
    method_comparisons[
        method_comparisons["region_name"].astype(str).eq(PRIMARY_REGION)
    ][
        [
            "level",
            "model_name",
            "region_name",
            "paired_units",
            "delta_ig_minus_gradcam_mean",
            "delta_ci_low",
            "delta_ci_high",
            "wilcoxon_two_sided_p",
            "wilcoxon_two_sided_p_fdr",
        ]
    ].to_string(index=False)
)

print("\nStatistical summary JSON:")
print(json.dumps(summary_json, indent=2))

print(f"\nConsolidated metrics: {HEATMAP_CONSOLIDATED_CSV}")
print(f"Patient-level metrics: {HEATMAP_PATIENT_LEVEL_CSV}")
print(f"Image-level stats: {HEATMAP_IMAGE_LEVEL_STATS_CSV}")
print(f"Patient-level stats: {HEATMAP_PATIENT_LEVEL_STATS_CSV}")
print(f"Model comparisons: {HEATMAP_MODEL_COMPARISON_CSV}")
print(f"Method comparisons: {HEATMAP_METHOD_COMPARISON_CSV}")
print(f"Label strata: {HEATMAP_LABEL_STRATA_CSV}")
print(f"Correctness strata: {HEATMAP_CORRECTNESS_STRATA_CSV}")
print(f"Summary JSON: {HEATMAP_STATISTICAL_SUMMARY_JSON}")

expected_tasks = sum(len(v) for v in FULL_HEATMAP_METHODS_BY_MODEL.values())

if heatmap_metrics[["model_name", "method_name"]].drop_duplicates().shape[0] != expected_tasks:
    raise RuntimeError("Unexpected number of completed model-method heatmap tasks.")

if heatmap_metrics["sample_id"].nunique() <= 0:
    raise RuntimeError("No image-level heatmap metrics found.")

if heatmap_metrics["patient_id"].nunique() <= 0:
    raise RuntimeError("No patient-level heatmap metrics found.")

od_plus_oc_rows = heatmap_metrics[
    heatmap_metrics["region_name"].astype(str) == PRIMARY_REGION
].copy()

if len(od_plus_oc_rows) == 0:
    raise RuntimeError(f"No {PRIMARY_REGION} rows found.")

if not np.isfinite(od_plus_oc_rows[PRIMARY_ENDPOINT]).all():
    raise RuntimeError(f"Non-finite {PRIMARY_ENDPOINT} values found for {PRIMARY_REGION}.")

if summary_json["n_inconsistent_patient_labels"] > 0:
    print("Warning: patient-level labels were aggregated using max(y_true) because mixed labels were detected.")

print("Status: PASS")

In [ ]:
# Cell 7: Run resumable AQPR anatomical occlusion with matched random controls on HYGD

import time
import hashlib
import gc
from tqdm.auto import tqdm

OCCLUSION_ROOT = RESULTS_08_DIR / "aqpr_panel_anatomical_occlusion"
OCCLUSION_ROOT.mkdir(parents=True, exist_ok=True)

OCCLUSION_PLAN_CSV = RESULTS_08_DIR / "notebook08_aqpr_anatomical_occlusion_plan.csv"
OCCLUSION_PROGRESS_CSV = RESULTS_08_DIR / "notebook08_aqpr_anatomical_occlusion_progress.csv"
OCCLUSION_CONFIG_JSON = RESULTS_08_DIR / "notebook08_aqpr_anatomical_occlusion_config.json"

# Automatic mode: leave as None.
# The cell will run the next incomplete model.
MODEL_TO_RUN = None

MAX_CASES_PER_EXECUTION = 96

RUN_OCCLUSION_ON_CUDA_ONLY = True

# We use blur as primary perturbation because outside_mean produced an unstable smoke-test artifact.
OCCLUSION_FILL_MODE = "blur"

# Avoid duplicate OD/OD_plus_OC definitions from the proxy mask.
OCCLUSION_REGIONS = [
    "OD_plus_OC",
    "OC",
    "Rim",
]

# Occlusion is much more expensive than heatmap enrichment.
# 32 matched controls per region is a practical primary setting.
N_OCCLUSION_RANDOM_CONTROLS = 32
OCCLUSION_RANDOM_CONTROL_MAX_ATTEMPTS = 5000
OCCLUSION_RANDOM_CONTROL_MIN_FOV_FRACTION = 0.985
OCCLUSION_RANDOM_CONTROL_MAX_IOU_WITH_TARGET = 0.03

PRIMARY_OCCLUSION_ENDPOINT = "observed_minus_random_probability_drop_mean"


def stable_int_seed_occlusion(*parts, modulo=2**32 - 1):
    text = "|".join([str(part) for part in parts])
    digest = hashlib.md5(text.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % modulo


def occlusion_task_output_paths(model_name):
    task_dir = OCCLUSION_ROOT / model_name
    task_dir.mkdir(parents=True, exist_ok=True)

    return {
        "task_dir": task_dir,
        "metrics_csv": task_dir / f"notebook08_{model_name}_hygd_anatomical_occlusion_metrics.csv",
        "case_log_csv": task_dir / f"notebook08_{model_name}_hygd_anatomical_occlusion_case_log.csv",
        "random_control_summary_csv": task_dir / f"notebook08_{model_name}_hygd_anatomical_occlusion_random_control_summary.csv",
        "region_summary_csv": task_dir / f"notebook08_{model_name}_hygd_anatomical_occlusion_region_summary.csv",
        "summary_json": task_dir / f"notebook08_{model_name}_hygd_anatomical_occlusion_summary.json",
    }


def load_existing_csv(path):
    path = Path(path)

    if path.exists():
        return pd.read_csv(path)

    return pd.DataFrame()


def get_completed_occlusion_sample_ids(case_log_csv):
    case_log = load_existing_csv(case_log_csv)

    if len(case_log) == 0:
        return set()

    if "completed" not in case_log.columns:
        return set()

    completed = case_log[as_bool_series(case_log["completed"])].copy()

    return set(completed["sample_id"].astype(str))


def build_occlusion_plan(analysis_df):
    rows = []

    for model_name in AQPR_PANEL_MODEL_NAMES:
        model_row = get_model_row_from_frozen_manifest(model_name)
        paths = occlusion_task_output_paths(model_name)

        rows.append(
            {
                "model_name": model_name,
                "aqpr_panel_role": AQPR_PANEL_ROLES[model_name],
                "checkpoint_path": str(model_row["checkpoint_path"]),
                "input_size": int(model_row["input_size"]),
                "temperature": float(model_row["temperature"]),
                "threshold": float(model_row["validation_selected_threshold"]),
                "expected_images": int(len(analysis_df)),
                "metrics_csv": str(paths["metrics_csv"]),
                "case_log_csv": str(paths["case_log_csv"]),
                "random_control_summary_csv": str(paths["random_control_summary_csv"]),
                "region_summary_csv": str(paths["region_summary_csv"]),
                "summary_json": str(paths["summary_json"]),
            }
        )

    return pd.DataFrame(rows)


def rgb_batch_to_tensor(rgb_images):
    tensors = []

    for image_rgb in rgb_images:
        image_float = image_rgb.astype(np.float32) / 255.0
        image_float = (image_float - IMAGENET_MEAN) / IMAGENET_STD
        image_chw = np.transpose(image_float, (2, 0, 1))
        tensors.append(torch.from_numpy(np.ascontiguousarray(image_chw)).float())

    return torch.stack(tensors, dim=0)


@torch.no_grad()
def predict_probabilities_from_rgb_batch(model, rgb_images, batch_size, device=DEVICE):
    model.eval()

    all_probs = []
    all_logits = []

    for start in range(0, len(rgb_images), int(batch_size)):
        batch_images = rgb_images[start:start + int(batch_size)]
        batch_tensor = rgb_batch_to_tensor(batch_images).to(device)

        logits = flatten_logits(model(batch_tensor))
        probs = torch.sigmoid(logits)

        all_logits.extend(logits.detach().cpu().numpy().astype(float).tolist())
        all_probs.extend(probs.detach().cpu().numpy().astype(float).tolist())

    return np.asarray(all_probs, dtype=float), np.asarray(all_logits, dtype=float)


def compute_occlusion_case_metrics(
    model,
    image_rgb,
    region_masks,
    fov_mask,
    sample_id,
    patient_id,
    model_name,
    row,
    batch_size,
):
    model_manifest_row = get_model_row_from_frozen_manifest(model_name)

    threshold = float(model_manifest_row["validation_selected_threshold"])
    temperature = float(model_manifest_row["temperature"])

    original_probs, original_logits = predict_probabilities_from_rgb_batch(
        model=model,
        rgb_images=[image_rgb],
        batch_size=batch_size,
        device=DEVICE,
    )

    original_probability = float(original_probs[0])
    original_logit = float(original_logits[0])

    predicted_label = int(original_probability >= threshold)
    prediction_correct = bool(predicted_label == int(row["gon_binary"]))

    temperature_scaled_probability = temperature_scaled_probability_from_probability(
        probability=original_probability,
        temperature=temperature,
    )

    metric_rows = []
    random_summary_rows = []

    for region_name in OCCLUSION_REGIONS:
        target_mask = region_masks[region_name].astype(bool)

        if target_mask.sum() == 0:
            continue

        rng_region = np.random.default_rng(
            stable_int_seed_occlusion(
                "cell7",
                model_name,
                sample_id,
                region_name,
                RANDOM_CONTROL_SEED,
            )
        )

        random_masks, random_audit = generate_matched_random_masks(
            target_mask=target_mask,
            fov_mask=fov_mask,
            exclusion_mask=region_masks["OD_plus_OC"],
            n_controls=N_OCCLUSION_RANDOM_CONTROLS,
            rng=rng_region,
            max_attempts=OCCLUSION_RANDOM_CONTROL_MAX_ATTEMPTS,
            min_fov_fraction=OCCLUSION_RANDOM_CONTROL_MIN_FOV_FRACTION,
            max_iou_with_target=OCCLUSION_RANDOM_CONTROL_MAX_IOU_WITH_TARGET,
        )

        observed_perturbed_rgb = perturb_rgb_with_mask(
            image_rgb=image_rgb,
            binary_mask=target_mask,
            fill_mode=OCCLUSION_FILL_MODE,
        )

        random_perturbed_images = [
            perturb_rgb_with_mask(
                image_rgb=image_rgb,
                binary_mask=random_mask,
                fill_mode=OCCLUSION_FILL_MODE,
            )
            for random_mask in random_masks
        ]

        perturbed_images = [observed_perturbed_rgb] + random_perturbed_images

        perturbed_probs, perturbed_logits = predict_probabilities_from_rgb_batch(
            model=model,
            rgb_images=perturbed_images,
            batch_size=batch_size,
            device=DEVICE,
        )

        observed_probability = float(perturbed_probs[0])
        observed_probability_drop = float(original_probability - observed_probability)

        random_probabilities = perturbed_probs[1:]
        random_probability_drops = original_probability - random_probabilities

        random_probability_drops = np.asarray(random_probability_drops, dtype=float)
        random_probability_drops = random_probability_drops[np.isfinite(random_probability_drops)]

        random_drop_mean = float(np.mean(random_probability_drops))
        random_drop_std = (
            float(np.std(random_probability_drops, ddof=1))
            if len(random_probability_drops) > 1
            else 0.0
        )

        observed_minus_random = float(observed_probability_drop - random_drop_mean)

        empirical_p_greater_equal = float(
            (np.sum(random_probability_drops >= observed_probability_drop) + 1)
            / (len(random_probability_drops) + 1)
        )

        accepted_random = random_audit[random_audit["accepted"].astype(bool)].copy()

        metric_rows.append(
            {
                "sample_id": str(sample_id),
                "patient_id": str(patient_id),
                "model_name": model_name,
                "aqpr_panel_role": AQPR_PANEL_ROLES[model_name],
                "region_name": region_name,
                "evaluation_dataset": "HYGD",
                "evaluation_split": "hygd_external_strict_anatomical",
                "y_true": int(row["gon_binary"]),
                "gon_label": str(row["gon_label"]),
                "original_probability": original_probability,
                "original_logit": original_logit,
                "temperature_scaled_probability": float(temperature_scaled_probability),
                "threshold": threshold,
                "predicted_label": predicted_label,
                "prediction_correct": prediction_correct,
                "occlusion_fill_mode": OCCLUSION_FILL_MODE,
                "observed_perturbed_probability": observed_probability,
                "observed_probability_drop": observed_probability_drop,
                "random_probability_drop_mean": random_drop_mean,
                "random_probability_drop_std": random_drop_std,
                "observed_minus_random_probability_drop_mean": observed_minus_random,
                "probability_drop_ratio": float(
                    observed_probability_drop / (random_drop_mean + 1e-8)
                ),
                "empirical_p_random_drop_greater_equal_observed": empirical_p_greater_equal,
                "mask_pixels": int(target_mask.sum()),
                "fov_pixels": int(fov_mask.sum()),
                "n_random_controls": int(len(random_probability_drops)),
            }
        )

        random_summary_rows.append(
            {
                "sample_id": str(sample_id),
                "patient_id": str(patient_id),
                "model_name": model_name,
                "region_name": region_name,
                "n_controls_requested": int(N_OCCLUSION_RANDOM_CONTROLS),
                "n_controls_accepted": int(len(accepted_random)),
                "attempts_total": int(len(random_audit)),
                "accepted_fov_fraction_mean": float(accepted_random["fov_fraction"].mean()),
                "accepted_iou_with_target_max": float(accepted_random["iou_with_target"].max()),
                "accepted_overlap_with_exclusion_max": float(
                    accepted_random["overlap_with_exclusion"].max()
                ),
                "random_probability_drop_mean": random_drop_mean,
                "random_probability_drop_std": random_drop_std,
            }
        )

    return pd.DataFrame(metric_rows), pd.DataFrame(random_summary_rows)


if DEVICE.type != "cuda" and RUN_OCCLUSION_ON_CUDA_ONLY:
    print(f"Device: {DEVICE}")
    print("Anatomical occlusion skipped because CUDA is not available.")
    print("Set RUN_OCCLUSION_ON_CUDA_ONLY = False only for intentional CPU execution.")
    print("Status: PASS")

else:
    if "hygd_strict_xai_df" not in globals():
        raise RuntimeError("Cell 2 must be executed before Cell 7.")

    if "load_aqpr_classifier" not in globals():
        raise RuntimeError("Cell 2 must be executed before Cell 7.")

    if "generate_matched_random_masks" not in globals():
        raise RuntimeError("Cell 4 must be executed before Cell 7.")

    if "perturb_rgb_with_mask" not in globals():
        raise RuntimeError("Cell 3 must be executed before Cell 7.")

    analysis_df = (
        hygd_strict_xai_df
        .copy()
        .sort_values(["patient_id", "sample_id"])
        .reset_index(drop=True)
    )

    analysis_df["sample_id"] = analysis_df["sample_id"].astype(str)
    analysis_df["patient_id"] = analysis_df["patient_id"].astype(str)

    occlusion_plan = build_occlusion_plan(analysis_df)

    progress_rows = []

    for _, plan_row in occlusion_plan.iterrows():
        completed_ids = get_completed_occlusion_sample_ids(plan_row["case_log_csv"])

        progress_rows.append(
            {
                **plan_row.to_dict(),
                "completed_images": int(len(completed_ids)),
                "pending_images": int(len(analysis_df) - len(completed_ids)),
                "task_complete": int(len(completed_ids)) == int(len(analysis_df)),
            }
        )

    occlusion_progress = pd.DataFrame(progress_rows)

    if MODEL_TO_RUN is not None:
        selected_tasks = occlusion_progress[
            occlusion_progress["model_name"].astype(str) == str(MODEL_TO_RUN)
        ].copy()

        if len(selected_tasks) != 1:
            raise RuntimeError(f"Could not identify requested occlusion task: {MODEL_TO_RUN}")

        task_row = selected_tasks.iloc[0].to_dict()

    else:
        pending_tasks = occlusion_progress[~occlusion_progress["task_complete"]].copy()

        if len(pending_tasks) == 0:
            occlusion_plan.to_csv(OCCLUSION_PLAN_CSV, index=False)
            occlusion_progress.to_csv(OCCLUSION_PROGRESS_CSV, index=False)

            print("All AQPR anatomical occlusion tasks are already complete.")
            print(
                occlusion_progress[
                    [
                        "model_name",
                        "completed_images",
                        "pending_images",
                        "task_complete",
                    ]
                ].to_string(index=False)
            )
            print("Status: PASS")

            raise SystemExit

        task_row = pending_tasks.iloc[0].to_dict()

    model_name = str(task_row["model_name"])
    model_row = get_model_row_from_frozen_manifest(model_name)

    checkpoint_path = Path(str(model_row["checkpoint_path"]))
    input_size = int(model_row["input_size"])
    model_batch_size = int(model_row["batch_size"]) if "batch_size" in model_row else 8
    occlusion_batch_size = max(1, min(16, model_batch_size))

    task_paths = occlusion_task_output_paths(model_name)

    completed_ids = get_completed_occlusion_sample_ids(task_paths["case_log_csv"])

    pending_df = analysis_df[
        ~analysis_df["sample_id"].astype(str).isin(completed_ids)
    ].copy().reset_index(drop=True)

    if MAX_CASES_PER_EXECUTION is not None:
        pending_df = pending_df.head(int(MAX_CASES_PER_EXECUTION)).copy()

    print("AQPR anatomical occlusion task:")
    print(f"  Model: {model_name}")
    print(f"  Device: {DEVICE}")
    print(f"  Input size: {input_size}")
    print(f"  Batch size: {occlusion_batch_size}")
    print(f"  Expected HYGD strict images: {len(analysis_df)}")
    print(f"  Already completed for this task: {len(completed_ids)}")
    print(f"  Cases scheduled in this execution: {len(pending_df)}")
    print(f"  Regions: {OCCLUSION_REGIONS}")
    print(f"  Fill mode: {OCCLUSION_FILL_MODE}")
    print(f"  Random controls per region: {N_OCCLUSION_RANDOM_CONTROLS}")

    if len(pending_df) == 0:
        print("This model task is already complete. Re-run Cell 7 to select the next pending model.")
        print("Status: PASS")

        raise SystemExit

    model, checkpoint, resolved_timm_name = load_aqpr_classifier(
        model_name=model_name,
        checkpoint_path=checkpoint_path,
        input_size=input_size,
        device=DEVICE,
    )

    existing_metrics = load_existing_csv(task_paths["metrics_csv"])
    existing_case_log = load_existing_csv(task_paths["case_log_csv"])
    existing_random_summary = load_existing_csv(task_paths["random_control_summary_csv"])

    new_metric_parts = []
    new_case_rows = []
    new_random_summary_parts = []

    start_time = time.time()

    iterator = tqdm(
        pending_df.iterrows(),
        total=len(pending_df),
        desc=f"{model_name} | anatomical occlusion",
    )

    for _, row in iterator:
        sample_id = str(row["sample_id"])
        patient_id = str(row["patient_id"])

        case_start = time.time()

        try:
            _, _, image_rgb_resized = preprocess_classifier_image_from_path(
                image_path=row["classifier_image_path"],
                input_size=input_size,
            )

            _, region_masks = build_region_masks_from_mask_path(
                mask_path=row["mask_512_path"],
                input_size=input_size,
            )

            fov_mask = compute_fov_mask_from_rgb(
                image_rgb=image_rgb_resized,
                threshold=8,
            )

            metrics_df, random_summary_df = compute_occlusion_case_metrics(
                model=model,
                image_rgb=image_rgb_resized,
                region_masks=region_masks,
                fov_mask=fov_mask,
                sample_id=sample_id,
                patient_id=patient_id,
                model_name=model_name,
                row=row,
                batch_size=occlusion_batch_size,
            )

            new_metric_parts.append(metrics_df)
            new_random_summary_parts.append(random_summary_df)

            new_case_rows.append(
                {
                    "sample_id": sample_id,
                    "patient_id": patient_id,
                    "model_name": model_name,
                    "completed": True,
                    "status": "pass",
                    "error": "",
                    "elapsed_sec": round(time.time() - case_start, 3),
                    "n_metric_rows": int(len(metrics_df)),
                    "n_random_summary_rows": int(len(random_summary_df)),
                }
            )

        except Exception as exc:
            new_case_rows.append(
                {
                    "sample_id": sample_id,
                    "patient_id": patient_id,
                    "model_name": model_name,
                    "completed": False,
                    "status": "fail",
                    "error": repr(exc),
                    "elapsed_sec": round(time.time() - case_start, 3),
                    "n_metric_rows": 0,
                    "n_random_summary_rows": 0,
                }
            )

        current_case_log = pd.DataFrame(new_case_rows)

        combined_case_log = pd.concat(
            [existing_case_log, current_case_log],
            axis=0,
            ignore_index=True,
        )

        combined_case_log = (
            combined_case_log
            .drop_duplicates(
                subset=["sample_id", "model_name"],
                keep="last",
            )
            .reset_index(drop=True)
        )

        combined_case_log.to_csv(task_paths["case_log_csv"], index=False)

        if len(new_metric_parts) > 0:
            combined_metrics = pd.concat(
                [existing_metrics] + new_metric_parts,
                axis=0,
                ignore_index=True,
            )

            combined_metrics = (
                combined_metrics
                .drop_duplicates(
                    subset=["sample_id", "model_name", "region_name"],
                    keep="last",
                )
                .reset_index(drop=True)
            )

            combined_metrics.to_csv(task_paths["metrics_csv"], index=False)

        if len(new_random_summary_parts) > 0:
            combined_random_summary = pd.concat(
                [existing_random_summary] + new_random_summary_parts,
                axis=0,
                ignore_index=True,
            )

            combined_random_summary = (
                combined_random_summary
                .drop_duplicates(
                    subset=["sample_id", "model_name", "region_name"],
                    keep="last",
                )
                .reset_index(drop=True)
            )

            combined_random_summary.to_csv(task_paths["random_control_summary_csv"], index=False)

    elapsed_min = round((time.time() - start_time) / 60.0, 3)

    case_log = pd.read_csv(task_paths["case_log_csv"])
    failed_rows = case_log[~as_bool_series(case_log["completed"])].copy()

    if len(failed_rows) > 0 and not Path(task_paths["metrics_csv"]).exists():
        print("\nFirst failed cases:")
        print(
            failed_rows[
                [
                    "sample_id",
                    "patient_id",
                    "model_name",
                    "status",
                    "error",
                ]
            ].head(20).to_string(index=False)
        )

        raise RuntimeError(
            "No occlusion metrics file was generated because all scheduled cases failed."
        )

    if not Path(task_paths["metrics_csv"]).exists():
        raise FileNotFoundError(f"Occlusion metrics CSV was not created: {task_paths['metrics_csv']}")

    if not Path(task_paths["random_control_summary_csv"]).exists():
        raise FileNotFoundError(
            f"Occlusion random-control summary CSV was not created: "
            f"{task_paths['random_control_summary_csv']}"
        )

    task_metrics = pd.read_csv(task_paths["metrics_csv"])
    random_summary = pd.read_csv(task_paths["random_control_summary_csv"])

    completed_total = int(as_bool_series(case_log["completed"]).sum())
    failed_total = int((~as_bool_series(case_log["completed"])).sum())
    task_complete = completed_total == len(analysis_df)

    region_summary = (
        task_metrics
        .groupby("region_name")
        .agg(
            n_rows=("sample_id", "count"),
            n_images=("sample_id", "nunique"),
            observed_probability_drop_mean=("observed_probability_drop", "mean"),
            random_probability_drop_mean=("random_probability_drop_mean", "mean"),
            observed_minus_random_probability_drop_mean=("observed_minus_random_probability_drop_mean", "mean"),
            probability_drop_ratio_median=("probability_drop_ratio", "median"),
            empirical_p_random_drop_greater_equal_observed_mean=(
                "empirical_p_random_drop_greater_equal_observed",
                "mean",
            ),
        )
        .reset_index()
    )

    region_summary.to_csv(task_paths["region_summary_csv"], index=False)

    task_summary = {
        "status": "PASS" if task_complete else "IN_PROGRESS",
        "model_name": model_name,
        "resolved_timm_model_name": resolved_timm_name,
        "external_dataset": "HYGD",
        "cohort": "hygd_external_strict_anatomical",
        "expected_images": int(len(analysis_df)),
        "completed_images": completed_total,
        "failed_images": failed_total,
        "scheduled_this_execution": int(len(pending_df)),
        "regions": OCCLUSION_REGIONS,
        "fill_mode": OCCLUSION_FILL_MODE,
        "n_random_controls": int(N_OCCLUSION_RANDOM_CONTROLS),
        "primary_endpoint": PRIMARY_OCCLUSION_ENDPOINT,
        "elapsed_min_current_execution": elapsed_min,
        "metrics_csv": str(task_paths["metrics_csv"]),
        "case_log_csv": str(task_paths["case_log_csv"]),
        "random_control_summary_csv": str(task_paths["random_control_summary_csv"]),
        "region_summary_csv": str(task_paths["region_summary_csv"]),
        "patient_level_external_analysis_allowed": True,
        "internal_airogslight_patient_level_claim_allowed": False,
    }

    with open(task_paths["summary_json"], "w", encoding="utf-8") as f:
        json.dump(task_summary, f, indent=2)

    progress_rows = []

    for _, plan_row in occlusion_plan.iterrows():
        completed_ids_updated = get_completed_occlusion_sample_ids(plan_row["case_log_csv"])

        progress_rows.append(
            {
                **plan_row.to_dict(),
                "completed_images": int(len(completed_ids_updated)),
                "pending_images": int(len(analysis_df) - len(completed_ids_updated)),
                "task_complete": int(len(completed_ids_updated)) == int(len(analysis_df)),
            }
        )

    occlusion_progress = pd.DataFrame(progress_rows)

    occlusion_plan.to_csv(OCCLUSION_PLAN_CSV, index=False)
    occlusion_progress.to_csv(OCCLUSION_PROGRESS_CSV, index=False)

    occlusion_config = {
        "notebook": "08_anatomical_explainability_validation",
        "cell": "Cell 7",
        "analysis": "AQPR panel anatomy-guided occlusion on HYGD",
        "external_dataset": "HYGD",
        "cohort": "hygd_external_strict_anatomical",
        "model_panel": AQPR_PANEL_MODEL_NAMES,
        "regions": OCCLUSION_REGIONS,
        "fill_mode": OCCLUSION_FILL_MODE,
        "n_random_controls": int(N_OCCLUSION_RANDOM_CONTROLS),
        "max_cases_per_execution": (
            None if MAX_CASES_PER_EXECUTION is None else int(MAX_CASES_PER_EXECUTION)
        ),
        "primary_endpoint": PRIMARY_OCCLUSION_ENDPOINT,
        "progress_csv": str(OCCLUSION_PROGRESS_CSV),
        "plan_csv": str(OCCLUSION_PLAN_CSV),
        "reason_for_blur_fill_mode": (
            "Blur occlusion is used as the primary perturbation because outside_mean "
            "created an unstable smoke-test artifact."
        ),
    }

    with open(OCCLUSION_CONFIG_JSON, "w", encoding="utf-8") as f:
        json.dump(occlusion_config, f, indent=2)

    del model, checkpoint

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

    print("\nCurrent occlusion task summary:")
    print(json.dumps(task_summary, indent=2))

    print("\nRegion summary for current occlusion task:")
    print(region_summary.to_string(index=False))

    print("\nOverall AQPR anatomical occlusion progress:")
    print(
        occlusion_progress[
            [
                "model_name",
                "completed_images",
                "pending_images",
                "task_complete",
            ]
        ].to_string(index=False)
    )

    print(f"\nMetrics CSV: {task_paths['metrics_csv']}")
    print(f"Case log CSV: {task_paths['case_log_csv']}")
    print(f"Random-control summary CSV: {task_paths['random_control_summary_csv']}")
    print(f"Region summary CSV: {task_paths['region_summary_csv']}")
    print(f"Overall progress CSV: {OCCLUSION_PROGRESS_CSV}")
    print(f"Config JSON: {OCCLUSION_CONFIG_JSON}")

    if failed_total > 0:
        print("\nFailed cases:")
        print(
            failed_rows[
                [
                    "sample_id",
                    "patient_id",
                    "model_name",
                    "status",
                    "error",
                ]
            ].tail(20).to_string(index=False)
        )

        raise RuntimeError(f"{model_name} anatomical occlusion has failed cases: {failed_total}")

    if len(task_metrics) == 0:
        raise RuntimeError("No anatomical occlusion metrics were generated for current task.")

    primary_region_metrics = task_metrics[
        task_metrics["region_name"].astype(str) == "OD_plus_OC"
    ].copy()

    if len(primary_region_metrics) == 0:
        raise RuntimeError("No OD_plus_OC occlusion metrics generated for current task.")

    if not np.isfinite(primary_region_metrics[PRIMARY_OCCLUSION_ENDPOINT]).all():
        raise RuntimeError(
            f"Non-finite {PRIMARY_OCCLUSION_ENDPOINT} values found for OD_plus_OC."
        )

    if occlusion_progress["task_complete"].all():
        print("All AQPR anatomical occlusion tasks are complete.")
        print("Status: PASS")
    else:
        print("Status: IN_PROGRESS - re-run Cell 7 until all models show task_complete=True.")

In [ ]:
# Cell 8: Consolidate AQPR anatomical occlusion metrics and run image-/patient-level statistics

from scipy.stats import wilcoxon, mannwhitneyu

OCCLUSION_ROOT = RESULTS_08_DIR / "aqpr_panel_anatomical_occlusion"
OCCLUSION_PROGRESS_CSV = RESULTS_08_DIR / "notebook08_aqpr_anatomical_occlusion_progress.csv"

OCCLUSION_CONSOLIDATED_CSV = RESULTS_08_DIR / "notebook08_aqpr_anatomical_occlusion_metrics_consolidated.csv"
OCCLUSION_PATIENT_LEVEL_CSV = RESULTS_08_DIR / "notebook08_aqpr_anatomical_occlusion_patient_level_metrics.csv"
OCCLUSION_IMAGE_LEVEL_STATS_CSV = RESULTS_08_DIR / "notebook08_aqpr_anatomical_occlusion_image_level_statistics.csv"
OCCLUSION_PATIENT_LEVEL_STATS_CSV = RESULTS_08_DIR / "notebook08_aqpr_anatomical_occlusion_patient_level_statistics.csv"
OCCLUSION_MODEL_COMPARISON_CSV = RESULTS_08_DIR / "notebook08_aqpr_anatomical_occlusion_paired_model_comparisons.csv"
OCCLUSION_LABEL_STRATA_CSV = RESULTS_08_DIR / "notebook08_aqpr_anatomical_occlusion_label_strata_statistics.csv"
OCCLUSION_CORRECTNESS_STRATA_CSV = RESULTS_08_DIR / "notebook08_aqpr_anatomical_occlusion_correctness_strata_statistics.csv"
OCCLUSION_STATISTICAL_SUMMARY_JSON = RESULTS_08_DIR / "notebook08_aqpr_anatomical_occlusion_statistical_summary.json"

PRIMARY_OCCLUSION_REGION = "OD_plus_OC"
PRIMARY_OCCLUSION_ENDPOINT = "observed_minus_random_probability_drop_mean"

SECONDARY_OCCLUSION_ENDPOINTS = [
    "observed_probability_drop",
    "random_probability_drop_mean",
    "probability_drop_ratio",
    "empirical_p_random_drop_greater_equal_observed",
]

OCC_BOOTSTRAP_N = 5000
OCC_BOOTSTRAP_SEED = SEED


def bootstrap_mean_ci_occlusion(values, n_bootstrap=OCC_BOOTSTRAP_N, seed=OCC_BOOTSTRAP_SEED):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan, np.nan, 0

    if len(values) == 1:
        value = float(values[0])
        return value, value, value, 1

    local_rng = np.random.default_rng(seed)
    sampled = local_rng.choice(values, size=(int(n_bootstrap), len(values)), replace=True)
    boot_means = sampled.mean(axis=1)

    return (
        float(np.mean(values)),
        float(np.percentile(boot_means, 2.5)),
        float(np.percentile(boot_means, 97.5)),
        int(len(boot_means)),
    )


def signed_rank_biserial_effect_occlusion(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    values = values[values != 0]

    if len(values) == 0:
        return np.nan

    ranks = pd.Series(np.abs(values)).rank(method="average").values

    positive_rank_sum = ranks[values > 0].sum()
    negative_rank_sum = ranks[values < 0].sum()
    total_rank_sum = ranks.sum()

    if total_rank_sum == 0:
        return np.nan

    return float((positive_rank_sum - negative_rank_sum) / total_rank_sum)


def safe_wilcoxon_against_zero_occlusion(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 2:
        return {
            "wilcoxon_statistic_two_sided": np.nan,
            "wilcoxon_p_two_sided": np.nan,
            "wilcoxon_statistic_greater": np.nan,
            "wilcoxon_p_greater": np.nan,
            "wilcoxon_statistic_less": np.nan,
            "wilcoxon_p_less": np.nan,
            "signed_rank_biserial_effect": np.nan,
            "n_valid_wilcoxon": int(len(values)),
        }

    if np.allclose(values, 0.0):
        return {
            "wilcoxon_statistic_two_sided": 0.0,
            "wilcoxon_p_two_sided": 1.0,
            "wilcoxon_statistic_greater": 0.0,
            "wilcoxon_p_greater": 1.0,
            "wilcoxon_statistic_less": 0.0,
            "wilcoxon_p_less": 1.0,
            "signed_rank_biserial_effect": 0.0,
            "n_valid_wilcoxon": int(len(values)),
        }

    try:
        two_sided = wilcoxon(values, alternative="two-sided", zero_method="wilcox")
        greater = wilcoxon(values, alternative="greater", zero_method="wilcox")
        less = wilcoxon(values, alternative="less", zero_method="wilcox")

        return {
            "wilcoxon_statistic_two_sided": float(two_sided.statistic),
            "wilcoxon_p_two_sided": float(two_sided.pvalue),
            "wilcoxon_statistic_greater": float(greater.statistic),
            "wilcoxon_p_greater": float(greater.pvalue),
            "wilcoxon_statistic_less": float(less.statistic),
            "wilcoxon_p_less": float(less.pvalue),
            "signed_rank_biserial_effect": signed_rank_biserial_effect_occlusion(values),
            "n_valid_wilcoxon": int(len(values)),
        }

    except ValueError:
        return {
            "wilcoxon_statistic_two_sided": np.nan,
            "wilcoxon_p_two_sided": np.nan,
            "wilcoxon_statistic_greater": np.nan,
            "wilcoxon_p_greater": np.nan,
            "wilcoxon_statistic_less": np.nan,
            "wilcoxon_p_less": np.nan,
            "signed_rank_biserial_effect": np.nan,
            "n_valid_wilcoxon": int(len(values)),
        }


def safe_wilcoxon_paired_occlusion(delta_values):
    delta_values = np.asarray(delta_values, dtype=float)
    delta_values = delta_values[np.isfinite(delta_values)]

    if len(delta_values) < 2:
        return np.nan, np.nan, np.nan, len(delta_values)

    if np.allclose(delta_values, 0.0):
        return 0.0, 1.0, 0.0, len(delta_values)

    try:
        stat, p_value = wilcoxon(
            delta_values,
            alternative="two-sided",
            zero_method="wilcox",
        )

        effect = signed_rank_biserial_effect_occlusion(delta_values)

        return float(stat), float(p_value), float(effect), len(delta_values)

    except ValueError:
        return np.nan, np.nan, np.nan, len(delta_values)


def safe_mannwhitney_occlusion(group_a, group_b):
    group_a = np.asarray(group_a, dtype=float)
    group_b = np.asarray(group_b, dtype=float)

    group_a = group_a[np.isfinite(group_a)]
    group_b = group_b[np.isfinite(group_b)]

    if len(group_a) < 2 or len(group_b) < 2:
        return np.nan, np.nan, np.nan, len(group_a), len(group_b)

    try:
        stat, p_value = mannwhitneyu(
            group_a,
            group_b,
            alternative="two-sided",
        )

        rank_biserial = float((2.0 * stat / (len(group_a) * len(group_b))) - 1.0)

        return float(stat), float(p_value), rank_biserial, len(group_a), len(group_b)

    except ValueError:
        return np.nan, np.nan, np.nan, len(group_a), len(group_b)


def bh_fdr_occlusion(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full_like(p_values, fill_value=np.nan, dtype=float)

    valid = np.isfinite(p_values)

    if valid.sum() == 0:
        return adjusted

    p = p_values[valid]
    m = len(p)

    order = np.argsort(p)
    ranked = p[order]

    adjusted_ranked = ranked * m / (np.arange(m) + 1)
    adjusted_ranked = np.minimum.accumulate(adjusted_ranked[::-1])[::-1]
    adjusted_ranked = np.clip(adjusted_ranked, 0.0, 1.0)

    valid_indices = np.where(valid)[0]
    adjusted[valid_indices[order]] = adjusted_ranked

    return adjusted


def summarize_occlusion_endpoint(df, level_name):
    rows = []

    group_cols = [
        "model_name",
        "aqpr_panel_role",
        "region_name",
        "occlusion_fill_mode",
    ]

    for keys, group in df.groupby(group_cols):
        model_name, role, region_name, fill_mode = keys

        values = pd.to_numeric(group[PRIMARY_OCCLUSION_ENDPOINT], errors="coerce").values
        values = values[np.isfinite(values)]

        mean_value, ci_low, ci_high, n_bootstrap = bootstrap_mean_ci_occlusion(values)
        wilcoxon_result = safe_wilcoxon_against_zero_occlusion(values)

        row = {
            "level": level_name,
            "model_name": model_name,
            "aqpr_panel_role": role,
            "region_name": region_name,
            "occlusion_fill_mode": fill_mode,
            "n_units": int(len(values)),
            "primary_endpoint": PRIMARY_OCCLUSION_ENDPOINT,
            "primary_endpoint_mean": mean_value,
            "primary_endpoint_median": float(np.median(values)) if len(values) > 0 else np.nan,
            "primary_endpoint_std": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
            "primary_endpoint_ci_low": ci_low,
            "primary_endpoint_ci_high": ci_high,
            "n_valid_bootstrap": int(n_bootstrap),
            **wilcoxon_result,
        }

        for endpoint in SECONDARY_OCCLUSION_ENDPOINTS:
            endpoint_values = pd.to_numeric(group[endpoint], errors="coerce").values
            endpoint_values = endpoint_values[np.isfinite(endpoint_values)]

            row[f"{endpoint}_mean"] = (
                float(np.mean(endpoint_values))
                if len(endpoint_values) > 0
                else np.nan
            )

            row[f"{endpoint}_median"] = (
                float(np.median(endpoint_values))
                if len(endpoint_values) > 0
                else np.nan
            )

        rows.append(row)

    output = pd.DataFrame(rows)

    if len(output) > 0:
        output["wilcoxon_p_greater_fdr"] = bh_fdr_occlusion(
            output["wilcoxon_p_greater"].values
        )
        output["wilcoxon_p_two_sided_fdr"] = bh_fdr_occlusion(
            output["wilcoxon_p_two_sided"].values
        )
        output["wilcoxon_p_less_fdr"] = bh_fdr_occlusion(
            output["wilcoxon_p_less"].values
        )

    return output


def build_patient_level_occlusion_metrics(metrics_df):
    label_consistency = (
        metrics_df
        .groupby("patient_id")
        .agg(
            n_unique_labels=("y_true", "nunique"),
            patient_y_true=("y_true", "max"),
        )
        .reset_index()
    )

    inconsistent_patients = label_consistency[
        label_consistency["n_unique_labels"] > 1
    ].copy()

    if len(inconsistent_patients) > 0:
        print("Warning: patients with mixed image-level labels detected.")
        print(inconsistent_patients.head(20).to_string(index=False))

    patient_df = (
        metrics_df
        .groupby(
            [
                "model_name",
                "aqpr_panel_role",
                "region_name",
                "occlusion_fill_mode",
                "patient_id",
            ]
        )
        .agg(
            n_images=("sample_id", "nunique"),
            y_true=("y_true", "max"),
            gon_label=("gon_label", "first"),
            original_probability_mean=("original_probability", "mean"),
            temperature_scaled_probability_mean=("temperature_scaled_probability", "mean"),
            prediction_correct_fraction=("prediction_correct", "mean"),
            observed_perturbed_probability_mean=("observed_perturbed_probability", "mean"),
            observed_probability_drop=("observed_probability_drop", "mean"),
            random_probability_drop_mean=("random_probability_drop_mean", "mean"),
            observed_minus_random_probability_drop_mean=(
                "observed_minus_random_probability_drop_mean",
                "mean",
            ),
            probability_drop_ratio=("probability_drop_ratio", "median"),
            empirical_p_random_drop_greater_equal_observed=(
                "empirical_p_random_drop_greater_equal_observed",
                "mean",
            ),
            mask_pixels_mean=("mask_pixels", "mean"),
            fov_pixels_mean=("fov_pixels", "mean"),
            n_random_controls_mean=("n_random_controls", "mean"),
        )
        .reset_index()
    )

    return patient_df, inconsistent_patients


def paired_model_comparisons_occlusion(df, level_name):
    rows = []
    id_col = "sample_id" if level_name == "image" else "patient_id"

    reference_model = PRIMARY_MODEL_NAME

    for region_name in sorted(df["region_name"].astype(str).unique()):
        region_df = df[df["region_name"].astype(str) == region_name].copy()

        reference = region_df[
            region_df["model_name"].astype(str) == reference_model
        ][[id_col, PRIMARY_OCCLUSION_ENDPOINT]].copy()

        reference = reference.rename(
            columns={PRIMARY_OCCLUSION_ENDPOINT: f"{reference_model}_{PRIMARY_OCCLUSION_ENDPOINT}"}
        )

        for model_name in SECONDARY_AQPR_MODEL_NAMES:
            candidate = region_df[
                region_df["model_name"].astype(str) == model_name
            ][[id_col, PRIMARY_OCCLUSION_ENDPOINT]].copy()

            if len(reference) == 0 or len(candidate) == 0:
                continue

            candidate = candidate.rename(
                columns={PRIMARY_OCCLUSION_ENDPOINT: f"{model_name}_{PRIMARY_OCCLUSION_ENDPOINT}"}
            )

            paired = reference.merge(candidate, on=id_col, how="inner")

            if len(paired) == 0:
                continue

            delta = (
                paired[f"{model_name}_{PRIMARY_OCCLUSION_ENDPOINT}"]
                - paired[f"{reference_model}_{PRIMARY_OCCLUSION_ENDPOINT}"]
            )

            stat, p_value, effect, n_valid = safe_wilcoxon_paired_occlusion(delta.values)
            mean_delta, ci_low, ci_high, n_bootstrap = bootstrap_mean_ci_occlusion(delta.values)

            rows.append(
                {
                    "level": level_name,
                    "reference_model": reference_model,
                    "comparison_model": model_name,
                    "region_name": region_name,
                    "paired_units": int(len(paired)),
                    "endpoint": PRIMARY_OCCLUSION_ENDPOINT,
                    "delta_comparison_minus_reference_mean": mean_delta,
                    "delta_comparison_minus_reference_median": float(np.median(delta)),
                    "delta_ci_low": ci_low,
                    "delta_ci_high": ci_high,
                    "wilcoxon_two_sided_statistic": stat,
                    "wilcoxon_two_sided_p": p_value,
                    "signed_rank_biserial_effect": effect,
                    "n_valid_wilcoxon": n_valid,
                    "n_valid_bootstrap": n_bootstrap,
                }
            )

    output = pd.DataFrame(rows)

    if len(output) > 0:
        output["wilcoxon_two_sided_p_fdr"] = bh_fdr_occlusion(
            output["wilcoxon_two_sided_p"].values
        )

    return output


def strata_statistics_occlusion(df, level_name, strata_col, positive_value, negative_value):
    rows = []

    group_cols = [
        "model_name",
        "aqpr_panel_role",
        "region_name",
        "occlusion_fill_mode",
    ]

    for keys, group in df.groupby(group_cols):
        model_name, role, region_name, fill_mode = keys

        group_a = pd.to_numeric(
            group.loc[group[strata_col] == positive_value, PRIMARY_OCCLUSION_ENDPOINT],
            errors="coerce",
        ).values

        group_b = pd.to_numeric(
            group.loc[group[strata_col] == negative_value, PRIMARY_OCCLUSION_ENDPOINT],
            errors="coerce",
        ).values

        stat, p_value, effect, n_a, n_b = safe_mannwhitney_occlusion(group_a, group_b)

        group_a = group_a[np.isfinite(group_a)]
        group_b = group_b[np.isfinite(group_b)]

        rows.append(
            {
                "level": level_name,
                "model_name": model_name,
                "aqpr_panel_role": role,
                "region_name": region_name,
                "occlusion_fill_mode": fill_mode,
                "strata_column": strata_col,
                "positive_value": positive_value,
                "negative_value": negative_value,
                "n_positive_group": int(n_a),
                "n_negative_group": int(n_b),
                "positive_group_mean": float(np.mean(group_a)) if len(group_a) > 0 else np.nan,
                "negative_group_mean": float(np.mean(group_b)) if len(group_b) > 0 else np.nan,
                "difference_positive_minus_negative": (
                    float(np.mean(group_a) - np.mean(group_b))
                    if len(group_a) > 0 and len(group_b) > 0
                    else np.nan
                ),
                "mannwhitney_two_sided_statistic": stat,
                "mannwhitney_two_sided_p": p_value,
                "rank_biserial_effect": effect,
            }
        )

    output = pd.DataFrame(rows)

    if len(output) > 0:
        output["mannwhitney_two_sided_p_fdr"] = bh_fdr_occlusion(
            output["mannwhitney_two_sided_p"].values
        )

    return output


if not OCCLUSION_PROGRESS_CSV.exists():
    raise FileNotFoundError(
        f"Missing Cell 7 progress CSV: {OCCLUSION_PROGRESS_CSV}. "
        "Run Cell 7 until all AQPR occlusion tasks are complete."
    )

occlusion_progress = pd.read_csv(OCCLUSION_PROGRESS_CSV)

if "task_complete" not in occlusion_progress.columns:
    raise ValueError("Missing task_complete column in occlusion progress CSV.")

occlusion_progress["task_complete"] = as_bool_series(occlusion_progress["task_complete"])

if not occlusion_progress["task_complete"].all():
    print("Incomplete Cell 7 tasks:")
    print(
        occlusion_progress[
            [
                "model_name",
                "completed_images",
                "pending_images",
                "task_complete",
            ]
        ].to_string(index=False)
    )

    raise RuntimeError("Cell 7 is not complete. Re-run Cell 7 until all models are complete.")

metric_parts = []

for _, row in occlusion_progress.iterrows():
    metrics_path = Path(str(row["metrics_csv"]))

    if not metrics_path.exists():
        raise FileNotFoundError(f"Missing occlusion metrics CSV: {metrics_path}")

    task_metrics = pd.read_csv(metrics_path)
    metric_parts.append(task_metrics)

occlusion_metrics = pd.concat(metric_parts, axis=0, ignore_index=True)

required_cols = [
    "sample_id",
    "patient_id",
    "model_name",
    "aqpr_panel_role",
    "region_name",
    "y_true",
    "gon_label",
    "original_probability",
    "temperature_scaled_probability",
    "threshold",
    "predicted_label",
    "prediction_correct",
    "occlusion_fill_mode",
    "observed_perturbed_probability",
    "observed_probability_drop",
    "random_probability_drop_mean",
    "observed_minus_random_probability_drop_mean",
    "probability_drop_ratio",
    "empirical_p_random_drop_greater_equal_observed",
    "mask_pixels",
    "fov_pixels",
    "n_random_controls",
]

missing_cols = [
    col for col in required_cols
    if col not in occlusion_metrics.columns
]

if missing_cols:
    raise ValueError(f"Missing columns in consolidated occlusion metrics: {missing_cols}")

occlusion_metrics["sample_id"] = occlusion_metrics["sample_id"].astype(str)
occlusion_metrics["patient_id"] = occlusion_metrics["patient_id"].astype(str)
occlusion_metrics["model_name"] = occlusion_metrics["model_name"].astype(str)
occlusion_metrics["region_name"] = occlusion_metrics["region_name"].astype(str)
occlusion_metrics["occlusion_fill_mode"] = occlusion_metrics["occlusion_fill_mode"].astype(str)
occlusion_metrics["method_name"] = "AnatomicalOcclusion"

occlusion_metrics["prediction_correct"] = as_bool_series(occlusion_metrics["prediction_correct"])

numeric_cols = [
    "y_true",
    "original_probability",
    "temperature_scaled_probability",
    "threshold",
    "predicted_label",
    "observed_perturbed_probability",
    "observed_probability_drop",
    "random_probability_drop_mean",
    "random_probability_drop_std",
    "observed_minus_random_probability_drop_mean",
    "probability_drop_ratio",
    "empirical_p_random_drop_greater_equal_observed",
    "mask_pixels",
    "fov_pixels",
    "n_random_controls",
]

for col in numeric_cols:
    if col in occlusion_metrics.columns:
        occlusion_metrics[col] = pd.to_numeric(occlusion_metrics[col], errors="coerce")

occlusion_metrics = (
    occlusion_metrics
    .drop_duplicates(
        subset=["sample_id", "model_name", "region_name"],
        keep="last",
    )
    .reset_index(drop=True)
)

occlusion_metrics.to_csv(OCCLUSION_CONSOLIDATED_CSV, index=False)

patient_occlusion_metrics, inconsistent_patients = build_patient_level_occlusion_metrics(
    occlusion_metrics
)

patient_occlusion_metrics.to_csv(OCCLUSION_PATIENT_LEVEL_CSV, index=False)

image_stats = summarize_occlusion_endpoint(
    df=occlusion_metrics,
    level_name="image",
)

patient_stats = summarize_occlusion_endpoint(
    df=patient_occlusion_metrics,
    level_name="patient",
)

image_stats.to_csv(OCCLUSION_IMAGE_LEVEL_STATS_CSV, index=False)
patient_stats.to_csv(OCCLUSION_PATIENT_LEVEL_STATS_CSV, index=False)

image_model_comparisons = paired_model_comparisons_occlusion(
    df=occlusion_metrics,
    level_name="image",
)

patient_model_comparisons = paired_model_comparisons_occlusion(
    df=patient_occlusion_metrics,
    level_name="patient",
)

model_comparisons = pd.concat(
    [image_model_comparisons, patient_model_comparisons],
    axis=0,
    ignore_index=True,
)

model_comparisons.to_csv(OCCLUSION_MODEL_COMPARISON_CSV, index=False)

image_label_strata = strata_statistics_occlusion(
    df=occlusion_metrics,
    level_name="image",
    strata_col="y_true",
    positive_value=1,
    negative_value=0,
)

patient_label_strata = strata_statistics_occlusion(
    df=patient_occlusion_metrics,
    level_name="patient",
    strata_col="y_true",
    positive_value=1,
    negative_value=0,
)

label_strata = pd.concat(
    [image_label_strata, patient_label_strata],
    axis=0,
    ignore_index=True,
)

label_strata.to_csv(OCCLUSION_LABEL_STRATA_CSV, index=False)

image_correctness_strata = strata_statistics_occlusion(
    df=occlusion_metrics,
    level_name="image",
    strata_col="prediction_correct",
    positive_value=True,
    negative_value=False,
)

patient_correctness_df = patient_occlusion_metrics.copy()
patient_correctness_df["prediction_correct"] = (
    patient_correctness_df["prediction_correct_fraction"] >= 0.5
)

patient_correctness_strata = strata_statistics_occlusion(
    df=patient_correctness_df,
    level_name="patient",
    strata_col="prediction_correct",
    positive_value=True,
    negative_value=False,
)

correctness_strata = pd.concat(
    [image_correctness_strata, patient_correctness_strata],
    axis=0,
    ignore_index=True,
)

correctness_strata.to_csv(OCCLUSION_CORRECTNESS_STRATA_CSV, index=False)

primary_patient_result = patient_stats[
    (patient_stats["model_name"].astype(str) == PRIMARY_MODEL_NAME)
    & (patient_stats["region_name"].astype(str) == PRIMARY_OCCLUSION_REGION)
].copy()

if len(primary_patient_result) != 1:
    raise RuntimeError("Expected exactly one primary patient-level occlusion OD_plus_OC result.")

primary_patient_result = primary_patient_result.iloc[0].to_dict()

summary_json = {
    "notebook": "08_anatomical_explainability_validation",
    "cell": "Cell 8",
    "analysis": "AQPR anatomical occlusion statistics",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_anatomical_dataset": "HYGD",
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "primary_endpoint": PRIMARY_OCCLUSION_ENDPOINT,
    "primary_region": PRIMARY_OCCLUSION_REGION,
    "occlusion_fill_mode": str(occlusion_metrics["occlusion_fill_mode"].iloc[0]),
    "n_models": int(occlusion_metrics["model_name"].nunique()),
    "n_image_level_rows": int(len(occlusion_metrics)),
    "n_patient_level_rows": int(len(patient_occlusion_metrics)),
    "n_unique_images": int(occlusion_metrics["sample_id"].nunique()),
    "n_unique_patients": int(occlusion_metrics["patient_id"].nunique()),
    "n_regions": int(occlusion_metrics["region_name"].nunique()),
    "n_inconsistent_patient_labels": int(len(inconsistent_patients)),
    "primary_patient_occlusion_od_plus_oc_mean": float(primary_patient_result["primary_endpoint_mean"]),
    "primary_patient_occlusion_od_plus_oc_ci_low": float(primary_patient_result["primary_endpoint_ci_low"]),
    "primary_patient_occlusion_od_plus_oc_ci_high": float(primary_patient_result["primary_endpoint_ci_high"]),
    "primary_patient_occlusion_od_plus_oc_wilcoxon_p_greater": float(
        primary_patient_result["wilcoxon_p_greater"]
    ),
    "primary_patient_occlusion_od_plus_oc_wilcoxon_p_greater_fdr": float(
        primary_patient_result["wilcoxon_p_greater_fdr"]
    ),
    "primary_patient_occlusion_od_plus_oc_signed_rank_biserial_effect": float(
        primary_patient_result["signed_rank_biserial_effect"]
    ),
    "patient_level_external_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
    "occlusion_consolidated_csv": str(OCCLUSION_CONSOLIDATED_CSV),
    "occlusion_patient_level_csv": str(OCCLUSION_PATIENT_LEVEL_CSV),
    "image_level_stats_csv": str(OCCLUSION_IMAGE_LEVEL_STATS_CSV),
    "patient_level_stats_csv": str(OCCLUSION_PATIENT_LEVEL_STATS_CSV),
    "model_comparison_csv": str(OCCLUSION_MODEL_COMPARISON_CSV),
    "label_strata_csv": str(OCCLUSION_LABEL_STRATA_CSV),
    "correctness_strata_csv": str(OCCLUSION_CORRECTNESS_STRATA_CSV),
}

with open(OCCLUSION_STATISTICAL_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=2)

print("Consolidated AQPR anatomical occlusion tasks:")
print(
    occlusion_progress[
        [
            "model_name",
            "completed_images",
            "pending_images",
            "task_complete",
        ]
    ].to_string(index=False)
)

print("\nPatient-level occlusion statistics:")
print(
    patient_stats[
        patient_stats["region_name"].astype(str).eq(PRIMARY_OCCLUSION_REGION)
    ][
        [
            "model_name",
            "region_name",
            "n_units",
            "primary_endpoint_mean",
            "primary_endpoint_ci_low",
            "primary_endpoint_ci_high",
            "wilcoxon_p_greater",
            "wilcoxon_p_greater_fdr",
            "wilcoxon_p_less",
            "wilcoxon_p_less_fdr",
            "signed_rank_biserial_effect",
            "observed_probability_drop_mean",
            "random_probability_drop_mean_mean",
            "probability_drop_ratio_median",
        ]
    ].to_string(index=False)
)

print("\nImage-level occlusion statistics:")
print(
    image_stats[
        image_stats["region_name"].astype(str).eq(PRIMARY_OCCLUSION_REGION)
    ][
        [
            "model_name",
            "region_name",
            "n_units",
            "primary_endpoint_mean",
            "primary_endpoint_ci_low",
            "primary_endpoint_ci_high",
            "wilcoxon_p_greater",
            "wilcoxon_p_greater_fdr",
            "wilcoxon_p_less",
            "wilcoxon_p_less_fdr",
            "signed_rank_biserial_effect",
            "observed_probability_drop_mean",
            "random_probability_drop_mean_mean",
            "probability_drop_ratio_median",
        ]
    ].to_string(index=False)
)

print("\nPaired occlusion model comparisons vs EfficientNetB0:")
print(
    model_comparisons[
        model_comparisons["region_name"].astype(str).eq(PRIMARY_OCCLUSION_REGION)
    ][
        [
            "level",
            "comparison_model",
            "region_name",
            "paired_units",
            "delta_comparison_minus_reference_mean",
            "delta_ci_low",
            "delta_ci_high",
            "wilcoxon_two_sided_p",
            "wilcoxon_two_sided_p_fdr",
            "signed_rank_biserial_effect",
        ]
    ].to_string(index=False)
)

print("\nOcclusion label-strata statistics:")
print(
    label_strata[
        label_strata["region_name"].astype(str).eq(PRIMARY_OCCLUSION_REGION)
    ][
        [
            "level",
            "model_name",
            "region_name",
            "n_positive_group",
            "n_negative_group",
            "positive_group_mean",
            "negative_group_mean",
            "difference_positive_minus_negative",
            "mannwhitney_two_sided_p",
            "mannwhitney_two_sided_p_fdr",
        ]
    ].to_string(index=False)
)

print("\nOcclusion statistical summary JSON:")
print(json.dumps(summary_json, indent=2))

print(f"\nConsolidated occlusion metrics: {OCCLUSION_CONSOLIDATED_CSV}")
print(f"Patient-level occlusion metrics: {OCCLUSION_PATIENT_LEVEL_CSV}")
print(f"Image-level occlusion stats: {OCCLUSION_IMAGE_LEVEL_STATS_CSV}")
print(f"Patient-level occlusion stats: {OCCLUSION_PATIENT_LEVEL_STATS_CSV}")
print(f"Model comparisons: {OCCLUSION_MODEL_COMPARISON_CSV}")
print(f"Label strata: {OCCLUSION_LABEL_STRATA_CSV}")
print(f"Correctness strata: {OCCLUSION_CORRECTNESS_STRATA_CSV}")
print(f"Summary JSON: {OCCLUSION_STATISTICAL_SUMMARY_JSON}")

expected_models = len(AQPR_PANEL_MODEL_NAMES)

if occlusion_metrics["model_name"].nunique() != expected_models:
    raise RuntimeError(
        f"Expected {expected_models} AQPR models in occlusion metrics, "
        f"found {occlusion_metrics['model_name'].nunique()}."
    )

if occlusion_metrics["sample_id"].nunique() <= 0:
    raise RuntimeError("No image-level occlusion metrics found.")

if occlusion_metrics["patient_id"].nunique() <= 0:
    raise RuntimeError("No patient-level occlusion metrics found.")

primary_region_rows = occlusion_metrics[
    occlusion_metrics["region_name"].astype(str) == PRIMARY_OCCLUSION_REGION
].copy()

if len(primary_region_rows) == 0:
    raise RuntimeError(f"No {PRIMARY_OCCLUSION_REGION} occlusion rows found.")

if not np.isfinite(primary_region_rows[PRIMARY_OCCLUSION_ENDPOINT]).all():
    raise RuntimeError(
        f"Non-finite {PRIMARY_OCCLUSION_ENDPOINT} values found for {PRIMARY_OCCLUSION_REGION}."
    )

if summary_json["n_inconsistent_patient_labels"] > 0:
    print("Warning: patient-level labels were aggregated using max(y_true) because mixed labels were detected.")

print("Status: PASS")

In [ ]:
# Cell 9: Integrated AQPR synthesis of heatmap enrichment and anatomical occlusion

import matplotlib.pyplot as plt

AQPR_SYNTHESIS_DIR = RESULTS_08_DIR / "aqpr_integrated_synthesis"
AQPR_SYNTHESIS_DIR.mkdir(parents=True, exist_ok=True)

HEATMAP_PATIENT_LEVEL_STATS_CSV = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_patient_level_statistics.csv"
HEATMAP_IMAGE_LEVEL_STATS_CSV = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_image_level_statistics.csv"
HEATMAP_CONSOLIDATED_CSV = RESULTS_08_DIR / "notebook08_aqpr_heatmap_xai_metrics_consolidated.csv"

OCCLUSION_PATIENT_LEVEL_STATS_CSV = RESULTS_08_DIR / "notebook08_aqpr_anatomical_occlusion_patient_level_statistics.csv"
OCCLUSION_IMAGE_LEVEL_STATS_CSV = RESULTS_08_DIR / "notebook08_aqpr_anatomical_occlusion_image_level_statistics.csv"
OCCLUSION_CONSOLIDATED_CSV = RESULTS_08_DIR / "notebook08_aqpr_anatomical_occlusion_metrics_consolidated.csv"

AQPR_PRIMARY_ENDPOINT_TABLE_CSV = AQPR_SYNTHESIS_DIR / "notebook08_aqpr_primary_endpoint_table.csv"
AQPR_MODEL_EVIDENCE_SUMMARY_CSV = AQPR_SYNTHESIS_DIR / "notebook08_aqpr_model_evidence_summary.csv"
AQPR_METHOD_EVIDENCE_SUMMARY_CSV = AQPR_SYNTHESIS_DIR / "notebook08_aqpr_method_evidence_summary.csv"
AQPR_MAIN_RESULT_TABLE_CSV = AQPR_SYNTHESIS_DIR / "notebook08_aqpr_main_result_table_for_manuscript.csv"
AQPR_SYNTHESIS_JSON = AQPR_SYNTHESIS_DIR / "notebook08_aqpr_integrated_synthesis_summary.json"

AQPR_HEATMAP_PATIENT_FIG_PNG = AQPR_SYNTHESIS_DIR / "notebook08_aqpr_patient_level_heatmap_enrichment_od_plus_oc.png"
AQPR_HEATMAP_PATIENT_FIG_PDF = AQPR_SYNTHESIS_DIR / "notebook08_aqpr_patient_level_heatmap_enrichment_od_plus_oc.pdf"

AQPR_OCCLUSION_PATIENT_FIG_PNG = AQPR_SYNTHESIS_DIR / "notebook08_aqpr_patient_level_occlusion_od_plus_oc.png"
AQPR_OCCLUSION_PATIENT_FIG_PDF = AQPR_SYNTHESIS_DIR / "notebook08_aqpr_patient_level_occlusion_od_plus_oc.pdf"

AQPR_EVIDENCE_MATRIX_FIG_PNG = AQPR_SYNTHESIS_DIR / "notebook08_aqpr_integrated_evidence_matrix.png"
AQPR_EVIDENCE_MATRIX_FIG_PDF = AQPR_SYNTHESIS_DIR / "notebook08_aqpr_integrated_evidence_matrix.pdf"

PRIMARY_REGION = "OD_plus_OC"
FDR_ALPHA = 0.05


required_inputs = [
    HEATMAP_PATIENT_LEVEL_STATS_CSV,
    HEATMAP_IMAGE_LEVEL_STATS_CSV,
    HEATMAP_CONSOLIDATED_CSV,
    OCCLUSION_PATIENT_LEVEL_STATS_CSV,
    OCCLUSION_IMAGE_LEVEL_STATS_CSV,
    OCCLUSION_CONSOLIDATED_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing required Cell 9 inputs: {missing_inputs}")


heatmap_patient_stats = pd.read_csv(HEATMAP_PATIENT_LEVEL_STATS_CSV)
heatmap_image_stats = pd.read_csv(HEATMAP_IMAGE_LEVEL_STATS_CSV)
heatmap_metrics = pd.read_csv(HEATMAP_CONSOLIDATED_CSV)

occlusion_patient_stats = pd.read_csv(OCCLUSION_PATIENT_LEVEL_STATS_CSV)
occlusion_image_stats = pd.read_csv(OCCLUSION_IMAGE_LEVEL_STATS_CSV)
occlusion_metrics = pd.read_csv(OCCLUSION_CONSOLIDATED_CSV)


def require_cell9_columns(df, required_cols, table_name):
    missing = [
        col for col in required_cols
        if col not in df.columns
    ]

    if missing:
        raise ValueError(f"Missing columns in {table_name}: {missing}")


require_cell9_columns(
    heatmap_patient_stats,
    [
        "level",
        "model_name",
        "method_name",
        "aqpr_panel_role",
        "region_name",
        "n_units",
        "primary_endpoint_mean",
        "primary_endpoint_ci_low",
        "primary_endpoint_ci_high",
        "wilcoxon_greater_than_zero_p_fdr",
        "observed_energy_fraction_mean",
        "enrichment_ratio_median",
    ],
    "heatmap patient-level statistics",
)

require_cell9_columns(
    occlusion_patient_stats,
    [
        "level",
        "model_name",
        "aqpr_panel_role",
        "region_name",
        "n_units",
        "primary_endpoint_mean",
        "primary_endpoint_ci_low",
        "primary_endpoint_ci_high",
        "wilcoxon_p_greater_fdr",
        "wilcoxon_p_less_fdr",
        "signed_rank_biserial_effect",
        "observed_probability_drop_mean",
        "random_probability_drop_mean_mean",
        "probability_drop_ratio_median",
    ],
    "occlusion patient-level statistics",
)


def classify_heatmap_evidence(row):
    mean_value = float(row["primary_endpoint_mean"])
    p_fdr = float(row["wilcoxon_greater_than_zero_p_fdr"])

    if np.isfinite(mean_value) and np.isfinite(p_fdr):
        if mean_value > 0 and p_fdr < FDR_ALPHA:
            return "positive_anatomical_enrichment"

        if mean_value <= 0 and p_fdr < FDR_ALPHA:
            return "non_positive_or_inverse"

    return "not_supported"


def classify_occlusion_evidence(row):
    mean_value = float(row["primary_endpoint_mean"])
    p_greater_fdr = float(row["wilcoxon_p_greater_fdr"])
    p_less_fdr = float(row["wilcoxon_p_less_fdr"])

    if np.isfinite(mean_value):
        if mean_value > 0 and np.isfinite(p_greater_fdr) and p_greater_fdr < FDR_ALPHA:
            return "positive_causal_perturbation"

        if mean_value < 0 and np.isfinite(p_less_fdr) and p_less_fdr < FDR_ALPHA:
            return "inverse_perturbation_response"

    return "not_supported"


def build_primary_endpoint_table():
    heatmap_patient_primary = heatmap_patient_stats[
        (heatmap_patient_stats["region_name"].astype(str) == PRIMARY_REGION)
    ].copy()

    heatmap_patient_primary["analysis_family"] = "heatmap_enrichment"
    heatmap_patient_primary["endpoint_direction"] = "higher_than_random_is_anatomically_plausible"
    heatmap_patient_primary["evidence_class"] = heatmap_patient_primary.apply(
        classify_heatmap_evidence,
        axis=1,
    )

    heatmap_out = heatmap_patient_primary[
        [
            "analysis_family",
            "model_name",
            "method_name",
            "aqpr_panel_role",
            "region_name",
            "n_units",
            "primary_endpoint_mean",
            "primary_endpoint_ci_low",
            "primary_endpoint_ci_high",
            "wilcoxon_greater_than_zero_p_fdr",
            "observed_energy_fraction_mean",
            "enrichment_ratio_median",
            "endpoint_direction",
            "evidence_class",
        ]
    ].copy()

    heatmap_out = heatmap_out.rename(
        columns={
            "wilcoxon_greater_than_zero_p_fdr": "primary_fdr_p_value",
            "observed_energy_fraction_mean": "observed_signal_mean",
            "enrichment_ratio_median": "ratio_or_effect_median",
        }
    )

    occlusion_patient_primary = occlusion_patient_stats[
        (occlusion_patient_stats["region_name"].astype(str) == PRIMARY_REGION)
    ].copy()

    occlusion_patient_primary["analysis_family"] = "anatomical_occlusion"
    occlusion_patient_primary["method_name"] = "AnatomicalOcclusion"
    occlusion_patient_primary["endpoint_direction"] = "positive_drop_vs_random_supports_causal_dependence"
    occlusion_patient_primary["evidence_class"] = occlusion_patient_primary.apply(
        classify_occlusion_evidence,
        axis=1,
    )

    occlusion_out = occlusion_patient_primary[
        [
            "analysis_family",
            "model_name",
            "method_name",
            "aqpr_panel_role",
            "region_name",
            "n_units",
            "primary_endpoint_mean",
            "primary_endpoint_ci_low",
            "primary_endpoint_ci_high",
            "wilcoxon_p_greater_fdr",
            "observed_probability_drop_mean",
            "probability_drop_ratio_median",
            "endpoint_direction",
            "evidence_class",
        ]
    ].copy()

    occlusion_out = occlusion_out.rename(
        columns={
            "wilcoxon_p_greater_fdr": "primary_fdr_p_value",
            "observed_probability_drop_mean": "observed_signal_mean",
            "probability_drop_ratio_median": "ratio_or_effect_median",
        }
    )

    primary_endpoint_table = pd.concat(
        [heatmap_out, occlusion_out],
        axis=0,
        ignore_index=True,
    )

    primary_endpoint_table["is_primary_model"] = (
        primary_endpoint_table["model_name"].astype(str) == PRIMARY_MODEL_NAME
    )

    primary_endpoint_table["is_primary_region"] = (
        primary_endpoint_table["region_name"].astype(str) == PRIMARY_REGION
    )

    return primary_endpoint_table


primary_endpoint_table = build_primary_endpoint_table()
primary_endpoint_table.to_csv(AQPR_PRIMARY_ENDPOINT_TABLE_CSV, index=False)


def get_single_value(df, model_name, analysis_family, method_name, column_name):
    sub = df[
        (df["model_name"].astype(str) == str(model_name))
        & (df["analysis_family"].astype(str) == str(analysis_family))
        & (df["method_name"].astype(str) == str(method_name))
        & (df["region_name"].astype(str) == PRIMARY_REGION)
    ].copy()

    if len(sub) == 0:
        return np.nan

    if len(sub) > 1:
        raise RuntimeError(
            f"Expected one row for {model_name}, {analysis_family}, {method_name}, found {len(sub)}"
        )

    return sub.iloc[0][column_name]


def get_evidence_class(df, model_name, analysis_family, method_name):
    return get_single_value(
        df=df,
        model_name=model_name,
        analysis_family=analysis_family,
        method_name=method_name,
        column_name="evidence_class",
    )


def integrated_pattern_from_classes(gradcam_class, ig_class, occlusion_class):
    heatmap_classes = [gradcam_class, ig_class]

    n_positive_heatmap = sum(
        str(value) == "positive_anatomical_enrichment"
        for value in heatmap_classes
    )

    if n_positive_heatmap >= 1 and occlusion_class == "positive_causal_perturbation":
        return "convergent_anatomical_and_perturbation_support"

    if n_positive_heatmap >= 1 and occlusion_class == "inverse_perturbation_response":
        return "saliency_enriched_but_occlusion_inverse"

    if n_positive_heatmap >= 1 and occlusion_class == "not_supported":
        return "saliency_enriched_without_causal_perturbation_support"

    if n_positive_heatmap == 0 and occlusion_class == "positive_causal_perturbation":
        return "perturbation_support_without_heatmap_support"

    return "weak_or_inconsistent_anatomical_reliability"


model_rows = []

for model_name in AQPR_PANEL_MODEL_NAMES:
    gradcam_class = get_evidence_class(
        primary_endpoint_table,
        model_name,
        "heatmap_enrichment",
        "GradCAM",
    )

    ig_class = get_evidence_class(
        primary_endpoint_table,
        model_name,
        "heatmap_enrichment",
        "IntegratedGradients",
    )

    occlusion_class = get_evidence_class(
        primary_endpoint_table,
        model_name,
        "anatomical_occlusion",
        "AnatomicalOcclusion",
    )

    integrated_pattern = integrated_pattern_from_classes(
        gradcam_class=gradcam_class,
        ig_class=ig_class,
        occlusion_class=occlusion_class,
    )

    model_rows.append(
        {
            "model_name": model_name,
            "aqpr_panel_role": AQPR_PANEL_ROLES[model_name],
            "is_primary_model": model_name == PRIMARY_MODEL_NAME,
            "gradcam_patient_level_evidence": gradcam_class,
            "integrated_gradients_patient_level_evidence": ig_class,
            "occlusion_patient_level_evidence": occlusion_class,
            "integrated_aqpr_pattern": integrated_pattern,
            "gradcam_patient_mean": get_single_value(
                primary_endpoint_table,
                model_name,
                "heatmap_enrichment",
                "GradCAM",
                "primary_endpoint_mean",
            ),
            "integrated_gradients_patient_mean": get_single_value(
                primary_endpoint_table,
                model_name,
                "heatmap_enrichment",
                "IntegratedGradients",
                "primary_endpoint_mean",
            ),
            "occlusion_patient_mean": get_single_value(
                primary_endpoint_table,
                model_name,
                "anatomical_occlusion",
                "AnatomicalOcclusion",
                "primary_endpoint_mean",
            ),
        }
    )

model_evidence_summary = pd.DataFrame(model_rows)
model_evidence_summary.to_csv(AQPR_MODEL_EVIDENCE_SUMMARY_CSV, index=False)


method_evidence_summary = (
    primary_endpoint_table
    .groupby(
        [
            "analysis_family",
            "method_name",
            "evidence_class",
        ]
    )
    .agg(
        n_models=("model_name", "nunique"),
        mean_primary_endpoint=("primary_endpoint_mean", "mean"),
        median_primary_endpoint=("primary_endpoint_mean", "median"),
    )
    .reset_index()
)

method_evidence_summary.to_csv(AQPR_METHOD_EVIDENCE_SUMMARY_CSV, index=False)


main_result_table = primary_endpoint_table[
    primary_endpoint_table["region_name"].astype(str).eq(PRIMARY_REGION)
].copy()

main_result_table = main_result_table[
    [
        "analysis_family",
        "model_name",
        "method_name",
        "aqpr_panel_role",
        "n_units",
        "primary_endpoint_mean",
        "primary_endpoint_ci_low",
        "primary_endpoint_ci_high",
        "primary_fdr_p_value",
        "observed_signal_mean",
        "ratio_or_effect_median",
        "evidence_class",
        "endpoint_direction",
    ]
].copy()

main_result_table = main_result_table.sort_values(
    [
        "analysis_family",
        "model_name",
        "method_name",
    ]
).reset_index(drop=True)

main_result_table.to_csv(AQPR_MAIN_RESULT_TABLE_CSV, index=False)


# ---------------------------------------------------------------------
# Figures
# ---------------------------------------------------------------------

def save_patient_level_heatmap_figure():
    plot_df = primary_endpoint_table[
        (primary_endpoint_table["analysis_family"].astype(str) == "heatmap_enrichment")
        & (primary_endpoint_table["region_name"].astype(str) == PRIMARY_REGION)
    ].copy()

    plot_df["display_label"] = (
        plot_df["model_name"].astype(str)
        + "\n"
        + plot_df["method_name"].astype(str).replace(
            {
                "IntegratedGradients": "IG",
            }
        )
    )

    plot_df = plot_df.sort_values(
        ["model_name", "method_name"]
    ).reset_index(drop=True)

    x = np.arange(len(plot_df))
    y = plot_df["primary_endpoint_mean"].astype(float).values

    yerr_low = y - plot_df["primary_endpoint_ci_low"].astype(float).values
    yerr_high = plot_df["primary_endpoint_ci_high"].astype(float).values - y

    fig, ax = plt.subplots(figsize=(12.0, 4.8))

    ax.errorbar(
        x,
        y,
        yerr=[yerr_low, yerr_high],
        fmt="o",
        capsize=4,
        linewidth=1.5,
    )

    ax.axhline(0.0, linestyle="--", linewidth=1.0)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["display_label"], rotation=45, ha="right")
    ax.set_ylabel("Patient-level OD+OC enrichment\nobserved minus matched random")
    ax.set_title("AQPR heatmap anatomical enrichment on HYGD")
    ax.grid(True, axis="y", alpha=0.25)

    fig.tight_layout()
    fig.savefig(AQPR_HEATMAP_PATIENT_FIG_PNG, dpi=300, bbox_inches="tight")
    fig.savefig(AQPR_HEATMAP_PATIENT_FIG_PDF, bbox_inches="tight")
    plt.close(fig)


def save_patient_level_occlusion_figure():
    plot_df = primary_endpoint_table[
        (primary_endpoint_table["analysis_family"].astype(str) == "anatomical_occlusion")
        & (primary_endpoint_table["region_name"].astype(str) == PRIMARY_REGION)
    ].copy()

    plot_df = plot_df.sort_values("model_name").reset_index(drop=True)

    x = np.arange(len(plot_df))
    y = plot_df["primary_endpoint_mean"].astype(float).values

    yerr_low = y - plot_df["primary_endpoint_ci_low"].astype(float).values
    yerr_high = plot_df["primary_endpoint_ci_high"].astype(float).values - y

    fig, ax = plt.subplots(figsize=(8.5, 4.8))

    ax.errorbar(
        x,
        y,
        yerr=[yerr_low, yerr_high],
        fmt="o",
        capsize=4,
        linewidth=1.5,
    )

    ax.axhline(0.0, linestyle="--", linewidth=1.0)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["model_name"], rotation=35, ha="right")
    ax.set_ylabel("Patient-level OD+OC occlusion effect\nobserved drop minus matched random drop")
    ax.set_title("AQPR anatomical occlusion on HYGD")
    ax.grid(True, axis="y", alpha=0.25)

    fig.tight_layout()
    fig.savefig(AQPR_OCCLUSION_PATIENT_FIG_PNG, dpi=300, bbox_inches="tight")
    fig.savefig(AQPR_OCCLUSION_PATIENT_FIG_PDF, bbox_inches="tight")
    plt.close(fig)


def save_evidence_matrix_figure():
    display_df = model_evidence_summary[
        [
            "model_name",
            "gradcam_patient_level_evidence",
            "integrated_gradients_patient_level_evidence",
            "occlusion_patient_level_evidence",
            "integrated_aqpr_pattern",
        ]
    ].copy()

    display_df = display_df.rename(
        columns={
            "model_name": "Model",
            "gradcam_patient_level_evidence": "Grad-CAM",
            "integrated_gradients_patient_level_evidence": "IG",
            "occlusion_patient_level_evidence": "Occlusion",
            "integrated_aqpr_pattern": "AQPR pattern",
        }
    )

    fig_height = 1.4 + 0.55 * len(display_df)
    fig, ax = plt.subplots(figsize=(15.0, fig_height))
    ax.axis("off")

    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        loc="center",
        cellLoc="left",
        colLoc="left",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(8.5)
    table.scale(1.0, 1.35)

    ax.set_title(
        "Integrated AQPR evidence matrix: heatmap enrichment versus anatomical occlusion",
        pad=14,
        fontsize=11,
        fontweight="bold",
    )

    fig.tight_layout()
    fig.savefig(AQPR_EVIDENCE_MATRIX_FIG_PNG, dpi=300, bbox_inches="tight")
    fig.savefig(AQPR_EVIDENCE_MATRIX_FIG_PDF, bbox_inches="tight")
    plt.close(fig)


save_patient_level_heatmap_figure()
save_patient_level_occlusion_figure()
save_evidence_matrix_figure()


# ---------------------------------------------------------------------
# Integrated interpretation
# ---------------------------------------------------------------------

primary_model_row = model_evidence_summary[
    model_evidence_summary["model_name"].astype(str) == PRIMARY_MODEL_NAME
].copy()

if len(primary_model_row) != 1:
    raise RuntimeError("Primary model was not found uniquely in AQPR model evidence summary.")

primary_model_row = primary_model_row.iloc[0].to_dict()

n_models_with_positive_ig = int(
    (
        model_evidence_summary["integrated_gradients_patient_level_evidence"].astype(str)
        == "positive_anatomical_enrichment"
    ).sum()
)

n_models_with_positive_gradcam = int(
    (
        model_evidence_summary["gradcam_patient_level_evidence"].astype(str)
        == "positive_anatomical_enrichment"
    ).sum()
)

n_models_with_positive_occlusion = int(
    (
        model_evidence_summary["occlusion_patient_level_evidence"].astype(str)
        == "positive_causal_perturbation"
    ).sum()
)

n_models_with_inverse_occlusion = int(
    (
        model_evidence_summary["occlusion_patient_level_evidence"].astype(str)
        == "inverse_perturbation_response"
    ).sum()
)

n_models_with_saliency_occlusion_conflict = int(
    (
        model_evidence_summary["integrated_aqpr_pattern"].astype(str)
        == "saliency_enriched_but_occlusion_inverse"
    ).sum()
)

synthesis_summary = {
    "notebook": "08_anatomical_explainability_validation",
    "cell": "Cell 9",
    "analysis": "Integrated AQPR synthesis of anatomical heatmap enrichment and anatomical occlusion",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_anatomical_dataset": "HYGD",
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "primary_region": PRIMARY_REGION,
    "fdr_alpha": float(FDR_ALPHA),
    "n_hygd_images_heatmap": int(heatmap_metrics["sample_id"].nunique()),
    "n_hygd_patients_heatmap": int(heatmap_metrics["patient_id"].nunique()),
    "n_hygd_images_occlusion": int(occlusion_metrics["sample_id"].nunique()),
    "n_hygd_patients_occlusion": int(occlusion_metrics["patient_id"].nunique()),
    "n_models_with_positive_gradcam_patient_level_enrichment": n_models_with_positive_gradcam,
    "n_models_with_positive_integrated_gradients_patient_level_enrichment": n_models_with_positive_ig,
    "n_models_with_positive_occlusion_patient_level_effect": n_models_with_positive_occlusion,
    "n_models_with_inverse_occlusion_patient_level_effect": n_models_with_inverse_occlusion,
    "n_models_with_saliency_occlusion_conflict": n_models_with_saliency_occlusion_conflict,
    "primary_model_integrated_pattern": str(primary_model_row["integrated_aqpr_pattern"]),
    "primary_model_gradcam_evidence": str(primary_model_row["gradcam_patient_level_evidence"]),
    "primary_model_integrated_gradients_evidence": str(
        primary_model_row["integrated_gradients_patient_level_evidence"]
    ),
    "primary_model_occlusion_evidence": str(primary_model_row["occlusion_patient_level_evidence"]),
    "main_interpretation": (
        "The frozen AIROGS-light classifier panel shows patient-level OD/OC heatmap enrichment "
        "on HYGD, especially with Integrated Gradients. However, anatomical occlusion does not "
        "provide convergent positive causal evidence for the validation-selected primary model; "
        "instead, the primary model shows an inverse perturbation response under blur occlusion. "
        "Therefore, the AQPR conclusion should be framed as partial anatomical plausibility with "
        "a perturbation-based reliability warning, not as definitive causal anatomical grounding."
    ),
    "claim_boundary": (
        "These results support AQPR as a reliability audit framework. They do not justify claiming "
        "that all high-performing classifiers causally rely on OD/OC anatomy."
    ),
    "patient_level_external_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
    "primary_endpoint_table_csv": str(AQPR_PRIMARY_ENDPOINT_TABLE_CSV),
    "model_evidence_summary_csv": str(AQPR_MODEL_EVIDENCE_SUMMARY_CSV),
    "method_evidence_summary_csv": str(AQPR_METHOD_EVIDENCE_SUMMARY_CSV),
    "main_result_table_csv": str(AQPR_MAIN_RESULT_TABLE_CSV),
    "heatmap_patient_figure_png": str(AQPR_HEATMAP_PATIENT_FIG_PNG),
    "occlusion_patient_figure_png": str(AQPR_OCCLUSION_PATIENT_FIG_PNG),
    "evidence_matrix_figure_png": str(AQPR_EVIDENCE_MATRIX_FIG_PNG),
}

with open(AQPR_SYNTHESIS_JSON, "w", encoding="utf-8") as f:
    json.dump(synthesis_summary, f, indent=2)


print("AQPR primary endpoint table:")
print(
    primary_endpoint_table[
        [
            "analysis_family",
            "model_name",
            "method_name",
            "region_name",
            "n_units",
            "primary_endpoint_mean",
            "primary_endpoint_ci_low",
            "primary_endpoint_ci_high",
            "primary_fdr_p_value",
            "evidence_class",
        ]
    ].to_string(index=False)
)

print("\nAQPR model evidence summary:")
print(model_evidence_summary.to_string(index=False))

print("\nAQPR method evidence summary:")
print(method_evidence_summary.to_string(index=False))

print("\nAQPR integrated synthesis summary:")
print(json.dumps(synthesis_summary, indent=2))

print(f"\nPrimary endpoint table: {AQPR_PRIMARY_ENDPOINT_TABLE_CSV}")
print(f"Model evidence summary: {AQPR_MODEL_EVIDENCE_SUMMARY_CSV}")
print(f"Method evidence summary: {AQPR_METHOD_EVIDENCE_SUMMARY_CSV}")
print(f"Main manuscript table: {AQPR_MAIN_RESULT_TABLE_CSV}")
print(f"Synthesis JSON: {AQPR_SYNTHESIS_JSON}")

print(f"\nHeatmap patient-level figure: {AQPR_HEATMAP_PATIENT_FIG_PNG}")
print(f"Occlusion patient-level figure: {AQPR_OCCLUSION_PATIENT_FIG_PNG}")
print(f"Evidence matrix figure: {AQPR_EVIDENCE_MATRIX_FIG_PNG}")


for required_output in [
    AQPR_PRIMARY_ENDPOINT_TABLE_CSV,
    AQPR_MODEL_EVIDENCE_SUMMARY_CSV,
    AQPR_METHOD_EVIDENCE_SUMMARY_CSV,
    AQPR_MAIN_RESULT_TABLE_CSV,
    AQPR_SYNTHESIS_JSON,
    AQPR_HEATMAP_PATIENT_FIG_PNG,
    AQPR_HEATMAP_PATIENT_FIG_PDF,
    AQPR_OCCLUSION_PATIENT_FIG_PNG,
    AQPR_OCCLUSION_PATIENT_FIG_PDF,
    AQPR_EVIDENCE_MATRIX_FIG_PNG,
    AQPR_EVIDENCE_MATRIX_FIG_PDF,
]:
    if not Path(required_output).exists():
        raise RuntimeError(f"Missing Cell 9 output: {required_output}")

if len(primary_endpoint_table) == 0:
    raise RuntimeError("Primary endpoint table is empty.")

if len(model_evidence_summary) != len(AQPR_PANEL_MODEL_NAMES):
    raise RuntimeError("Model evidence summary does not contain the full AQPR panel.")

if synthesis_summary["n_models_with_positive_integrated_gradients_patient_level_enrichment"] <= 0:
    raise RuntimeError("No positive Integrated Gradients anatomical enrichment was detected.")

if synthesis_summary["primary_model_integrated_pattern"] == "convergent_anatomical_and_perturbation_support":
    print("Primary model shows convergent heatmap and occlusion support.")
else:
    print(
        "Primary model does not show convergent heatmap and occlusion support; "
        "this must be reported as an AQPR reliability warning."
    )

print("Status: PASS")

In [ ]:
# Cell 10: Finalize Notebook 08 AQPR anatomical explainability audit and downstream handoff

AQPR_SYNTHESIS_DIR = RESULTS_08_DIR / "aqpr_integrated_synthesis"

AQPR_PRIMARY_ENDPOINT_TABLE_CSV = (
    AQPR_SYNTHESIS_DIR
    / "notebook08_aqpr_primary_endpoint_table.csv"
)

AQPR_MODEL_EVIDENCE_SUMMARY_CSV = (
    AQPR_SYNTHESIS_DIR
    / "notebook08_aqpr_model_evidence_summary.csv"
)

AQPR_METHOD_EVIDENCE_SUMMARY_CSV = (
    AQPR_SYNTHESIS_DIR
    / "notebook08_aqpr_method_evidence_summary.csv"
)

AQPR_MAIN_RESULT_TABLE_CSV = (
    AQPR_SYNTHESIS_DIR
    / "notebook08_aqpr_main_result_table_for_manuscript.csv"
)

AQPR_SYNTHESIS_JSON = (
    AQPR_SYNTHESIS_DIR
    / "notebook08_aqpr_integrated_synthesis_summary.json"
)

AQPR_HEATMAP_PATIENT_FIG_PNG = (
    AQPR_SYNTHESIS_DIR
    / "notebook08_aqpr_patient_level_heatmap_enrichment_od_plus_oc.png"
)

AQPR_OCCLUSION_PATIENT_FIG_PNG = (
    AQPR_SYNTHESIS_DIR
    / "notebook08_aqpr_patient_level_occlusion_od_plus_oc.png"
)

AQPR_EVIDENCE_MATRIX_FIG_PNG = (
    AQPR_SYNTHESIS_DIR
    / "notebook08_aqpr_integrated_evidence_matrix.png"
)

HEATMAP_CONSOLIDATED_CSV = (
    RESULTS_08_DIR
    / "notebook08_aqpr_heatmap_xai_metrics_consolidated.csv"
)

HEATMAP_PATIENT_LEVEL_CSV = (
    RESULTS_08_DIR
    / "notebook08_aqpr_heatmap_xai_patient_level_metrics.csv"
)

HEATMAP_PATIENT_LEVEL_STATS_CSV = (
    RESULTS_08_DIR
    / "notebook08_aqpr_heatmap_xai_patient_level_statistics.csv"
)

OCCLUSION_CONSOLIDATED_CSV = (
    RESULTS_08_DIR
    / "notebook08_aqpr_anatomical_occlusion_metrics_consolidated.csv"
)

OCCLUSION_PATIENT_LEVEL_CSV = (
    RESULTS_08_DIR
    / "notebook08_aqpr_anatomical_occlusion_patient_level_metrics.csv"
)

OCCLUSION_PATIENT_LEVEL_STATS_CSV = (
    RESULTS_08_DIR
    / "notebook08_aqpr_anatomical_occlusion_patient_level_statistics.csv"
)

NOTEBOOK08_FINAL_SYNTHESIS_CSV = (
    RESULTS_08_DIR
    / "notebook08_final_comparative_xai_synthesis.csv"
)

NOTEBOOK08_FINAL_INTERPRETATION_CSV = (
    RESULTS_08_DIR
    / "notebook08_final_interpretation_matrix.csv"
)

NOTEBOOK08_FINAL_ARTIFACT_REGISTRY_CSV = (
    RESULTS_08_DIR
    / "notebook08_final_aqpr_artifact_registry.csv"
)

NOTEBOOK08_FINAL_CLAIM_BOUNDARIES_CSV = (
    RESULTS_08_DIR
    / "notebook08_final_claim_boundaries.csv"
)

NOTEBOOK08_DOWNSTREAM_HANDOFF_JSON = (
    RESULTS_08_DIR
    / "notebook08_downstream_handoff.json"
)

NOTEBOOK08_FINAL_STATUS_JSON = (
    RESULTS_08_DIR
    / "notebook08_final_status.json"
)

NOTEBOOK08_FINAL_STATUS_TXT = (
    RESULTS_08_DIR
    / "notebook08_FINAL_STATUS.txt"
)


# ---------------------------------------------------------------------
# Required final artifacts
# ---------------------------------------------------------------------

required_final_artifacts = {
    "aqpr_primary_endpoint_table": AQPR_PRIMARY_ENDPOINT_TABLE_CSV,
    "aqpr_model_evidence_summary": AQPR_MODEL_EVIDENCE_SUMMARY_CSV,
    "aqpr_method_evidence_summary": AQPR_METHOD_EVIDENCE_SUMMARY_CSV,
    "aqpr_main_result_table": AQPR_MAIN_RESULT_TABLE_CSV,
    "aqpr_integrated_synthesis_json": AQPR_SYNTHESIS_JSON,
    "aqpr_heatmap_patient_figure": AQPR_HEATMAP_PATIENT_FIG_PNG,
    "aqpr_occlusion_patient_figure": AQPR_OCCLUSION_PATIENT_FIG_PNG,
    "aqpr_evidence_matrix_figure": AQPR_EVIDENCE_MATRIX_FIG_PNG,
    "heatmap_consolidated_metrics": HEATMAP_CONSOLIDATED_CSV,
    "heatmap_patient_level_metrics": HEATMAP_PATIENT_LEVEL_CSV,
    "heatmap_patient_level_stats": HEATMAP_PATIENT_LEVEL_STATS_CSV,
    "occlusion_consolidated_metrics": OCCLUSION_CONSOLIDATED_CSV,
    "occlusion_patient_level_metrics": OCCLUSION_PATIENT_LEVEL_CSV,
    "occlusion_patient_level_stats": OCCLUSION_PATIENT_LEVEL_STATS_CSV,
}

artifact_registry = pd.DataFrame(
    [
        {
            "artifact_key": key,
            "path": str(path),
            "exists": Path(path).exists(),
            "artifact_role": (
                "main_result"
                if key in {
                    "aqpr_primary_endpoint_table",
                    "aqpr_model_evidence_summary",
                    "aqpr_main_result_table",
                    "aqpr_integrated_synthesis_json",
                }
                else "supporting_result"
            ),
        }
        for key, path in required_final_artifacts.items()
    ]
)

artifact_registry.to_csv(NOTEBOOK08_FINAL_ARTIFACT_REGISTRY_CSV, index=False)

missing_artifacts = artifact_registry.loc[
    ~artifact_registry["exists"],
    ["artifact_key", "path"],
]

if len(missing_artifacts) > 0:
    print(missing_artifacts.to_string(index=False))
    raise FileNotFoundError("Notebook 08 finalization failed because required artifacts are missing.")


# ---------------------------------------------------------------------
# Load final synthesis artifacts
# ---------------------------------------------------------------------

primary_endpoint_table = pd.read_csv(AQPR_PRIMARY_ENDPOINT_TABLE_CSV)
model_evidence_summary = pd.read_csv(AQPR_MODEL_EVIDENCE_SUMMARY_CSV)
method_evidence_summary = pd.read_csv(AQPR_METHOD_EVIDENCE_SUMMARY_CSV)
main_result_table = pd.read_csv(AQPR_MAIN_RESULT_TABLE_CSV)

with open(AQPR_SYNTHESIS_JSON, "r", encoding="utf-8") as f:
    aqpr_synthesis_summary = json.load(f)

required_primary_cols = [
    "analysis_family",
    "model_name",
    "method_name",
    "region_name",
    "n_units",
    "primary_endpoint_mean",
    "primary_endpoint_ci_low",
    "primary_endpoint_ci_high",
    "primary_fdr_p_value",
    "evidence_class",
]

missing_primary_cols = [
    col for col in required_primary_cols
    if col not in primary_endpoint_table.columns
]

if missing_primary_cols:
    raise ValueError(f"Missing columns in AQPR primary endpoint table: {missing_primary_cols}")

required_model_cols = [
    "model_name",
    "aqpr_panel_role",
    "is_primary_model",
    "gradcam_patient_level_evidence",
    "integrated_gradients_patient_level_evidence",
    "occlusion_patient_level_evidence",
    "integrated_aqpr_pattern",
]

missing_model_cols = [
    col for col in required_model_cols
    if col not in model_evidence_summary.columns
]

if missing_model_cols:
    raise ValueError(f"Missing columns in AQPR model evidence summary: {missing_model_cols}")


# ---------------------------------------------------------------------
# Build final comparative synthesis table
# ---------------------------------------------------------------------

final_synthesis = primary_endpoint_table.copy()

final_synthesis["notebook"] = "08_anatomical_explainability_validation"
final_synthesis["analysis_scope"] = "external_HYGD_patient_level_anatomical_reliability_audit"
final_synthesis["primary_training_dataset"] = "EyePACS-AIROGS-light-V2"
final_synthesis["external_evaluation_dataset"] = "HYGD"
final_synthesis["patient_level_external_analysis_allowed"] = True
final_synthesis["internal_airogslight_patient_level_claim_allowed"] = False

final_synthesis["claim_strength"] = np.select(
    [
        final_synthesis["evidence_class"].astype(str).eq("positive_anatomical_enrichment"),
        final_synthesis["evidence_class"].astype(str).eq("positive_causal_perturbation"),
        final_synthesis["evidence_class"].astype(str).eq("inverse_perturbation_response"),
    ],
    [
        "supports_anatomical_saliency_plausibility",
        "supports_positive_perturbation_dependence",
        "supports_reliability_warning",
    ],
    default="does_not_support_positive_anatomical_claim",
)

final_synthesis["manuscript_use"] = np.select(
    [
        final_synthesis["analysis_family"].astype(str).eq("heatmap_enrichment"),
        final_synthesis["analysis_family"].astype(str).eq("anatomical_occlusion"),
    ],
    [
        "xai_anatomical_enrichment_result",
        "perturbation_reliability_result",
    ],
    default="supporting_result",
)

final_synthesis.to_csv(NOTEBOOK08_FINAL_SYNTHESIS_CSV, index=False)


# ---------------------------------------------------------------------
# Build final interpretation matrix
# ---------------------------------------------------------------------

interpretation_matrix = model_evidence_summary.copy()

interpretation_matrix["final_aqpr_interpretation"] = interpretation_matrix["integrated_aqpr_pattern"].map(
    {
        "convergent_anatomical_and_perturbation_support": (
            "Heatmap enrichment and perturbation evidence converge; strongest anatomical reliability pattern."
        ),
        "saliency_enriched_but_occlusion_inverse": (
            "Heatmap evidence is anatomically enriched, but perturbation evidence is inverse; report as reliability warning."
        ),
        "saliency_enriched_without_causal_perturbation_support": (
            "Heatmap evidence is anatomically enriched, but perturbation evidence is not significant."
        ),
        "perturbation_support_without_heatmap_support": (
            "Perturbation evidence is positive without consistent saliency support."
        ),
        "weak_or_inconsistent_anatomical_reliability": (
            "No consistent anatomical reliability pattern."
        ),
    }
)

interpretation_matrix["safe_manuscript_claim"] = interpretation_matrix["integrated_aqpr_pattern"].map(
    {
        "convergent_anatomical_and_perturbation_support": (
            "This architecture showed convergent OD/OC saliency and perturbation support on HYGD."
        ),
        "saliency_enriched_but_occlusion_inverse": (
            "This architecture showed OD/OC saliency enrichment but failed perturbation convergence."
        ),
        "saliency_enriched_without_causal_perturbation_support": (
            "This architecture showed OD/OC saliency enrichment without perturbation confirmation."
        ),
        "perturbation_support_without_heatmap_support": (
            "This architecture showed perturbation support without consistent heatmap support."
        ),
        "weak_or_inconsistent_anatomical_reliability": (
            "This architecture did not show a consistent AQPR anatomical reliability pattern."
        ),
    }
)

interpretation_matrix.to_csv(NOTEBOOK08_FINAL_INTERPRETATION_CSV, index=False)


# ---------------------------------------------------------------------
# Claim boundaries
# ---------------------------------------------------------------------

claim_boundaries = pd.DataFrame(
    [
        {
            "claim_type": "allowed",
            "claim": (
                "The frozen AIROGS-light AQPR panel was externally audited on HYGD using OD/OC proxy masks, "
                "matched random controls, image-level statistics, and patient-level aggregation."
            ),
            "rationale": "Directly supported by Notebook 08 outputs.",
        },
        {
            "claim_type": "allowed",
            "claim": (
                "The primary EfficientNetB0 model showed positive patient-level OD+OC enrichment using Grad-CAM "
                "and Integrated Gradients."
            ),
            "rationale": "Supported by positive FDR-controlled heatmap enrichment endpoints.",
        },
        {
            "claim_type": "allowed",
            "claim": (
                "The primary EfficientNetB0 model did not show convergent positive anatomical occlusion evidence; "
                "its OD+OC blur-occlusion response was inverse."
            ),
            "rationale": "Supported by the anatomical occlusion endpoint and integrated AQPR synthesis.",
        },
        {
            "claim_type": "allowed",
            "claim": (
                "AQPR identifies model-dependent anatomical reliability patterns: ConvNeXtTiny showed convergent "
                "saliency and perturbation support, whereas the validation-selected primary model did not."
            ),
            "rationale": "Supported by the model evidence summary.",
        },
        {
            "claim_type": "not_allowed",
            "claim": (
                "All high-performing glaucoma classifiers causally rely on OD/OC anatomy."
            ),
            "rationale": "Contradicted by inverse occlusion responses in the primary model and other panel members.",
        },
        {
            "claim_type": "not_allowed",
            "claim": (
                "AIROGS-light internal performance supports patient-level conclusions."
            ),
            "rationale": "AIROGS-light has no reliable patient identifiers in this pipeline; patient-level analysis is external HYGD only.",
        },
        {
            "claim_type": "not_allowed",
            "claim": (
                "Heatmap localization alone proves causal anatomical reasoning."
            ),
            "rationale": "Notebook 08 explicitly separates saliency enrichment from perturbation evidence.",
        },
    ]
)

claim_boundaries.to_csv(NOTEBOOK08_FINAL_CLAIM_BOUNDARIES_CSV, index=False)


# ---------------------------------------------------------------------
# Final status and downstream handoff
# ---------------------------------------------------------------------

primary_model_row = interpretation_matrix[
    interpretation_matrix["model_name"].astype(str) == PRIMARY_MODEL_NAME
].copy()

if len(primary_model_row) != 1:
    raise RuntimeError("Primary model is not uniquely represented in the final interpretation matrix.")

primary_model_row = primary_model_row.iloc[0].to_dict()

n_models = int(model_evidence_summary["model_name"].nunique())
n_positive_ig = int(
    model_evidence_summary["integrated_gradients_patient_level_evidence"]
    .astype(str)
    .eq("positive_anatomical_enrichment")
    .sum()
)
n_positive_gradcam = int(
    model_evidence_summary["gradcam_patient_level_evidence"]
    .astype(str)
    .eq("positive_anatomical_enrichment")
    .sum()
)
n_positive_occlusion = int(
    model_evidence_summary["occlusion_patient_level_evidence"]
    .astype(str)
    .eq("positive_causal_perturbation")
    .sum()
)
n_inverse_occlusion = int(
    model_evidence_summary["occlusion_patient_level_evidence"]
    .astype(str)
    .eq("inverse_perturbation_response")
    .sum()
)

notebook08_status = {
    "notebook": "08_anatomical_explainability_validation",
    "status": "PASS",
    "analysis_status": "complete",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_anatomical_dataset": "HYGD",
    "objective": (
        "AQPR anatomical explainability validation using heatmap enrichment, anatomical occlusion, "
        "matched random controls, and patient-level external aggregation."
    ),
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "n_models_in_panel": n_models,
    "n_models_with_positive_gradcam_patient_level_enrichment": n_positive_gradcam,
    "n_models_with_positive_integrated_gradients_patient_level_enrichment": n_positive_ig,
    "n_models_with_positive_occlusion_patient_level_effect": n_positive_occlusion,
    "n_models_with_inverse_occlusion_patient_level_effect": n_inverse_occlusion,
    "primary_model_integrated_aqpr_pattern": str(primary_model_row["integrated_aqpr_pattern"]),
    "primary_model_final_interpretation": str(primary_model_row["final_aqpr_interpretation"]),
    "primary_model_safe_manuscript_claim": str(primary_model_row["safe_manuscript_claim"]),
    "main_conclusion": (
        "The AQPR panel showed OD/OC saliency enrichment on HYGD, especially with Integrated Gradients. "
        "However, perturbation evidence was model-dependent and inverse for the validation-selected primary model. "
        "The defensible conclusion is partial anatomical plausibility with a perturbation-based reliability warning."
    ),
    "ready_for_downstream_notebooks": True,
    "recommended_next_step": (
        "Proceed to the next notebook only after adapting it to the current AQPR design. "
        "Do not use the obsolete multifinalist/DeiTSmall/MobileNetV3Large assumptions from the original Notebook 08."
    ),
    "patient_level_external_analysis_allowed": True,
    "internal_airogslight_patient_level_claim_allowed": False,
    "final_synthesis_csv": str(NOTEBOOK08_FINAL_SYNTHESIS_CSV),
    "final_interpretation_csv": str(NOTEBOOK08_FINAL_INTERPRETATION_CSV),
    "final_claim_boundaries_csv": str(NOTEBOOK08_FINAL_CLAIM_BOUNDARIES_CSV),
    "final_artifact_registry_csv": str(NOTEBOOK08_FINAL_ARTIFACT_REGISTRY_CSV),
    "aqpr_synthesis_json": str(AQPR_SYNTHESIS_JSON),
    "downstream_handoff_json": str(NOTEBOOK08_DOWNSTREAM_HANDOFF_JSON),
}

downstream_handoff = {
    "source_notebook": "08_anatomical_explainability_validation",
    "status": "PASS",
    "use_these_outputs_downstream": {
        "final_synthesis_csv": str(NOTEBOOK08_FINAL_SYNTHESIS_CSV),
        "final_interpretation_csv": str(NOTEBOOK08_FINAL_INTERPRETATION_CSV),
        "claim_boundaries_csv": str(NOTEBOOK08_FINAL_CLAIM_BOUNDARIES_CSV),
        "primary_endpoint_table_csv": str(AQPR_PRIMARY_ENDPOINT_TABLE_CSV),
        "model_evidence_summary_csv": str(AQPR_MODEL_EVIDENCE_SUMMARY_CSV),
        "heatmap_metrics_csv": str(HEATMAP_CONSOLIDATED_CSV),
        "heatmap_patient_metrics_csv": str(HEATMAP_PATIENT_LEVEL_CSV),
        "heatmap_patient_stats_csv": str(HEATMAP_PATIENT_LEVEL_STATS_CSV),
        "occlusion_metrics_csv": str(OCCLUSION_CONSOLIDATED_CSV),
        "occlusion_patient_metrics_csv": str(OCCLUSION_PATIENT_LEVEL_CSV),
        "occlusion_patient_stats_csv": str(OCCLUSION_PATIENT_LEVEL_STATS_CSV),
    },
    "do_not_use_obsolete_assumptions": [
        "single HYGD-trained classifier",
        "primary_test_strict from old HYGD classifier benchmark",
        "DeiTSmall/MobileNetV3Large multifinalist design",
        "patient-level AIROGS-light internal claims",
        "heatmaps as standalone causal evidence",
    ],
    "required_methodological_boundary": (
        "Notebook 08 supports AQPR as a reliability audit. It does not support claiming universal causal OD/OC reliance."
    ),
}

with open(NOTEBOOK08_FINAL_STATUS_JSON, "w", encoding="utf-8") as f:
    json.dump(notebook08_status, f, indent=2)

with open(NOTEBOOK08_DOWNSTREAM_HANDOFF_JSON, "w", encoding="utf-8") as f:
    json.dump(downstream_handoff, f, indent=2)

with open(NOTEBOOK08_FINAL_STATUS_TXT, "w", encoding="utf-8") as f:
    f.write("Notebook 08 final status: PASS\n")
    f.write("Analysis: AQPR anatomical explainability validation on HYGD\n")
    f.write(f"Primary model: {PRIMARY_MODEL_NAME}\n")
    f.write(f"Primary AQPR pattern: {notebook08_status['primary_model_integrated_aqpr_pattern']}\n")
    f.write("\nMain conclusion:\n")
    f.write(notebook08_status["main_conclusion"] + "\n")


# ---------------------------------------------------------------------
# Console report
# ---------------------------------------------------------------------

print("Notebook 08 final artifact registry:")
print(artifact_registry.to_string(index=False))

print("\nNotebook 08 final comparative synthesis:")
print(
    final_synthesis[
        [
            "analysis_family",
            "model_name",
            "method_name",
            "region_name",
            "primary_endpoint_mean",
            "primary_endpoint_ci_low",
            "primary_endpoint_ci_high",
            "primary_fdr_p_value",
            "evidence_class",
            "claim_strength",
        ]
    ].to_string(index=False)
)

print("\nNotebook 08 final interpretation matrix:")
print(
    interpretation_matrix[
        [
            "model_name",
            "aqpr_panel_role",
            "gradcam_patient_level_evidence",
            "integrated_gradients_patient_level_evidence",
            "occlusion_patient_level_evidence",
            "integrated_aqpr_pattern",
            "safe_manuscript_claim",
        ]
    ].to_string(index=False)
)

print("\nNotebook 08 claim boundaries:")
print(claim_boundaries.to_string(index=False))

print("\nNotebook 08 final status:")
print(json.dumps(notebook08_status, indent=2))

print(f"\nFinal synthesis CSV: {NOTEBOOK08_FINAL_SYNTHESIS_CSV}")
print(f"Final interpretation CSV: {NOTEBOOK08_FINAL_INTERPRETATION_CSV}")
print(f"Claim boundaries CSV: {NOTEBOOK08_FINAL_CLAIM_BOUNDARIES_CSV}")
print(f"Artifact registry CSV: {NOTEBOOK08_FINAL_ARTIFACT_REGISTRY_CSV}")
print(f"Downstream handoff JSON: {NOTEBOOK08_DOWNSTREAM_HANDOFF_JSON}")
print(f"Final status JSON: {NOTEBOOK08_FINAL_STATUS_JSON}")
print(f"Final status TXT: {NOTEBOOK08_FINAL_STATUS_TXT}")


for required_output in [
    NOTEBOOK08_FINAL_SYNTHESIS_CSV,
    NOTEBOOK08_FINAL_INTERPRETATION_CSV,
    NOTEBOOK08_FINAL_CLAIM_BOUNDARIES_CSV,
    NOTEBOOK08_FINAL_ARTIFACT_REGISTRY_CSV,
    NOTEBOOK08_DOWNSTREAM_HANDOFF_JSON,
    NOTEBOOK08_FINAL_STATUS_JSON,
    NOTEBOOK08_FINAL_STATUS_TXT,
]:
    if not Path(required_output).exists():
        raise RuntimeError(f"Missing final Notebook 08 output: {required_output}")

if notebook08_status["status"] != "PASS":
    raise RuntimeError("Notebook 08 final status is not PASS.")

if notebook08_status["primary_model_integrated_aqpr_pattern"] == "convergent_anatomical_and_perturbation_support":
    print("Primary model has convergent AQPR anatomical support.")
else:
    print("Primary model has an AQPR reliability warning; this must be preserved in the manuscript.")

print("Status: PASS")